Если окружение ещё не настроено:

```python
# %pip install pandas pyarrow numpy matplotlib seaborn scikit-learn jupyter ipykernel
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

# Модифицируйте в соответствии со своей задачей
TRACK = "solo"  # "solo" or "team"
TRAIN_DAYS = 14
MAX_TRAIN_ROWS = 1_500_000
RIDGE_ALPHA = 4.0
RANDOM_STATE = 42

# Меняйте конфигурацию при необходимости
TRACK_CONFIG = {
    "solo": {
        "train_path": "train_solo_track.parquet",
        "test_path": "test_solo_track.parquet",
        "target_col": "target_1h",
        "forecast_points": 8,
    },
    "team": {
        "train_path": "train_team_track.parquet",
        "test_path": "test_team_track.parquet",
        "target_col": "target_2h",
        "forecast_points": 10,
    },
}

CONFIG = TRACK_CONFIG[TRACK]
TARGET_COL = CONFIG["target_col"]
FORECAST_POINTS = CONFIG["forecast_points"]
FUTURE_TARGET_COLS = [f"target_step_{step}" for step in range(1, FORECAST_POINTS + 1)]


## Загрузка данных


In [ ]:
train_df = pd.read_parquet(CONFIG["train_path"])
test_df = pd.read_parquet(CONFIG["test_path"])

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print("track:", TRACK)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)


In [ ]:
display(train_df.head())
display(test_df.head())


In [ ]:
print("Train date range:", train_df["timestamp"].min(), "->", train_df["timestamp"].max())
print("Test date range:", test_df["timestamp"].min(), "->", test_df["timestamp"].max())
print("Train routes:", train_df["route_id"].nunique())
print("Test routes:", test_df["route_id"].nunique())


## EDA train-данных


In [ ]:
overview = pd.DataFrame(
    {
        "dtype": train_df.dtypes.astype(str),
        "missing_cnt": train_df.isna().sum(),
        "missing_pct": (train_df.isna().mean() * 100).round(4),
        "n_unique": train_df.nunique(dropna=False),
    }
)
overview


In [ ]:
status_cols = sorted([col for col in train_df.columns if col.startswith("status_")])
print("Status columns:", status_cols)
print("Target column:", TARGET_COL)
print("Forecast points:", FORECAST_POINTS)


## Распределения


In [ ]:
fig, axes = plt.subplots(len(status_cols), 1, figsize=(14, 4 * len(status_cols)))

if len(status_cols) == 1:
    axes = [axes]

for i, col in enumerate(status_cols):
    sns.histplot(train_df[col].clip(upper=train_df[col].quantile(0.99)), bins=30, ax=axes[i], kde=False)
    axes[i].set_title(f"{col} (clipped at p99)")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 4))
sns.histplot(train_df[TARGET_COL].clip(upper=train_df[TARGET_COL].quantile(0.99)), bins=25, kde=False)
plt.title(f"{TARGET_COL} (clipped at p99)")
plt.show()


## Генерируем будущие таргеты


In [ ]:
route_group = train_df.groupby("route_id", sort=False)

for step in range(1, FORECAST_POINTS + 1):
    train_df[f"target_step_{step}"] = route_group[TARGET_COL].shift(-step)

train_df[["route_id", "timestamp", TARGET_COL] + FUTURE_TARGET_COLS].head(10)


In [ ]:
supervised_df = train_df.dropna(subset=FUTURE_TARGET_COLS).copy()
print("Rows with future targets:", supervised_df.shape)


## Корреляции


In [ ]:
corr_cols = status_cols + FUTURE_TARGET_COLS
corr = supervised_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlations on supervised train")
plt.show()


## Подготовка train и test


In [ ]:
feature_cols = [col for col in train_df.columns if col not in {TARGET_COL, "timestamp", "id", *FUTURE_TARGET_COLS}]

print("Feature columns:", feature_cols)


In [ ]:
train_model_df = supervised_df[feature_cols + ["timestamp"] + FUTURE_TARGET_COLS].copy()
train_model_df = train_model_df.rename(columns={"timestamp": "source_timestamp"})

train_ts_max = train_model_df["source_timestamp"].max()
train_window_start = train_ts_max - pd.Timedelta(days=TRAIN_DAYS)
train_model_df = train_model_df[train_model_df["source_timestamp"] >= train_window_start].copy()

print("Recent train rows:", train_model_df.shape)


In [ ]:
# последний момент факта, из которого делаем прогноз
inference_ts = train_df["timestamp"].max()
test_model_df = train_df[train_df["timestamp"] == inference_ts]

print("Test rows:", test_model_df.shape)


## Time-based split


In [ ]:
train_model_df = train_model_df.sort_values("source_timestamp").copy()
split_point = train_model_df["source_timestamp"].quantile(0.8)

fit_df = train_model_df[train_model_df["source_timestamp"] <= split_point].copy()
valid_df = train_model_df[train_model_df["source_timestamp"] > split_point].copy()

if len(fit_df) > MAX_TRAIN_ROWS:
    fit_df = fit_df.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)

print("Fit rows:", fit_df.shape)
print("Valid rows:", valid_df.shape)


In [ ]:
X_fit = fit_df[feature_cols].copy()
y_fit = fit_df[FUTURE_TARGET_COLS].copy()

X_valid = valid_df[feature_cols].copy()
y_valid = valid_df[FUTURE_TARGET_COLS].copy()

X_test = test_model_df[feature_cols].copy()


## Линейный baseline


In [ ]:
categorical_features = [col for col in feature_cols if col.endswith("_id")]
numeric_features = [col for col in feature_cols if col not in categorical_features]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=RIDGE_ALPHA)),
    ]
)


In [ ]:
model.fit(X_fit, y_fit)

In [ ]:
fit_pred_df = pd.DataFrame(model.predict(X_fit), columns=FUTURE_TARGET_COLS, index=fit_df.index)
valid_pred_df = pd.DataFrame(model.predict(X_valid), columns=FUTURE_TARGET_COLS, index=valid_df.index)
test_pred_df = pd.DataFrame(model.predict(X_test), columns=FUTURE_TARGET_COLS, index=test_model_df.index)

valid_pred_df.head()


## Метрики


In [ ]:
class WapePlusRbias:
    """Calculates as WAPE + Relative Bias."""

    @property
    def name(self) -> str:
        """Возвращает имя метрики."""
        return "wape_plus_rbias"

    def calculate(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Рассчитывает значение метрики."""
        wape = (np.abs(y_pred - y_true)).sum() / y_true.sum()
        rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
        return wape + rbias


metric = WapePlusRbias()

In [ ]:
print('Метрики на тесте (по горизонтам):')
display(np.round(metric.calculate(y_fit, fit_pred_df), 2))

print('Общая метрика на тесте:')
print(f'{metric.calculate(y_fit.to_numpy().flatten(), fit_pred_df.to_numpy().flatten()):.2f}')

In [ ]:
print('Метрики на валидации (по горизонтам):')
display(np.round(metric.calculate(y_valid, valid_pred_df), 2))

print('Общая метрика на валидации:')
print(f'{metric.calculate(y_valid.to_numpy().flatten(), valid_pred_df.to_numpy().flatten()):.2f}')

## Конвертируем прогноз в нужный формат 


In [ ]:
# добавляем к прогнозу маршруты
test_pred_df['route_id'] = X_test['route_id']

# разворачиваем target_step_* в строки
forecast_df = test_pred_df.melt(
    id_vars="route_id",
    value_vars=[c for c in test_pred_df.columns if c.startswith("target_step_")],
    var_name="step",
    value_name="forecast"
)

# достаем номер шага из target_step_1, target_step_2, ...
forecast_df["step_num"] = forecast_df["step"].str.extract(r"(\d+)").astype(int)

# строим timestamp: каждый шаг = +30 минут от времени прогноза
forecast_df["timestamp"] = inference_ts + pd.to_timedelta(forecast_df["step_num"] * 30, unit="m")

# оставляем нужные столбцы
forecast_df = forecast_df[["route_id", "timestamp", "forecast"]].sort_values(
    ["route_id", "timestamp"]
).reset_index(drop=True)

forecast_df = test_df.merge(forecast_df, 'outer')[["id", "forecast"]]
forecast_df = forecast_df.rename(columns={"forecast": "y_pred"})

In [ ]:
forecast_df.head()

In [ ]:
# проверяем, что все точки получены
assert forecast_df['id'].isna().sum() == 0


## Выгрузка CSV


In [ ]:
submission_path =  f"submission_{TRACK}.csv"
joined_path =  f"test_with_forecast_{TRACK}.csv"

forecast_df.to_csv(submission_path, index=False)

print("submission saved to:", submission_path)

In [ ]:
# ============================================================
# Check sequential hypothesis: does route (j-1) lead route j?
# ============================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from statsmodels.tsa.stattools import grangercausalitytests
import warnings

warnings.filterwarnings("ignore")

# ---------- Config ----------
TRAIN_PATH = "train_solo_track.parquet"
TARGET_COL = "target_1h"
MAX_LAG = 8                  # 8 шагов по 30 минут = 4 часа
NEIGHBOR_RADIUS = 3          # сначала проверяем соседей по id
MIN_POINTS = 200             # минимум точек для пары
USE_GRANGER = True           # можно выключить, если долго

STATUS_CUR = ["status_1", "status_2", "status_3"]
STATUS_PREV = ["status_4", "status_5", "status_6"]

# ---------- Load ----------
train_df = pd.read_parquet(TRAIN_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Unique routes:", train_df["route_id"].nunique())

# ---------- Wide target matrix ----------
target_wide = (
    train_df.pivot(index="timestamp", columns="route_id", values=TARGET_COL)
    .sort_index()
    .asfreq("30min")
)

# мягкая импутация внутри маршрута
target_wide = target_wide.interpolate(method="time").ffill().bfill()

route_ids = sorted(target_wide.columns.tolist())

# ---------- Optional: wide status matrices ----------
status_wide = {}
for col in STATUS_CUR + STATUS_PREV:
    if col in train_df.columns:
        tmp = (
            train_df.pivot(index="timestamp", columns="route_id", values=col)
            .sort_index()
            .asfreq("30min")
            .interpolate(method="time")
            .ffill()
            .bfill()
        )
        status_wide[col] = tmp

# ---------- Helper: lagged correlation ----------
def best_lag_corr(x, y, max_lag=8):
    best_corr = -np.inf
    best_lag = None
    for lag in range(1, max_lag + 1):
        x_lag = x.shift(lag)
        valid = x_lag.notna() & y.notna()
        if valid.sum() < MIN_POINTS:
            continue
        c = np.corrcoef(x_lag[valid], y[valid])[0, 1]
        if np.isfinite(c) and c > best_corr:
            best_corr = c
            best_lag = lag
    if best_lag is None:
        return np.nan, np.nan
    return best_corr, best_lag

# ---------- Helper: Granger ----------
def best_granger_pvalue(x, y, max_lag=8):
    df = pd.concat([y, x], axis=1).dropna()
    df.columns = ["y", "x"]  # test: x causes y
    if len(df) < MIN_POINTS:
        return np.nan
    try:
        res = grangercausalitytests(df[["y", "x"]], maxlag=max_lag, verbose=False)
        pvals = [res[lag][0]["ssr_ftest"][1] for lag in range(1, max_lag + 1)]
        return float(np.nanmin(pvals))
    except Exception:
        return np.nan

# ---------- Main check on target ----------
rows = []

for j in tqdm(route_ids, desc="Checking routes"):
    y = target_wide[j]

    candidates = [i for i in route_ids if i != j and abs(i - j) <= NEIGHBOR_RADIUS]
    if not candidates:
        continue

    local_scores = []
    for i in candidates:
        x = target_wide[i]
        corr, lag = best_lag_corr(x, y, max_lag=MAX_LAG)
        if np.isnan(corr):
            continue

        g_pval = np.nan
        if USE_GRANGER:
            g_pval = best_granger_pvalue(x, y, max_lag=MAX_LAG)

        local_scores.append({
            "route_id": j,
            "candidate_parent": i,
            "best_corr": corr,
            "best_lag": lag,
            "granger_pval": g_pval,
            "is_prev_id": int(i == j - 1),
        })

    if not local_scores:
        continue

    local_df = pd.DataFrame(local_scores).sort_values(
        ["best_corr", "is_prev_id"], ascending=[False, False]
    )

    best = local_df.iloc[0].to_dict()

    # отрыв от второго места
    if len(local_df) > 1:
        second_corr = local_df.iloc[1]["best_corr"]
        margin = best["best_corr"] - second_corr
    else:
        margin = np.nan

    best["margin_to_second"] = margin
    rows.append(best)

result_df = pd.DataFrame(rows).sort_values("route_id").reset_index(drop=True)

# ---------- Sequential metrics ----------
result_df["pred_prev"] = (result_df["candidate_parent"] == result_df["route_id"] - 1).astype(int)
seq_rate = result_df["pred_prev"].mean()

stable_mask = result_df["margin_to_second"].fillna(0) > 0.05
stable_seq_rate = result_df.loc[stable_mask, "pred_prev"].mean() if stable_mask.any() else np.nan

print("\n=== Sequential hypothesis metrics ===")
print("Routes checked:", len(result_df))
print("seq_rate:", round(seq_rate, 4))
print("stable_seq_rate:", round(stable_seq_rate, 4) if pd.notna(stable_seq_rate) else np.nan)

print("\nTop rows:")
print(result_df.head(20).to_string(index=False))

result_df.to_csv("route_sequential_check.csv", index=False)
print("\nSaved: route_sequential_check.csv")

In [ ]:
# ============================================================
# Build dependency graph between routes from time series
# ============================================================
import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm
from statsmodels.tsa.stattools import grangercausalitytests
import warnings

warnings.filterwarnings("ignore")

# ---------- Config ----------
TRAIN_PATH = "train_solo_track.parquet"
TARGET_COL = "target_1h"
MAX_LAG = 8
TOP_SCREEN = 20       # быстрый screening по корреляции
TOP_IN = 3            # сколько входящих ребер оставлять на узел
TOP_OUT = 3           # сколько исходящих ребер оставлять на узел
MIN_POINTS = 200
USE_GRANGER = True

# ---------- Load ----------
df = pd.read_parquet(TRAIN_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

# ---------- Wide matrix ----------
y = (
    df.pivot(index="timestamp", columns="route_id", values=TARGET_COL)
      .sort_index()
      .asfreq("30min")
)

# мягкая импутация
y = y.interpolate(method="time").ffill().bfill()

# ---------- De-seasonalize / de-mean ----------
# 1) центрирование по маршруту
y = y - y.mean(axis=0)

# 2) простая сезонная поправка по часу и дню недели
tmp = y.copy()
tmp["hour"] = tmp.index.hour
tmp["dow"] = tmp.index.dayofweek

seasonal = tmp.groupby(["dow", "hour"]).transform("mean")
y_adj = y - seasonal.drop(columns=["hour", "dow"], errors="ignore")

# 3) first difference
y_adj = y_adj.diff().dropna()

route_ids = list(y_adj.columns)

# ---------- Helpers ----------
def best_lag_corr(x, y, max_lag=8):
    best_corr, best_lag = -np.inf, None
    for lag in range(1, max_lag + 1):
        x_lag = x.shift(lag)
        valid = x_lag.notna() & y.notna()
        if valid.sum() < MIN_POINTS:
            continue
        c = np.corrcoef(x_lag[valid], y[valid])[0, 1]
        if np.isfinite(c) and c > best_corr:
            best_corr, best_lag = c, lag
    if best_lag is None:
        return np.nan, np.nan
    return float(best_corr), int(best_lag)

def best_granger_pvalue(x, y, max_lag=8):
    z = pd.concat([y, x], axis=1).dropna()
    z.columns = ["y", "x"]   # test x -> y
    if len(z) < MIN_POINTS:
        return np.nan
    try:
        res = grangercausalitytests(z[["y", "x"]], maxlag=max_lag, verbose=False)
        pvals = [res[k][0]["ssr_ftest"][1] for k in range(1, max_lag + 1)]
        return float(np.nanmin(pvals))
    except:
        return np.nan

def score_edge(corr, pval):
    if not np.isfinite(corr):
        return np.nan
    ppart = 0.0 if not np.isfinite(pval) else max(0.0, -np.log10(max(pval, 1e-12)))
    return 0.7 * max(corr, 0) + 0.3 * min(ppart / 12, 1.0)

# ---------- Stage 1: fast screening ----------
screen_rows = []

for j in tqdm(route_ids, desc="Screening"):
    yj = y_adj[j]
    cand = []
    for i in route_ids:
        if i == j:
            continue
        corr, lag = best_lag_corr(y_adj[i], yj, max_lag=MAX_LAG)
        if np.isfinite(corr):
            cand.append((i, corr, lag))
    cand = sorted(cand, key=lambda x: x[1], reverse=True)[:TOP_SCREEN]
    for i, corr, lag in cand:
        screen_rows.append({"src": i, "dst": j, "corr": corr, "lag": lag})

screen_df = pd.DataFrame(screen_rows)

# ---------- Stage 2: refine with Granger ----------
edge_rows = []

for j in tqdm(route_ids, desc="Refining"):
    sub = screen_df[screen_df["dst"] == j].sort_values("corr", ascending=False)
    for _, r in sub.iterrows():
        i = r["src"]
        corr = r["corr"]
        lag = r["lag"]
        pval = best_granger_pvalue(y_adj[i], y_adj[j], max_lag=MAX_LAG) if USE_GRANGER else np.nan
        s = score_edge(corr, pval)
        edge_rows.append({
            "src": i,
            "dst": j,
            "best_corr": corr,
            "best_lag": lag,
            "granger_pval": pval,
            "score": s
        })

edges = pd.DataFrame(edge_rows).dropna(subset=["score"])

# ---------- Sparsify ----------
# top incoming
top_in_df = (
    edges.sort_values(["dst", "score"], ascending=[True, False])
         .groupby("dst")
         .head(TOP_IN)
)

# top outgoing
top_out_df = (
    edges.sort_values(["src", "score"], ascending=[True, False])
         .groupby("src")
         .head(TOP_OUT)
)

final_edges = pd.concat([top_in_df, top_out_df], ignore_index=True).drop_duplicates(["src", "dst"])

# optional threshold
final_edges = final_edges[final_edges["score"] > final_edges["score"].quantile(0.75)].copy()

# ---------- Build graph ----------
G = nx.DiGraph()
for rid in route_ids:
    G.add_node(int(rid))

for _, r in final_edges.iterrows():
    G.add_edge(int(r["src"]), int(r["dst"]),
               weight=float(r["score"]),
               lag=int(r["best_lag"]),
               corr=float(r["best_corr"]),
               pval=float(r["granger_pval"]) if pd.notna(r["granger_pval"]) else np.nan)

# ---------- Node metrics ----------
pagerank = nx.pagerank(G, weight="weight")
in_deg = dict(G.in_degree(weight="weight"))
out_deg = dict(G.out_degree(weight="weight"))

nodes = pd.DataFrame({
    "route_id": list(G.nodes()),
    "pagerank": [pagerank.get(n, 0.0) for n in G.nodes()],
    "in_weight": [in_deg.get(n, 0.0) for n in G.nodes()],
    "out_weight": [out_deg.get(n, 0.0) for n in G.nodes()],
})

# weakly connected components
comp_map = {}
for comp_id, comp in enumerate(nx.weakly_connected_components(G)):
    for n in comp:
        comp_map[n] = comp_id
nodes["component_id"] = nodes["route_id"].map(comp_map)

# ---------- Save ----------
final_edges = final_edges.sort_values("score", ascending=False).reset_index(drop=True)
nodes = nodes.sort_values("pagerank", ascending=False).reset_index(drop=True)

final_edges.to_csv("route_dependency_edges.csv", index=False)
nodes.to_csv("route_dependency_nodes.csv", index=False)

print("Saved: route_dependency_edges.csv")
print("Saved: route_dependency_nodes.csv")
print("\nTop edges:")
print(final_edges.head(20).to_string(index=False))
print("\nTop nodes:")
print(nodes.head(20).to_string(index=False))
print("\nGraph stats:")
print("nodes =", G.number_of_nodes())
print("edges =", G.number_of_edges())

In [ ]:
# ============================================================
# Full visualization: route dependency graph
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

os.makedirs("output", exist_ok=True)

# ============================================================
# 1. Load & normalize types
# ============================================================
edges = pd.read_csv("route_dependency_edges.csv")
nodes_df = pd.read_csv("route_dependency_nodes.csv")

edges["src"] = edges["src"].astype(int)
edges["dst"] = edges["dst"].astype(int)
nodes_df["route_id"] = nodes_df["route_id"].astype(int)

# ============================================================
# 2. Build full graph
# ============================================================
G = nx.DiGraph()

for _, r in nodes_df.iterrows():
    G.add_node(
        int(r["route_id"]),
        pagerank=float(r["pagerank"]),
        in_weight=float(r["in_weight"]),
        out_weight=float(r["out_weight"]),
    )

for _, r in edges.iterrows():
    G.add_edge(
        int(r["src"]), int(r["dst"]),
        weight=float(r["score"]),
        corr=float(r["best_corr"]),
        lag=int(r["best_lag"]),
        pval=float(r["granger_pval"]) if pd.notna(r["granger_pval"]) else np.nan,
    )

# ============================================================
# 3. Top-N subgraph для визуализации (берём топ-200 ребер)
# ============================================================
TOP_EDGES = 200

draw_edges_df = edges.sort_values("score", ascending=False).head(TOP_EDGES).copy()
draw_node_set = set(draw_edges_df["src"]).union(set(draw_edges_df["dst"]))
H = G.subgraph(draw_node_set).copy()

# ============================================================
# 4. Communities через встроенный networkx Louvain
# ============================================================
H_und = H.to_undirected()
communities = nx.community.louvain_communities(H_und, weight="weight", seed=42)

partition = {}
for cid, comm in enumerate(communities):
    for node in comm:
        partition[int(node)] = cid

nx.set_node_attributes(H, {int(n): v for n, v in partition.items()}, "community")

n_comm = len(communities)
print(f"H: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges, {n_comm} communities")
print(f"Community sizes: {sorted([len(c) for c in communities], reverse=True)}")

# ============================================================
# 5. Static PNG — главный граф
# ============================================================
fig, ax = plt.subplots(figsize=(20, 16))

pos = nx.spring_layout(H, k=0.5, iterations=200, seed=42, weight="weight")

# размер узла — pagerank, цвет — community
pr_vals = np.array([H.nodes[n]["pagerank"] for n in H.nodes()])
pr_norm = (pr_vals - pr_vals.min()) / (pr_vals.max() - pr_vals.min() + 1e-9)
node_sizes = 100 + 2000 * pr_norm

comm_vals = [H.nodes[n]["community"] for n in H.nodes()]
cmap = plt.cm.get_cmap("tab20", n_comm)
node_colors = [cmap(c) for c in comm_vals]

# ширина ребра — вес
edge_widths = [0.5 + 6 * H[u][v]["weight"] for u, v in H.edges()]
edge_colors = [H[u][v]["weight"] for u, v in H.edges()]

nodes_draw = nx.draw_networkx_nodes(
    H, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.88,
    ax=ax,
)
nx.draw_networkx_edges(
    H, pos,
    width=edge_widths,
    alpha=0.20,
    arrows=True,
    arrowsize=12,
    arrowstyle="-|>",
    edge_color="silver",
    ax=ax,
)

# подписываем топ-25 по pagerank
top25 = nodes_df.sort_values("pagerank", ascending=False).head(25)["route_id"].tolist()
labels = {n: str(n) for n in H.nodes() if n in top25}
nx.draw_networkx_labels(H, pos, labels=labels, font_size=8, font_color="white", ax=ax)

# легенда сообществ
patches = [
    mpatches.Patch(color=cmap(i), label=f"Community {i} ({len(communities[i])} routes)")
    for i in range(n_comm)
]
ax.legend(handles=patches, loc="upper left", fontsize=8, framealpha=0.7)

ax.set_title(
    f"Route dependency graph — top {TOP_EDGES} edges\n"
    "Node size = PageRank · Color = Community · Arrows = lead-lag direction",
    fontsize=14, pad=14,
)
ax.axis("off")
plt.tight_layout()
plt.savefig("output/route_graph_top200.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved: output/route_graph_top200.png")

# ============================================================
# 6. Heatmap: inter-community flow
# ============================================================
edges["src_comm"] = edges["src"].map(partition)
edges["dst_comm"] = edges["dst"].map(partition)

flow = (
    edges.dropna(subset=["src_comm", "dst_comm"])
         .groupby(["src_comm", "dst_comm"])["score"]
         .sum()
         .reset_index()
)

mat = flow.pivot(index="src_comm", columns="dst_comm", values="score").fillna(0)
mat.index = [f"C{i}" for i in mat.index]
mat.columns = [f"C{i}" for i in mat.columns]

fig, ax = plt.subplots(figsize=(max(6, n_comm + 2), max(5, n_comm + 1)))
sns.heatmap(
    mat,
    cmap="magma",
    annot=(n_comm <= 12),
    fmt=".2f",
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Inter-community flow (sum of edge scores)", fontsize=13)
ax.set_xlabel("Destination community")
ax.set_ylabel("Source community")
plt.tight_layout()
plt.savefig("output/route_community_flow_heatmap.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved: output/route_community_flow_heatmap.png")

# ============================================================
# 7. PageRank distribution (lollipop chart)
# ============================================================
top_nodes = nodes_df.sort_values("pagerank", ascending=False).head(30).copy()

fig, ax = plt.subplots(figsize=(16, 7))
x = np.arange(len(top_nodes))

# классифицируем узел по in/out balance
top_nodes["role"] = np.where(
    top_nodes["out_weight"] > top_nodes["in_weight"] * 1.5, "Source",
    np.where(top_nodes["in_weight"] > top_nodes["out_weight"] * 1.5, "Sink", "Hub")
)
role_colors = {"Source": "#22c55e", "Sink": "#ef4444", "Hub": "#3b82f6"}
colors = top_nodes["role"].map(role_colors).tolist()

ax.vlines(x, 0, top_nodes["pagerank"].values, colors="gray", linewidth=1.2, alpha=0.5)
ax.scatter(x, top_nodes["pagerank"].values, c=colors, s=90, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(top_nodes["route_id"].astype(str).tolist(), rotation=45, ha="right", fontsize=9)
ax.set_ylabel("PageRank")
ax.set_title("Top-30 routes by PageRank · green = Source · red = Sink · blue = Hub")

legend_patches = [
    mpatches.Patch(color="#22c55e", label="Source (out >> in)"),
    mpatches.Patch(color="#ef4444", label="Sink (in >> out)"),
    mpatches.Patch(color="#3b82f6", label="Hub (balanced)"),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("output/route_pagerank_top30.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved: output/route_pagerank_top30.png")

# ============================================================
# 8. In/Out weight scatter — кто Source, кто Sink
# ============================================================
fig, ax = plt.subplots(figsize=(10, 8))

scatter_data = nodes_df.copy()
scatter_data["role"] = np.where(
    scatter_data["out_weight"] > scatter_data["in_weight"] * 1.5, "Source",
    np.where(scatter_data["in_weight"] > scatter_data["out_weight"] * 1.5, "Sink", "Hub")
)

for role, color in role_colors.items():
    sub = scatter_data[scatter_data["role"] == role]
    ax.scatter(sub["out_weight"], sub["in_weight"], c=color, alpha=0.6, s=40, label=role)

# аннотируем экстремальные точки
extremes = pd.concat([
    scatter_data.nlargest(5, "out_weight"),
    scatter_data.nlargest(5, "in_weight"),
    scatter_data.nlargest(5, "pagerank"),
]).drop_duplicates("route_id")

for _, r in extremes.iterrows():
    ax.annotate(
        str(int(r["route_id"])),
        (r["out_weight"], r["in_weight"]),
        textcoords="offset points",
        xytext=(5, 3),
        fontsize=7.5,
        color="white",
    )

max_val = max(scatter_data["out_weight"].max(), scatter_data["in_weight"].max()) * 1.05
ax.plot([0, max_val], [0, max_val], "w--", alpha=0.3, linewidth=1)
ax.set_xlabel("Out-weight (influence emitted)")
ax.set_ylabel("In-weight (influence received)")
ax.set_title("Route roles: Sources vs Sinks vs Hubs (in/out weighted degree)")
ax.legend(fontsize=9)
ax.set_facecolor("#0f172a")
fig.patch.set_facecolor("#0f172a")
for spine in ax.spines.values():
    spine.set_edgecolor("#334155")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
plt.tight_layout()
plt.savefig("output/route_in_out_scatter.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved: output/route_in_out_scatter.png")

# ============================================================
# 9. Interactive HTML через PyVis
# ============================================================
try:
    from pyvis.network import Network

    net = Network(
        height="900px",
        width="100%",
        directed=True,
        bgcolor="#0f172a",
        font_color="white",
    )
    net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=150)

    for n in H.nodes():
        pr = float(H.nodes[n]["pagerank"])
        comm = int(H.nodes[n].get("community", 0))
        role = "Source" if H.nodes[n]["out_weight"] > H.nodes[n]["in_weight"] * 1.5 \
               else "Sink" if H.nodes[n]["in_weight"] > H.nodes[n]["out_weight"] * 1.5 \
               else "Hub"
        role_hex = {"Source": "#22c55e", "Sink": "#ef4444", "Hub": "#3b82f6"}[role]
        title = (
            f"<b>route {n}</b><br>"
            f"Role: {role}<br>"
            f"PageRank: {pr:.5f}<br>"
            f"In-weight: {H.nodes[n]['in_weight']:.3f}<br>"
            f"Out-weight: {H.nodes[n]['out_weight']:.3f}<br>"
            f"Community: {comm}"
        )
        net.add_node(
            str(int(n)),
            label=str(int(n)),
            title=title,
            size=float(8 + 200 * pr),
            color=role_hex,
            group=comm,
        )

    for u, v, d in H.edges(data=True):
        pval_str = f"{d['pval']:.2e}" if np.isfinite(d.get("pval", np.nan)) else "n/a"
        net.add_edge(
            str(int(u)),
            str(int(v)),
            value=float(2 + 10 * d["weight"]),
            title=f"score={d['weight']:.3f}  corr={d['corr']:.3f}  lag={d['lag']}  pval={pval_str}",
            color="#94a3b8",
        )

    net.save_graph("output/route_graph_interactive.html")
    print("Saved: output/route_graph_interactive.html")

except ImportError:
    print("PyVis не установлен — HTML-граф пропускаем. pip install pyvis")

# ============================================================
# 10. Summary таблица по ролям
# ============================================================
summary = nodes_df.copy()
summary["role"] = np.where(
    summary["out_weight"] > summary["in_weight"] * 1.5, "Source",
    np.where(summary["in_weight"] > summary["out_weight"] * 1.5, "Sink", "Hub")
)
summary["community"] = summary["route_id"].map(partition)

summary.to_csv("output/route_node_roles.csv", index=False)
print("\nSaved: output/route_node_roles.csv")

print("\n=== Role distribution ===")
print(summary["role"].value_counts())
print("\n=== Community distribution ===")
print(summary["community"].value_counts().sort_index())

print("\n=== Top-10 Sources (out >> in) ===")
print(summary[summary["role"] == "Source"].nlargest(10, "out_weight")
      [["route_id", "pagerank", "in_weight", "out_weight", "community"]].to_string(index=False))

print("\n=== Top-10 Sinks (in >> out) ===")
print(summary[summary["role"] == "Sink"].nlargest(10, "in_weight")
      [["route_id", "pagerank", "in_weight", "out_weight", "community"]].to_string(index=False))

In [ ]:
# ============================================================
# SARIMA + Ridge correction on residuals via parent routes
# Solo track: target_1h, horizon = 8 half-hour points
# ============================================================
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import Ridge
from tqdm import tqdm

warnings.filterwarnings("ignore")

# -------------------- Metric --------------------
def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    denom = y_true.sum() + 1e-9
    wape = np.abs(y_pred - y_true).sum() / denom
    rbias = np.abs(y_pred.sum() / denom - 1)
    return wape + rbias, wape, rbias

# -------------------- Config --------------------
TRACK = "solo"
TRAIN_DAYS = 14

SARIMA_ORDER          = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 0, 24)

TRACK_CONFIG = {
    "solo": {
        "train_path": "train_solo_track.parquet",
        "test_path": "test_solo_track.parquet",
        "target_col": "target_1h",
        "forecast_points": 8,
    },
    "team": {
        "train_path": "train_team_track.parquet",
        "test_path": "test_team_track.parquet",
        "target_col": "target_2h",
        "forecast_points": 10,
    },
}

cfg            = TRACK_CONFIG[TRACK]
TARGET_COL     = cfg["target_col"]
FORECAST_POINTS = cfg["forecast_points"]

# -------------------- Graph config --------------------
TOP_K_PARENTS  = 3     # сколько родителей на маршрут
MIN_SCORE      = 0  # порог силы связи
RIDGE_ALPHA    = 10.0  # регуляризация Ridge

# -------------------- Data --------------------
train_df = pd.read_parquet(cfg["train_path"])
test_df  = pd.read_parquet(cfg["test_path"])

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Routes:      {train_df['route_id'].nunique()}")

# -------------------- Graph: загрузка родителей --------------------
edges_df = pd.read_csv("route_dependency_edges.csv")
edges_df["src"] = edges_df["src"].astype(int)
edges_df["dst"] = edges_df["dst"].astype(int)

parents_dict = {}  # {route_id: [(parent_id, lag, score), ...]}
for dst, grp in (
    edges_df[edges_df["score"] >= MIN_SCORE]
    .sort_values(["dst", "score"], ascending=[True, False])
    .groupby("dst")
):
    parents_dict[int(dst)] = [
        (int(r["src"]), int(r["best_lag"]), float(r["score"]))
        for _, r in grp.head(TOP_K_PARENTS).iterrows()
    ]

routes_with_parents = len(parents_dict)
print(f"Routes with parents: {routes_with_parents}")

# -------------------- Wide target matrix --------------------
# Нужна для быстрого доступа к значениям родителей по любому timestamp
cutoff_wide = train_df["timestamp"].max() - pd.Timedelta(days=TRAIN_DAYS)
train_recent = train_df[train_df["timestamp"] > cutoff_wide].copy()

target_wide = (
    train_recent
    .pivot(index="timestamp", columns="route_id", values=TARGET_COL)
    .sort_index()
    .asfreq("30min")
    .interpolate(method="time")
    .ffill().bfill()
)

# -------------------- Helpers --------------------
def get_parent_signal(ts, parents, target_wide):
    """
    Для момента ts: возвращает dict {col_name: normalized_value} по каждому родителю.
    Берём значение родителя в момент (ts - lag × 30min) — это прошлое, известно.
    """
    signals = {}
    for parent_id, lag, score in parents:
        if parent_id not in target_wide.columns:
            continue
        ps = target_wide[parent_id]
        mu, std = float(ps.mean()), float(ps.std())
        if std < 1e-8:
            continue
        lookup_ts = ts - pd.Timedelta(minutes=30 * lag)
        if lookup_ts in ps.index:
            v = float(ps[lookup_ts])
        elif lookup_ts <= ps.index[-1]:
            v = float(ps.asof(lookup_ts))
        else:
            v = float(ps.iloc[-1])
        signals[f"p{parent_id}_lag{lag}"] = (v - mu) / std
    return signals

def build_parent_matrix(timestamps, parents, target_wide):
    """Матрица сигналов (n_timestamps × K) для обучения/применения Ridge."""
    rows = []
    cols = None
    for ts in timestamps:
        sig = get_parent_signal(ts, parents, target_wide)
        if cols is None:
            cols = list(sig.keys())
        rows.append([sig.get(c, 0.0) for c in cols])
    return np.array(rows), cols or []

# -------------------- SARIMA fit: возвращает forecast + residuals --------------------
def sarima_fit(route_history_df, future_timestamps):
    future_timestamps = pd.to_datetime(pd.Series(future_timestamps)).sort_values()
    future_index = pd.DatetimeIndex(future_timestamps)
    has_future = len(future_index) > 0

    route_train_30 = (
        route_history_df
        .set_index("timestamp")[TARGET_COL]
        .sort_index()
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
    )

    empty = pd.Series(dtype=float)

    if len(route_train_30) == 0:
        return pd.Series(dtype=float), empty, empty, empty, False

    cutoff = route_train_30.index.max() - pd.Timedelta(days=TRAIN_DAYS)
    route_train_30 = route_train_30[route_train_30.index > cutoff]

    if len(route_train_30) < 96:
        mean_val = float(route_train_30.mean())
        preds = pd.Series(mean_val, index=future_index) if has_future else pd.Series(dtype=float)
        return preds, empty, empty, empty, False

    route_train_1h = route_train_30.resample("1h").mean().interpolate(method="time")
    last_train_ts  = route_train_1h.index[-1]

    # n_hours нужен только если есть что прогнозировать
    if has_future:
        max_future_ts = future_index.max().ceil("1h")
        n_hours = int((max_future_ts - last_train_ts) / pd.Timedelta("1h")) + 2
    else:
        n_hours = 0

    try:
        model = SARIMAX(
            route_train_1h,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False, maxiter=200, method="lbfgs")

        fitted_1h    = result.fittedvalues
        residuals_1h = route_train_1h - fitted_1h

        if has_future:
            hourly_idx  = pd.date_range(
                start=last_train_ts + pd.Timedelta("1h"),
                periods=n_hours, freq="1h",
            )
            forecast_1h = pd.Series(result.forecast(steps=n_hours).values, index=hourly_idx)
            forecast_30 = forecast_1h.resample("30min").interpolate(method="linear")
            preds = pd.Series(
                [max(0.0, float(forecast_30.get(ts, forecast_30.iloc[-1]))) for ts in future_index],
                index=future_index,
            )
        else:
            preds = pd.Series(dtype=float)

        return preds, route_train_1h, fitted_1h, residuals_1h, True

    except Exception:
        if has_future:
            fallback_preds = []
            for ts in future_index:
                h, m = ts.hour, ts.minute
                mask = (route_train_30.index.hour == h) & (route_train_30.index.minute == m)
                fb = route_train_30[mask].mean() if mask.any() else route_train_30.mean()
                fallback_preds.append(max(0.0, float(fb)))
            preds = pd.Series(fallback_preds, index=future_index)
        else:
            preds = pd.Series(dtype=float)
        return preds, empty, empty, empty, False
        
# ============================================================
# PASS 1: обучаем SARIMA по всем маршрутам, собираем residuals
#         + обучаем Ridge для каждого маршрута с родителями
# ============================================================
print("\n── Pass 1: fit SARIMA + Ridge per route ──")

route_ids    = train_df["route_id"].unique()
ridge_models = {}           # {route_id: (Ridge, col_names)}
sarima_cache = {}           # {route_id: (train_1h, residuals_1h)} — для валидации

for route_id in tqdm(route_ids, desc="SARIMA+Ridge fit"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp").copy()
    parents     = parents_dict.get(route_id, [])

    # --- full-train SARIMA для получения residuals ---
    _, train_1h, fitted_1h, residuals_1h, success = sarima_fit(route_train, [])

    if not success or residuals_1h.empty or len(residuals_1h) < 30:
        continue

    sarima_cache[route_id] = (train_1h, residuals_1h)

    if not parents:
        continue

    # --- строим parent-сигналы на 1h-шкале train ---
    # конвертируем 1h-индекс обратно в 30min-lookup моменты
    ts_list = residuals_1h.index
    X, col_names = build_parent_matrix(ts_list, parents, target_wide)

    if X.shape[0] < 20 or X.shape[1] == 0:
        continue

    y = residuals_1h.values

    # --- NaN safety ---
    # shift(lag) даёт NaN в первых lag строках; убираем такие строки
    nan_mask = ~np.isnan(X).any(axis=1) & np.isfinite(y)
    X_clean  = X[nan_mask]
    y_clean  = y[nan_mask]

    if len(X_clean) < 20:
        continue

    ridge = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    ridge.fit(X_clean, y_clean)
    ridge_models[route_id] = (ridge, col_names)

print(f"Ridge models trained: {len(ridge_models)} routes")

# ============================================================
# Validation: последние FORECAST_POINTS точек как local test
# ============================================================
print("\n── Validation ──")
val_rows = []

for route_id in tqdm(route_ids, desc="Val"):
    route_df = train_df[train_df["route_id"] == route_id].sort_values("timestamp").reset_index(drop=True)

    if len(route_df) <= FORECAST_POINTS + 100:
        continue

    val_part  = route_df.tail(FORECAST_POINTS).copy()
    hist_part = route_df.iloc[:-FORECAST_POINTS].copy()
    parents   = parents_dict.get(route_id, [])

    # --- SARIMA на hist_part ---
    pred_series, _, _, _, _ = sarima_fit(hist_part, val_part["timestamp"].values)

    # --- Ridge correction ---
    ridge_info = ridge_models.get(route_id)
    if ridge_info and parents:
        ridge, col_names = ridge_info
        for i, ts in enumerate(val_part["timestamp"].values):
            ts = pd.Timestamp(ts)
            sig = get_parent_signal(ts, parents, target_wide)
            x_row = np.array([[sig.get(c, 0.0) for c in col_names]])
            correction = float(ridge.predict(x_row)[0])
            pred_series.iloc[i] = max(0.0, pred_series.iloc[i] + correction)

    tmp = val_part[["route_id", "timestamp", TARGET_COL]].copy()
    tmp["y_pred"] = pred_series.values
    val_rows.append(tmp)

val_df     = pd.concat(val_rows, ignore_index=True)
y_true_val = val_df[TARGET_COL].values
y_pred_val = val_df["y_pred"].values

total_raw, wape_raw, rbias_raw = wape_rbias(y_true_val, y_pred_val)
print(f"\n── Val (raw) ──")
print(f"  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  total={total_raw:.4f}")

calib_scale = float(y_true_val.sum() / (y_pred_val.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_val * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_val, y_pred_cal)
print(f"── Val (calibrated, scale={calib_scale:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  total={total_cal:.4f}")

val_df["y_pred_cal"] = y_pred_cal
val_df.to_csv("sarima_ridge_validation.csv", index=False)
print("Saved: sarima_ridge_validation.csv")

# ============================================================
# PASS 2: full train → test predictions + Ridge correction
# ============================================================
print("\n── Pass 2: full train → test ──")

predictions_raw = {}
test_route_ids  = test_df["route_id"].unique()

for route_id in tqdm(test_route_ids, desc="SARIMA+Ridge test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp").copy()
    route_test  = test_df[test_df["route_id"] == route_id].sort_values("timestamp").copy()
    parents     = parents_dict.get(route_id, [])

    pred_series, _, _, _, _ = sarima_fit(route_train, route_test["timestamp"].values)

    # --- Ridge correction ---
    ridge_info = ridge_models.get(route_id)
    if ridge_info and parents:
        ridge, col_names = ridge_info
        for i, ts in enumerate(route_test["timestamp"].values):
            ts = pd.Timestamp(ts)
            sig = get_parent_signal(ts, parents, target_wide)
            x_row = np.array([[sig.get(c, 0.0) for c in col_names]])
            correction = float(ridge.predict(x_row)[0])
            pred_series.iloc[i] = max(0.0, pred_series.iloc[i] + correction)

    for (_, row), pred in zip(route_test.iterrows(), pred_series.values):
        predictions_raw[row["id"]] = max(0.0, float(pred))

# --- Submissions ---
submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_raw.to_csv("submission_sarima_ridge_raw.csv", index=False)

submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)
submission_cal.to_csv("submission_sarima_ridge_calibrated.csv", index=False)

print("\n✅ Saved:")
print("  submission_sarima_ridge_raw.csv")
print("  submission_sarima_ridge_calibrated.csv")

print("\nRaw submission stats:")
print(submission_raw["y_pred"].describe().round(2))
print("\nCalibrated submission stats:")
print(submission_cal["y_pred"].describe().round(2))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

train = pd.read_parquet("train_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

STATUS_COLS = [f"status_{i}" for i in range(1, 7)]
TARGET = "target_1h"
val_cutoff = train["timestamp"].max() - pd.Timedelta(days=14)

def wape(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    return np.abs(yp - yt).sum() / (yt.sum() + 1e-9)

# Baseline для сравнения
route_mean = (train[train["timestamp"] <= val_cutoff]
              .groupby("route_id")[TARGET].mean())
val_all = train[train["timestamp"] > val_cutoff].copy()
val_all["pred_mean"] = val_all["route_id"].map(route_mean)
baseline_wape = wape(val_all[TARGET].values, val_all["pred_mean"].values)
print(f"Baseline WAPE (route mean): {baseline_wape:.4f}\n")

# ── Берём один маршрут для визуализации ──
r = train[train["route_id"] == 410].sort_values("timestamp").copy()
r_val = r[r["timestamp"] > val_cutoff].copy()

# ════════════════════════════════════════════════════════
# ГИПОТЕЗА 1: правильная единица — сумма status_3 за 1 час
# (target_1h = за час, а status за 30 мин)
# ════════════════════════════════════════════════════════
print("══ Гипотеза 1: status_3_1h = status_3 + status_3.shift(2) ══")
conv = (train.groupby("route_id")
        .apply(lambda g: g[TARGET].sum() / (g["status_3"].sum() + 1e-9))
        .rename("cf").reset_index())
train = train.merge(conv, on="route_id")

for lag in [1, 2, 4, 8]:
    train["s3_1h"] = (
        train.groupby("route_id")["status_3"]
        .transform(lambda x: x.shift(lag) + x.shift(lag + 2))
        * train["cf"] / 2
    )
    val2 = train[train["timestamp"] > val_cutoff].dropna(subset=["s3_1h"])
    w = wape(val2[TARGET].values, val2["s3_1h"].values)
    print(f"  lag={lag} ({lag*0.5:.1f}h):  WAPE={w:.4f}")

# ════════════════════════════════════════════════════════
# ГИПОТЕЗА 2: нелинейная связь — квантильные бины
# Делим status_3 на децили → смотрим средний таргет в каждом
# ════════════════════════════════════════════════════════
print("\n══ Гипотеза 2: нелинейная связь (квантильные бины) ══")
r_nonzero = r[r["status_3"] > 0].copy()
r_nonzero["s3_bin"] = pd.qcut(r_nonzero["status_3"], q=10, labels=False, duplicates="drop")
bin_stats = r_nonzero.groupby("s3_bin").agg(
    s3_mean=("status_3", "mean"),
    target_mean=(TARGET, "mean"),
    count=("status_3", "count")
).round(0)
print(bin_stats.to_string())

# ════════════════════════════════════════════════════════
# ГИПОТЕЗА 3: условная связь — только рабочие часы
# ════════════════════════════════════════════════════════
print("\n══ Гипотеза 3: корреляция в рабочие часы (8-20) vs ночь (0-8) ══")
r["hour"] = r["timestamp"].dt.hour
for name, mask in [("рабочие (8-20)", (r["hour"] >= 8) & (r["hour"] < 20)),
                   ("ночь (20-8)",     (r["hour"] < 8)  | (r["hour"] >= 20))]:
    sub = r[mask]
    corr = sub["status_3"].corr(sub[TARGET])
    corr_lag2 = sub["status_3"].shift(2).corr(sub[TARGET])
    print(f"  {name}: corr(lag=0)={corr:.4f}  corr(lag=2)={corr_lag2:.4f}  "
          f"n={len(sub)}  target_mean={sub[TARGET].mean():.0f}")

# ════════════════════════════════════════════════════════
# ГИПОТЕЗА 4: остаточная корреляция после удаления сезонности
# ════════════════════════════════════════════════════════
print("\n══ Гипотеза 4: остаточная корреляция (deseasonalized) ══")
r2 = r.copy()
r2["hh"] = r2["timestamp"].dt.hour * 2 + r2["timestamp"].dt.minute // 30
r2["dow"] = r2["timestamp"].dt.dayofweek

# Среднее по часу+деньнедели = сезонная компонента
season = r2.groupby(["dow", "hh"])[[TARGET, "status_3"]].transform("mean")
r2["target_resid"]  = r2[TARGET]   - season[TARGET]
r2["s3_resid"]      = r2["status_3"] - season["status_3"]

for lag in [0, 1, 2, 4, 8, 48]:
    c = r2["s3_resid"].shift(lag).corr(r2["target_resid"])
    print(f"  lag={lag:>3} ({lag*0.5:>4.1f}h):  residual corr = {c:.4f}")

# ════════════════════════════════════════════════════════
# ГИПОТЕЗА 5: бинарный признак "склад работает"
# ════════════════════════════════════════════════════════
print("\n══ Гипотеза 5: бинарный признак (any_status > 0) ══")
r["any_active"] = ((r[[c for c in STATUS_COLS if c in r.columns]] > 0).any(axis=1)).astype(int)
print(r.groupby("any_active")[TARGET].agg(["mean", "median", "count"]).round(0).to_string())

corr_bin = r["any_active"].corr(r[TARGET])
print(f"  Corr(any_active, target): {corr_bin:.4f}")

# ════════════════════════════════════════════════════════
# ГИПОТЕЗА 6: status_6 — финальный этап предыдущего склада
# ════════════════════════════════════════════════════════
print("\n══ Гипотеза 6: status_6 (финальный этап предыдущего склада) ══")
conv6 = (train.groupby("route_id")
         .apply(lambda g: g[TARGET].sum() / (g["status_6"].sum() + 1e-9))
         .rename("cf6").reset_index())
train = train.merge(conv6, on="route_id")

for lag in [1, 2, 4, 8, 16, 48, 96]:
    train["pred_s6"] = (
        train.groupby("route_id")["status_6"]
        .transform(lambda x: x.shift(lag)) * train["cf6"]
    )
    val3 = train[train["timestamp"] > val_cutoff].dropna(subset=["pred_s6"])
    w = wape(val3[TARGET].values, val3["pred_s6"].values)
    print(f"  lag={lag:>3} ({lag*0.5:>4.1f}h):  WAPE={w:.4f}")

# ════════════════════════════════════════════════════════
# Визуализация: scatter status_3 vs target по часам дня
# ════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

r["hour_group"] = pd.cut(r["timestamp"].dt.hour,
                          bins=[0, 4, 8, 12, 16, 20, 24],
                          labels=["0-4", "4-8", "8-12", "12-16", "16-20", "20-24"],
                          right=False)

for ax, (grp_name, grp) in zip(axes, r.groupby("hour_group", observed=True)):
    ax.scatter(grp["status_3"], grp[TARGET], alpha=0.2, s=8, color="steelblue")
    corr = grp["status_3"].corr(grp[TARGET])
    ax.set_title(f"Часы {grp_name}h | corr={corr:.3f} | n={len(grp)}")
    ax.set_xlabel("status_3")
    ax.set_ylabel("target_1h")
    ax.grid(alpha=0.3)

plt.suptitle("Route 410: status_3 vs target_1h по временным интервалам дня", fontsize=12)
plt.tight_layout()
plt.savefig("status_hypotheses.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

TARGET_COL      = "target_1h"
FORECAST_POINTS = 8
status_cols     = sorted([c for c in train_df.columns if c.startswith("status_")])
FUTURE_COLS     = [f"target_step_{s}" for s in range(1, FORECAST_POINTS + 1)]

route_group = train_df.groupby("route_id", sort=False)
for step in range(1, FORECAST_POINTS + 1):
    train_df[f"target_step_{step}"] = route_group[TARGET_COL].shift(-step)

supervised_df = train_df.dropna(subset=FUTURE_COLS).copy()

corr_cols = status_cols + FUTURE_COLS
corr = supervised_df[corr_cols].corr(numeric_only=True)

# ── Числовой вывод: только блок status × future_target ──
block = corr.loc[status_cols, FUTURE_COLS]
print("=== Correlation: status[t] → target[t+h] ===")
print(block.round(4).to_string())

# ── Дополнительно: auto-correlation target[t] → target[t+h] ──
auto_block = corr.loc[FUTURE_COLS, FUTURE_COLS]
print("\n=== Auto-correlation: target[t+h1] → target[t+h2] ===")
print(auto_block.round(4).to_string())

# ── Лучший лаг и лучший статус для каждого горизонта ──
print("\n=== Best status per horizon ===")
print(f"{'horizon':<12} {'best_status':<14} {'corr':>8}")
print("-" * 36)
for col in FUTURE_COLS:
    h = col.replace("target_step_", "h=")
    best_s = block[col].abs().idxmax()
    best_c = block.loc[best_s, col]
    print(f"{h:<12} {best_s:<14} {best_c:>8.4f}")

# ── Тепловая карта только status × future ──
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.heatmap(
    block,
    annot=True, fmt=".3f", cmap="coolwarm", center=0,
    vmin=-0.3, vmax=0.3,
    ax=axes[0],
)
axes[0].set_title("status[t] → target[t+h]")
axes[0].set_xlabel("Горизонт (h)")
axes[0].set_ylabel("Status")

# Линейный график: как меняется корреляция с горизонтом
for sc in status_cols:
    axes[1].plot(
        range(1, FORECAST_POINTS + 1),
        [block.loc[sc, f"target_step_{h}"] for h in range(1, FORECAST_POINTS + 1)],
        marker="o", label=sc, linewidth=1.5,
    )
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Горизонт h (шагов по 30 мин)")
axes[1].set_ylabel("Pearson corr")
axes[1].set_title("Корреляция статуса с будущим таргетом по горизонтам")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("status_future_corr.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

TARGET_COL      = "target_1h"
FORECAST_POINTS = 8
status_cols     = sorted([c for c in train_df.columns if c.startswith("status_")])
FUTURE_COLS     = [f"target_step_{s}" for s in range(1, FORECAST_POINTS + 1)]

route_group = train_df.groupby("route_id", sort=False)
for step in range(1, FORECAST_POINTS + 1):
    train_df[f"target_step_{step}"] = route_group[TARGET_COL].shift(-step)

supervised_df = train_df.dropna(subset=FUTURE_COLS).copy()

# ── Тест: кросс-секционная vs временная корреляция ──
# Деминим по route_id → убираем межмаршрутную вариацию
# Если корреляция после демина падает — она была кросс-секционной
print("=== Сырая корреляция (все маршруты вместе) ===")
raw_corr = supervised_df[status_cols + FUTURE_COLS].corr().loc[status_cols, FUTURE_COLS]
print(raw_corr.round(4).to_string())

print("\n=== Within-route корреляция (demeaned по route_id) ===")
demeaned = supervised_df.copy()
for col in status_cols + FUTURE_COLS:
    demeaned[col] = (demeaned[col]
                     - demeaned.groupby("route_id")[col].transform("mean"))

within_corr = demeaned[status_cols + FUTURE_COLS].corr().loc[status_cols, FUTURE_COLS]
print(within_corr.round(4).to_string())

print("\n=== Разница (raw - within) — сколько от кросс-секции ===")
print((raw_corr - within_corr).round(4).to_string())

# ── Дополнительно: within-route по одному маршруту ──
print("\n=== Корреляция только на маршруте 410 (чисто временная) ===")
r = supervised_df[supervised_df["route_id"] == 410]
single_corr = r[status_cols + FUTURE_COLS].corr().loc[status_cols, FUTURE_COLS]
print(single_corr.round(4).to_string())

# ── Визуально: сравнение raw vs within ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (title, mat) in zip(axes, [
    ("Raw corr (все маршруты)",  raw_corr),
    ("Within-route corr",         within_corr),
    ("Single route 410",          single_corr),
]):
    im = ax.imshow(mat.values, cmap="coolwarm", vmin=-0.3, vmax=0.6, aspect="auto")
    ax.set_xticks(range(len(FUTURE_COLS)))
    ax.set_xticklabels([f"h={i+1}" for i in range(len(FUTURE_COLS))], fontsize=9)
    ax.set_yticks(range(len(status_cols)))
    ax.set_yticklabels(status_cols, fontsize=9)
    for i in range(len(status_cols)):
        for j in range(len(FUTURE_COLS)):
            ax.text(j, i, f"{mat.values[i,j]:.2f}",
                    ha="center", va="center", fontsize=7,
                    color="white" if abs(mat.values[i,j]) > 0.3 else "black")
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig("corr_raw_vs_within.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
df_chronos = pd.read_csv("submission_chronos2_calibrated.csv")
df_lgb = pd.read_csv("submission_lgb_direct.csv")

In [ ]:
df_mixed = df_chronos.copy()
df_mixed["y_pred"] = df_mixed["y_pred"]*0.5 + df_lgb["y_pred"]*0.5

In [ ]:
df_mixed.to_csv("submission_chronos2_lgb_calibrated_blending.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

train = pd.read_parquet("train_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])

train.head()

In [ ]:
last_times = set(sorted(train.timestamp.unique())[-8:])

In [ ]:
grouped = train[train["timestamp"].isin(last_times)].groupby("route_id", as_index=False)["target_1h"].agg(["mean", "median", "std"])

In [ ]:
grouped.sort_values(by="mean", ascending=False).head()

In [ ]:
grouped.sort_values(by="mean", ascending=False)["mean"].hist()

In [ ]:
grouped["std"].hist()

In [ ]:
grouped["std_frac"] = grouped["std"] / grouped["mean"]
grouped.sort_values(by="mean", ascending=False).head(50)

In [ ]:
# ============================================================
# Blending: LGB + Chronos-2
# Вариант 1: Hard switch (где Chronos хуже → берём LGB)
# Вариант 2: Weighted blend (веса пропорциональны качеству)
# ============================================================
import numpy as np
import pandas as pd

# -------------------- Загружаем CV-статистики --------------------
lgb_route   = pd.read_csv("lgb_cv_agg_route.csv")[["route_id", "wape", "calib_scale"]]\
                .rename(columns={"wape": "wape_lgb", "calib_scale": "calib_lgb"})
chr_route   = pd.read_csv("chronos2_cv_agg_route.csv")[["route_id", "wape", "calib_scale"]]\
                .rename(columns={"wape": "wape_chr", "calib_scale": "calib_chr"})

route_stats = lgb_route.merge(chr_route, on="route_id", how="inner")
print(f"Маршрутов с обеими оценками: {len(route_stats)}")

# Кто лучше на каждом маршруте
route_stats["lgb_wins"]  = route_stats["wape_lgb"] <= route_stats["wape_chr"]
route_stats["delta_wape"] = route_stats["wape_chr"] - route_stats["wape_lgb"]  # >0 = chronos хуже

print(f"\nLGB лучше:     {route_stats['lgb_wins'].sum()} маршрутов")
print(f"Chronos лучше: {(~route_stats['lgb_wins']).sum()} маршрутов")
print(f"\nСредний WAPE LGB:     {route_stats['wape_lgb'].mean():.4f}")
print(f"Средний WAPE Chronos: {route_stats['wape_chr'].mean():.4f}")

# -------------------- Загружаем сабмиты --------------------
# test_df нужен чтобы знать route_id для каждого id
test_df = pd.read_parquet("test_solo_track.parquet")
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

sub_lgb = pd.read_csv("submission_lgb_direct.csv")\
            .rename(columns={"y_pred": "y_pred_lgb"})
sub_chr = pd.read_csv("submission_chronos2_calibrated.csv")\
            .rename(columns={"y_pred": "y_pred_chr"})

# Джойним с route_id
sub = (
    test_df[["id", "route_id"]]
    .merge(sub_lgb, on="id")
    .merge(sub_chr, on="id")
    .merge(route_stats[["route_id", "wape_lgb", "wape_chr", "lgb_wins", "delta_wape"]], on="route_id", how="left")
)

# Маршруты без CV-статистики (мало точек) → берём LGB как дефолт
sub["wape_lgb"]  = sub["wape_lgb"].fillna(0.3)
sub["wape_chr"]  = sub["wape_chr"].fillna(0.5)
sub["lgb_wins"]  = sub["lgb_wins"].fillna(True)

print(f"\nВсего строк в сабмите: {len(sub)}")

# ============================================================
# Вариант 1: HARD SWITCH
# Где LGB лучше → LGB, иначе → Chronos
# ============================================================
sub["y_pred_hard"] = np.where(
    sub["lgb_wins"],
    sub["y_pred_lgb"],
    sub["y_pred_chr"],
)

# Дополнительно: можно сделать порог — переключаемся только если
# разница в WAPE существенная (>0.02), иначе берём среднее
THRESHOLD = 0.02
sub["y_pred_hard_thresh"] = np.where(
    sub["delta_wape"].fillna(0) > THRESHOLD,   # Chronos хуже на >0.02 → LGB
    sub["y_pred_lgb"],
    np.where(
        sub["delta_wape"].fillna(0) < -THRESHOLD,  # LGB хуже на >0.02 → Chronos
        sub["y_pred_chr"],
        (sub["y_pred_lgb"] + sub["y_pred_chr"]) / 2,  # примерно одинаково → среднее
    )
)

# ============================================================
# Вариант 2: WEIGHTED BLEND
# Вес пропорционален качеству: чем ниже WAPE → больший вес
# w_lgb = wape_chr / (wape_lgb + wape_chr)
# w_chr = wape_lgb / (wape_lgb + wape_chr)
# (лучшая модель получает больший вес)
# ============================================================
wape_sum = sub["wape_lgb"] + sub["wape_chr"] + 1e-9
sub["w_lgb"] = sub["wape_chr"] / wape_sum
sub["w_chr"] = sub["wape_lgb"] / wape_sum

sub["y_pred_weighted"] = (
    sub["w_lgb"] * sub["y_pred_lgb"] +
    sub["w_chr"] * sub["y_pred_chr"]
).clip(lower=0)

# ============================================================
# Аналитика весов
# ============================================================
print("\n── Распределение весов LGB (weighted blend) ──")
print(sub.groupby("route_id")["w_lgb"].first().describe().round(3))

print("\n── Топ-10 маршрутов где Chronos сильно лучше LGB ──")
top_chr = route_stats.sort_values("delta_wape").head(10)
print(top_chr[["route_id", "wape_lgb", "wape_chr", "delta_wape"]].to_string(index=False))

print("\n── Топ-10 маршрутов где LGB сильно лучше Chronos ──")
top_lgb = route_stats.sort_values("delta_wape", ascending=False).head(10)
print(top_lgb[["route_id", "wape_lgb", "wape_chr", "delta_wape"]].to_string(index=False))

# ============================================================
# Сохраняем все варианты
# ============================================================
def save_sub(col, fname):
    out = sub[["id", col]].rename(columns={col: "y_pred"})
    out = out.sort_values("id").reset_index(drop=True)
    out.to_csv(fname, index=False)
    print(f"✅ {fname}  |  mean={out['y_pred'].mean():.3f}  "
          f"std={out['y_pred'].std():.3f}  "
          f"zeros={( out['y_pred']==0).sum()}")

print("\n── Сабмиты ──")
save_sub("y_pred_lgb",         "submission_lgb_only.csv")
save_sub("y_pred_chr",         "submission_chr_only.csv")
save_sub("y_pred_hard",        "submission_blend_hard.csv")
save_sub("y_pred_hard_thresh", "submission_blend_hard_thresh.csv")
save_sub("y_pred_weighted",    "submission_blend_weighted.csv")

# ============================================================
# Сравнение на CV (псевдо-оценка через CV WAPE)
# ============================================================
print("\n── Ожидаемое качество по CV (средневзвешенный WAPE) ──")
# Взвешиваем по числу тест-точек на маршруте
route_counts = sub.groupby("route_id")["id"].count().reset_index()\
                  .rename(columns={"id": "n_test"})
rs = route_stats.merge(route_counts, on="route_id", how="left").fillna({"n_test": 8})

for label, w_lgb_col in [("Hard switch",    rs["lgb_wins"].astype(float)),
                          ("Weighted blend", rs["wape_chr"] / (rs["wape_lgb"] + rs["wape_chr"] + 1e-9))]:
    expected_wape = (
        (w_lgb_col * rs["wape_lgb"] + (1 - w_lgb_col) * rs["wape_chr"]) * rs["n_test"]
    ).sum() / rs["n_test"].sum()
    print(f"  {label:20s}  expected WAPE ≈ {expected_wape:.4f}")

lgb_base  = (rs["wape_lgb"] * rs["n_test"]).sum() / rs["n_test"].sum()
chr_base  = (rs["wape_chr"] * rs["n_test"]).sum() / rs["n_test"].sum()
print(f"  {'LGB only':20s}  expected WAPE ≈ {lgb_base:.4f}")
print(f"  {'Chronos only':20s}  expected WAPE ≈ {chr_base:.4f}")

In [ ]:
lgb_route   = pd.read_csv("sarima_cv_agg_route.csv")[["route_id", "total", "calib_scale"]]\
                .rename(columns={"total": "total_lgb", "calib_scale": "calib_lgb"})
chr_route   = pd.read_csv("chronos2_cv_agg_route.csv")[["route_id", "total", "calib_scale"]]\
                .rename(columns={"total": "total_chr", "calib_scale": "calib_chr"})
route_stats = lgb_route.merge(chr_route, on="route_id", how="inner")
route_stats.head()

In [ ]:
route_stats["mae_diff"] = route_stats["mae_chr"] - route_stats["mae_lgb"]
sarima_best = route_stats.sort_values(by="mae_diff", ascending=False).head(5)["route_id"].tolist()

In [ ]:
test = pd.read_parquet("/Users/ruaczcf/Downloads/wruuQNI2/test_solo_track.parquet")
test.head()

In [ ]:
sarima_cal = pd.read_csv("/Users/ruaczcf/Downloads/wruuQNI2/submission_sarima_calibrated.csv")
chronos_cal = pd.read_csv("/Users/ruaczcf/Downloads/wruuQNI2/submission_chronos2_calibrated.csv")
sarima_cal_routes = sarima_cal.merge(test, how="inner", on="id")
chronos_cal_routes = chronos_cal.merge(test, how="inner", on="id")

In [ ]:
set(sarima_best) - set(routes_to_swap)

In [ ]:
united_submission = pd.concat(
    [sarima_cal_routes[sarima_cal_routes.route_id.isin(routes_to_swap)],
    chronos_cal_routes[~chronos_cal_routes.route_id.isin(routes_to_swap)]]
)
united_submission.head()

In [ ]:
sarima_cal_routes[sarima_cal_routes.route_id.isin(sarima_best)].head()

In [ ]:
chronos_cal_routes[chronos_cal_routes.route_id.isin(sarima_best)].head()

In [ ]:
united_submission[["id", "y_pred"]].sort_values(by="id").to_csv("chronos_sarima_united_horizon1-4.csv")

In [ ]:
united_submission[["id", "y_pred"]].sort_values(by="id").head()

In [ ]:
 # Загружаем сырые CV-данные обеих моделей
lgb_raw = pd.read_csv("sarima_cv_raw.csv")[["route_id", "timestamp", "h", "y_true", "y_pred"]]\
            .rename(columns={"y_pred": "y_pred_lgb"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id", "timestamp", "h", "y_true", "y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})

merged = lgb_raw.merge(chr_raw, on=["route_id", "timestamp", "h", "y_true"], how="inner")

merged.head()

In [ ]:
# Глобальный bias = среднее из двух моделей
global_bias_lgb = -(-8945)  # нужно добавить ~8945
global_bias_chr = -(-8216)  # нужно добавить ~8216

# Для Chronos: пересчитай calib_scale с учётом bias
# calib_scale уже должен это делать — проверь его значение
print(f"calib_scale Chronos: {calib_scale:.4f}")
# Если < 1.0 → модель занижает, scale < 1 это неправильно
# Должно быть > 1.0 чтобы поднять прогноз

In [ ]:
# ============================================================
# Blending: SARIMA + Chronos-2 (+ LGB для тройного бленда)
# Стратегия: blend → calibrate (не calibrate → blend)
# ============================================================
import numpy as np
import pandas as pd

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- CV данные --------------------
sar_raw = pd.read_csv("sarima_cv_raw.csv")[["route_id","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_sar"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})
lgb_raw = pd.read_csv("lgb_cv_raw.csv")[["route_id","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_lgb"})

cv = sar_raw\
    .merge(chr_raw, on=["route_id","timestamp","h","y_true"], how="inner")\
    .merge(lgb_raw, on=["route_id","timestamp","h","y_true"], how="inner")

print(f"CV точек для оценки бленда: {len(cv)}")

# -------------------- Сабмиты --------------------
test_df  = pd.read_parquet("test_solo_track.parquet")
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])
test_df = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

sub_sar = pd.read_csv("submission_sarima_raw.csv").rename(columns={"y_pred":"y_pred_sar"})
sub_chr = pd.read_csv("submission_chronos2_raw.csv").rename(columns={"y_pred":"y_pred_chr"})
sub_lgb = pd.read_csv("submission_lgb_direct.csv").rename(columns={"y_pred":"y_pred_lgb"})

sub = test_df[["id","route_id"]]\
    .merge(sub_sar, on="id")\
    .merge(sub_chr, on="id")\
    .merge(sub_lgb, on="id")

# ============================================================
# Поиск оптимального веса SARIMA+Chronos по CV (grid search)
# ============================================================
print("\n── Grid search: оптимальный вес SARIMA (Chronos = 1 - w) ──")
best_w, best_total = None, 1e9
results = []

for w_sar in np.arange(0.0, 1.01, 0.05):
    w_chr = 1 - w_sar
    y_blend = cv["y_pred_sar"] * w_sar + cv["y_pred_chr"] * w_chr

    # calib_scale на CV
    cs = float(cv["y_true"].sum() / (y_blend.clip(lower=0).sum() + 1e-9))
    y_blend_cal = np.clip(y_blend * cs, 0, None)

    tot, wape, rb = wape_rbias(cv["y_true"].values, y_blend_cal.values)
    results.append({"w_sar": w_sar, "w_chr": w_chr, "calib_scale": cs,
                    "wape": wape, "rbias": rb, "total": tot})
    if tot < best_total:
        best_total = tot
        best_w = w_sar

grid_df = pd.DataFrame(results)
print(grid_df[["w_sar","w_chr","calib_scale","wape","rbias","total"]].to_string(index=False))
print(f"\n✅ Оптимальный w_sar={best_w:.2f}  w_chr={1-best_w:.2f}  total={best_total:.4f}")

# Финальный calib_scale при оптимальных весах
opt_blend_cv = cv["y_pred_sar"] * best_w + cv["y_pred_chr"] * (1 - best_w)
CALIB_SCALE_2 = float(cv["y_true"].sum() / (opt_blend_cv.clip(lower=0).sum() + 1e-9))
print(f"  calib_scale (2-model blend) = {CALIB_SCALE_2:.4f}")

# ============================================================
# Тройной бленд: SARIMA + Chronos + LGB
# Перебираем сетку w_sar + w_chr + w_lgb = 1
# ============================================================
print("\n── Grid search: тройной бленд (шаг 0.1) ──")
best3, best3_total = None, 1e9
rows3 = []

for w_sar in np.arange(0.0, 1.01, 0.1):
    for w_chr in np.arange(0.0, 1.01 - w_sar, 0.1):
        w_lgb = round(1 - w_sar - w_chr, 2)
        if w_lgb < 0:
            continue
        y_blend = (cv["y_pred_sar"] * w_sar +
                   cv["y_pred_chr"] * w_chr +
                   cv["y_pred_lgb"] * w_lgb)
        cs  = float(cv["y_true"].sum() / (y_blend.clip(lower=0).sum() + 1e-9))
        ybc = np.clip(y_blend * cs, 0, None)
        tot, wape, rb = wape_rbias(cv["y_true"].values, ybc.values)
        rows3.append({"w_sar": w_sar, "w_chr": w_chr, "w_lgb": w_lgb,
                      "calib_scale": cs, "wape": wape, "rbias": rb, "total": tot})
        if tot < best3_total:
            best3_total = tot
            best3 = {"w_sar": w_sar, "w_chr": w_chr, "w_lgb": w_lgb, "cs": cs}

grid3_df = pd.DataFrame(rows3).sort_values("total")
print(grid3_df.head(15)[["w_sar","w_chr","w_lgb","calib_scale","wape","rbias","total"]]
      .to_string(index=False))
print(f"\n✅ Лучший тройной бленд: {best3}  total={best3_total:.4f}")

# ============================================================
# Сравнение всех вариантов на CV
# ============================================================
print("\n" + "=" * 65)
print("СРАВНЕНИЕ ВАРИАНТОВ НА CV")
print("=" * 65)

candidates = {
    "SARIMA raw":          cv["y_pred_sar"],
    "Chronos raw":         cv["y_pred_chr"],
    "LGB raw":             cv["y_pred_lgb"],
}
for label, yp in candidates.items():
    cs  = float(cv["y_true"].sum() / (yp.clip(lower=0).sum() + 1e-9))
    ypc = np.clip(yp * cs, 0, None)
    tot, wape, rb = wape_rbias(cv["y_true"].values, ypc.values)
    print(f"  {label:30s}  WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}  scale={cs:.3f}")

# Лучший 2-бленд
y2 = cv["y_pred_sar"] * best_w + cv["y_pred_chr"] * (1 - best_w)
yc2 = np.clip(y2 * CALIB_SCALE_2, 0, None)
t, w, r = wape_rbias(cv["y_true"].values, yc2.values)
print(f"  {'SARIMA+Chronos blend+calib':30s}  WAPE={w:.4f}  |RBias|={r:.4f}  Total={t:.4f}  scale={CALIB_SCALE_2:.3f}")

# Лучший 3-бленд
y3 = (cv["y_pred_sar"] * best3["w_sar"] +
      cv["y_pred_chr"] * best3["w_chr"] +
      cv["y_pred_lgb"] * best3["w_lgb"])
yc3 = np.clip(y3 * best3["cs"], 0, None)
t3, w3, r3 = wape_rbias(cv["y_true"].values, yc3.values)
print(f"  {'SAR+CHR+LGB blend+calib':30s}  WAPE={w3:.4f}  |RBias|={r3:.4f}  Total={t3:.4f}  scale={best3['cs']:.3f}")

# ============================================================
# САБМИТЫ
# ============================================================
def make_submit(df, w_sar, w_chr, w_lgb, cs, fname):
    y = (df["y_pred_sar"] * w_sar +
         df["y_pred_chr"] * w_chr +
         df["y_pred_lgb"] * w_lgb)
    out = df[["id"]].copy()
    out["y_pred"] = np.clip(y * cs, 0, None)
    out = out.sort_values("id").reset_index(drop=True)
    out.to_csv(fname, index=False)
    print(f"✅ {fname}  mean={out['y_pred'].mean():.1f}  std={out['y_pred'].std():.1f}")
    return out

print("\n── Сабмиты ──")
make_submit(sub, best_w,        1-best_w,        0,
            CALIB_SCALE_2, "submission_blend_sar_chr.csv")
make_submit(sub, best3["w_sar"], best3["w_chr"], best3["w_lgb"],
            best3["cs"],   "submission_blend_3models.csv")

# Для сравнения — каждая модель калиброванная отдельно
for col, fname in [("y_pred_sar", "submission_sarima_cal_cv.csv"),
                   ("y_pred_chr", "submission_chronos_cal_cv.csv"),
                   ("y_pred_lgb", "submission_lgb_cal_cv.csv")]:
    cs = float(cv["y_true"].sum() / (sub[col].clip(lower=0).sum() + 1e-9))
    out = sub[["id"]].copy()
    out["y_pred"] = np.clip(sub[col] * cs, 0, None)
    out.sort_values("id").reset_index(drop=True).to_csv(fname, index=False)
    print(f"✅ {fname}")

In [ ]:
import numpy as np
import pandas as pd

sar_raw = pd.read_csv("sarima_cv_raw.csv")[["route_id","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_sar"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})

cv = chr_raw.merge(sar_raw, on=["route_id","timestamp","h","y_true"], how="inner")

# ============================================================
# Анализ: насколько разные модели по горизонтам
# ============================================================
print("── WAPE по горизонтам: Chronos vs SARIMA ──")
for h in range(1, 9):
    g = cv[cv["h"] == h]
    s = g["y_true"].sum() + 1e-9
    wape_chr = g["y_pred_chr"].sub(g["y_true"]).abs().sum() / s
    wape_sar = g["y_pred_sar"].sub(g["y_true"]).abs().sum() / s
    winner = "SAR ✅" if wape_sar < wape_chr else "CHR ✅"
    print(f"  h={h}  Chronos={wape_chr:.4f}  SARIMA={wape_sar:.4f}  → {winner}")

# ============================================================
# Жадный отбор маршрутов по h=1..4 (гипотеза паблик-теста)
# ============================================================
for HORIZON_FILTER, label in [
    ([1, 2, 3, 4], "h=1..4 (паблик гипотеза)"),
    # ([1],          "h=1 только"),
    # ([1, 2, 3, 4, 5, 6, 7, 8], "h=1..8 (полный)"),
]:
    cv_h = cv[cv["h"].isin(HORIZON_FILTER)].copy()

    SUM_TRUE     = cv_h["y_true"].sum()
    base_sum_ae  = cv_h["y_pred_chr"].sub(cv_h["y_true"]).abs().sum()
    base_sum_pred = cv_h["y_pred_chr"].sum()
    base_wape    = base_sum_ae / SUM_TRUE
    base_rbias   = abs(base_sum_pred / SUM_TRUE - 1)
    base_total   = base_wape + base_rbias

    route_impact = []
    for route_id, grp in cv_h.groupby("route_id"):
        ae_chr   = grp["y_pred_chr"].sub(grp["y_true"]).abs().sum()
        ae_sar   = grp["y_pred_sar"].sub(grp["y_true"]).abs().sum()
        pred_chr = grp["y_pred_chr"].sum()
        pred_sar = grp["y_pred_sar"].sum()

        new_sum_ae   = base_sum_ae   - ae_chr  + ae_sar
        new_sum_pred = base_sum_pred - pred_chr + pred_sar
        new_total    = new_sum_ae / SUM_TRUE + abs(new_sum_pred / SUM_TRUE - 1)

        route_impact.append({
            "route_id":    route_id,
            "delta_ae":    ae_sar - ae_chr,
            "delta_pred":  pred_sar - pred_chr,
            "delta_total": new_total - base_total,
        })

    impact_df = pd.DataFrame(route_impact).sort_values("delta_total")

    # Жадный отбор
    routes_to_swap = []
    cur_ae   = base_sum_ae
    cur_pred = base_sum_pred

    for _, row in impact_df.iterrows():
        if row["delta_total"] >= -0.00010:
            break
        grp = cv_h[cv_h["route_id"] == row["route_id"]]
        cur_ae   += grp["y_pred_sar"].sub(grp["y_true"]).abs().sum() \
                  - grp["y_pred_chr"].sub(grp["y_true"]).abs().sum()
        cur_pred += grp["y_pred_sar"].sum() - grp["y_pred_chr"].sum()
        routes_to_swap.append(row["route_id"])
        print(f"  +route {row['route_id']:>6}  delta_total={row['delta_total']:+.5f}  ")

    final_total = cur_ae / SUM_TRUE + abs(cur_pred / SUM_TRUE - 1)

    print(f"\n{'='*55}")
    print(f"Отбор по {label}")
    print(f"  Baseline total:  {base_total:.4f}")
    print(f"  After swap:      {final_total:.4f}  (Δ={final_total-base_total:+.4f})")
    print(f"  Маршрутов swap:  {len(routes_to_swap)}")
    print(f"  Маршруты: {sorted(routes_to_swap)}")

    impact_df.to_csv(f"route_swap_impact_{'h'+''.join(map(str,HORIZON_FILTER[:2]))}.csv",
                     index=False)

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json, os

# pio.templates.default = "perplexity"
os.makedirs("output", exist_ok=True)

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

# ============================================================
# 1. Загружаем данные
# ============================================================
chr_raw  = pd.read_csv("chronos2_cv_raw.csv")
train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40

# Берём только fold 5 — самый свежий
cv5 = chr_raw[chr_raw["fold"] == 5].copy()

# Добавляем timestamp если его нет в chr_raw — берём из train_df
if "timestamp" not in cv5.columns:
    # Последние 8 точек каждого маршрута = fold 5
    last_ts = (
        train_df.groupby("route_id")
        .tail(8)
        [["route_id","timestamp"]]
        .copy()
    )
    last_ts["h"] = last_ts.groupby("route_id").cumcount() + 1
    cv5 = cv5.merge(last_ts, on=["route_id","h"], how="left")

print(f"cv5: {cv5.shape}")
print(cv5.columns.tolist())
print(cv5.head(3))

In [ ]:
# ============================================================
# Калибровка: cs считаем на фолдах 1-4, применяем на fold 5
# ============================================================
lgb_raw = pd.read_csv("lgb_cv_raw.csv")[["route_id","fold","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_lgb"})
chr_raw_all = pd.read_csv("chronos2_cv_raw.csv")

cv_train_chr = chr_raw_all[chr_raw_all["fold"].isin([1,2,3,4])]

cs = float(
    cv_train_chr["y_true"].sum() /
    (cv_train_chr["y_pred"].clip(lower=0).sum() + 1e-9)
)
print(f"Calibration scalar cs = {cs:.4f}")

# Применяем к fold 5
cv5["y_pred"] = np.clip(cv5["y_pred"] * cs, 0, None)

# Проверяем что стало
base_raw = wape_rbias(cv5["y_true"].values, cv5["y_pred"].values)
# base_cal = wape_rbias(cv5["y_true"].values, cv5["y_pred_cal"].values)
print(f"WAPE fold5 raw:        {base_raw:.4f}")
# print(f"WAPE fold5 calibrated: {base_cal:.4f}")

In [ ]:
# ============================================================
# 2. Топ-15 самых проблемных маршрутов для Chronos (fold 5)
# ============================================================
route_wape = cv5.groupby("route_id").apply(
    lambda df: pd.Series({
        "wape_chr":  wape_rbias(df["y_true"], df["y_pred"]),
        "vol":       float(df["y_true"].sum()),
        "n":         len(df),
    })
).reset_index()

# Сортируем по wape, берём только маршруты с ненулевым объёмом
route_wape = route_wape[route_wape["vol"] > 0].sort_values("wape_chr", ascending=False)
TOP_N = 15
top_routes = route_wape.head(TOP_N)["route_id"].tolist()
print("Топ-15 проблемных маршрутов:")
print(route_wape.head(TOP_N)[["route_id","wape_chr","vol"]].to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# НАСТРОЙКИ
# ============================================================
TOP_N         = 15   # сколько маршрутов смотреть
N_HIST_LONG   = 720  # точек в верхнем subplot
N_HIST_SHORT  = 48   # точек контекста перед предиктом ← настраивай

TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

# ============================================================
# Загрузка
# ============================================================
chr_raw  = pd.read_csv("chronos2_cv_raw.csv")
train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

cv5 = chr_raw[chr_raw["fold"] == 5].copy()

# Достаём timestamp для fold 5 из train_df (последние 8 точек каждого маршрута)
fold5_ts = (
    train_df.groupby("route_id", group_keys=False)
    .tail(8)[["route_id","timestamp"]]
    .copy()
    .reset_index(drop=True)
)
fold5_ts["h"] = fold5_ts.groupby("route_id").cumcount() + 1
cv5 = cv5.drop(columns=["timestamp"], errors="ignore")
cv5 = cv5.merge(fold5_ts, on=["route_id","h"], how="left")

# ============================================================
# Топ-N проблемных маршрутов
# ============================================================
route_wape = (
    cv5.groupby("route_id")
    .apply(lambda df: pd.Series({
        "wape_chr": wape_rbias(df["y_true"], df["y_pred"]),
        "vol":      float(df["y_true"].sum()),
    }))
    .reset_index()
)
route_wape = route_wape[route_wape["vol"] > 0].sort_values("wape_chr", ascending=False)
top_routes = route_wape.head(TOP_N)["route_id"].tolist()
print(route_wape.head(TOP_N)[["route_id","wape_chr","vol"]].to_string(index=False))

# ============================================================
# Графики
# ============================================================
for route_id in top_routes:
    hist = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    pred = cv5[cv5["route_id"] == route_id].sort_values("h").reset_index(drop=True)
    wape_val = wape_rbias(pred["y_true"], pred["y_pred"])
    vol_val  = route_wape[route_wape["route_id"] == route_id]["vol"].values[0]

    n_long  = min(N_HIST_LONG,  len(hist) - TOTAL_VAL_PTS)
    n_short = min(N_HIST_SHORT, len(hist) - TOTAL_VAL_PTS)

    hist_long  = hist.iloc[-(TOTAL_VAL_PTS + n_long) : -TOTAL_VAL_PTS].reset_index(drop=True)
    FOLD5_PTS  = 8  # размер fold 5
    hist_short = hist.iloc[-(FOLD5_PTS + n_short): -FOLD5_PTS]

    # Собираем нижний subplot как один непрерывный ряд
    context_df = pd.DataFrame({
        "timestamp": hist_short["timestamp"],
        "y_true":    hist_short[TARGET_COL].values,
        "y_pred":    np.nan,
        "part":      "context",
    })
    pred_df = pd.DataFrame({
        "timestamp": pred["timestamp"],
        "y_true":    pred["y_true"].values,
        "y_pred":    pred["y_pred"].values,
        "part":      "pred",
    })
    # Стыковочная точка — последняя точка контекста добавляется в pred для непрерывности линии
    bridge = context_df.iloc[[-1]].copy()
    bridge["part"] = "pred"

    lower = pd.concat([context_df, bridge, pred_df], ignore_index=True).sort_values("timestamp")

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"История: последние {n_long} точек (~{n_long//24} дней)",
            f"Контекст ({n_short} т.) + предикт fold 5 (h=1..8)",
        ],
        vertical_spacing=0.20,
        row_heights=[0.40, 0.60],
    )

    # ── Верхний: история ──
    fig.add_trace(go.Scatter(
        x=hist_long["timestamp"].tolist(),
        y=hist_long[TARGET_COL].tolist(),
        mode="lines",
        name="История",
        line=dict(width=1.2),
    ), row=1, col=1)

    # ── Нижний: факт (контекст + fold 5) ──
    fig.add_trace(go.Scatter(
        x=lower["timestamp"].tolist(),
        y=lower["y_true"].tolist(),
        mode="lines+markers",
        name="Факт",
        line=dict(width=2),
        marker=dict(size=5),
    ), row=2, col=1)

    # ── Нижний: предикт Chronos (только на горизонте pred) ──
    fig.add_trace(go.Scatter(
        x=lower[lower["part"] == "pred"]["timestamp"].tolist(),
        y=lower[lower["part"] == "pred"]["y_pred"].tolist(),
        mode="lines+markers",
        name="Chronos",
        line=dict(width=2, dash="dash"),
        marker=dict(size=7, symbol="diamond"),
    ), row=2, col=1)

    # Граница контекст/предикт — через annotation, не vline
    split_ts = pred["timestamp"].iloc[0]
    fig.add_annotation(
        x=split_ts, y=1.02,
        xref="x2", yref="paper",
        text="▼ предикт",
        showarrow=False,
        font=dict(size=11, color="gray"),
    )

    fig.update_layout(
        title=dict(
            text=(
                f"Маршрут {route_id}  ·  WAPE={wape_val:.2f}  ·  vol={vol_val:.0f}"
            )
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.05,
                    xanchor="center", x=0.5),
        height=680,
    )
    fig.update_xaxes(title_text="Время", row=1, col=1)
    fig.update_xaxes(title_text="Время", row=2, col=1)
    fig.update_yaxes(title_text="Значение", row=1, col=1)
    fig.update_yaxes(title_text="Значение", row=2, col=1)

    fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# НАСТРОЙКИ
# ============================================================
TOP_N         = 15   # сколько маршрутов смотреть
N_HIST_LONG   = 720  # точек в верхнем subplot
N_HIST_SHORT  = 48   # точек контекста перед предиктом ← настраивай

TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

def mae(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / y_true.shape[0])

# ============================================================
# Загрузка
# ============================================================
chr_raw  = pd.read_csv("chronos2_cv_raw.csv")
train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

cv5 = chr_raw[chr_raw["fold"] == 5].copy()

# Достаём timestamp для fold 5 из train_df (последние 8 точек каждого маршрута)
fold5_ts = (
    train_df.groupby("route_id", group_keys=False)
    .tail(8)[["route_id","timestamp"]]
    .copy()
    .reset_index(drop=True)
)
fold5_ts["h"] = fold5_ts.groupby("route_id").cumcount() + 1
cv5 = cv5.drop(columns=["timestamp"], errors="ignore")
cv5 = cv5.merge(fold5_ts, on=["route_id","h"], how="left")

# ============================================================
# Топ-N проблемных маршрутов
# ============================================================
route_wape = (
    cv5.groupby("route_id")
    .apply(lambda df: pd.Series({
        "wape_chr": mae(df["y_true"], df["y_pred"]),
        "vol":      float(df["y_true"].sum()),
    }))
    .reset_index()
)
route_wape = route_wape[route_wape["vol"] > 0].sort_values("wape_chr", ascending=False)
top_routes = route_wape.head(TOP_N)["route_id"].tolist()
print(route_wape.head(TOP_N)[["route_id","wape_chr","vol"]].to_string(index=False))

# ============================================================
# Графики
# ============================================================
for route_id in top_routes:
    hist = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    pred = cv5[cv5["route_id"] == route_id].sort_values("h").reset_index(drop=True)
    wape_val = wape_rbias(pred["y_true"], pred["y_pred"])
    vol_val  = route_wape[route_wape["route_id"] == route_id]["vol"].values[0]

    n_long  = min(N_HIST_LONG,  len(hist) - TOTAL_VAL_PTS)
    n_short = min(N_HIST_SHORT, len(hist) - TOTAL_VAL_PTS)

    hist_long  = hist.iloc[-(TOTAL_VAL_PTS + n_long) : -TOTAL_VAL_PTS].reset_index(drop=True)
    FOLD5_PTS  = 8  # размер fold 5
    hist_short = hist.iloc[-(FOLD5_PTS + n_short): -FOLD5_PTS]

    # Собираем нижний subplot как один непрерывный ряд
    context_df = pd.DataFrame({
        "timestamp": hist_short["timestamp"],
        "y_true":    hist_short[TARGET_COL].values,
        "y_pred":    np.nan,
        "part":      "context",
    })
    pred_df = pd.DataFrame({
        "timestamp": pred["timestamp"],
        "y_true":    pred["y_true"].values,
        "y_pred":    pred["y_pred"].values,
        "part":      "pred",
    })
    # Стыковочная точка — последняя точка контекста добавляется в pred для непрерывности линии
    bridge = context_df.iloc[[-1]].copy()
    bridge["part"] = "pred"

    lower = pd.concat([context_df, bridge, pred_df], ignore_index=True).sort_values("timestamp")

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"История: последние {n_long} точек (~{n_long//24} дней)",
            f"Контекст ({n_short} т.) + предикт fold 5 (h=1..8)",
        ],
        vertical_spacing=0.20,
        row_heights=[0.40, 0.60],
    )

    # ── Верхний: история ──
    fig.add_trace(go.Scatter(
        x=hist_long["timestamp"].tolist(),
        y=hist_long[TARGET_COL].tolist(),
        mode="lines",
        name="История",
        line=dict(width=1.2),
    ), row=1, col=1)

    # ── Нижний: факт (контекст + fold 5) ──
    fig.add_trace(go.Scatter(
        x=lower["timestamp"].tolist(),
        y=lower["y_true"].tolist(),
        mode="lines+markers",
        name="Факт",
        line=dict(width=2),
        marker=dict(size=5),
    ), row=2, col=1)

    # ── Нижний: предикт Chronos (только на горизонте pred) ──
    fig.add_trace(go.Scatter(
        x=lower[lower["part"] == "pred"]["timestamp"].tolist(),
        y=lower[lower["part"] == "pred"]["y_pred"].tolist(),
        mode="lines+markers",
        name="Chronos",
        line=dict(width=2, dash="dash"),
        marker=dict(size=7, symbol="diamond"),
    ), row=2, col=1)

    # Граница контекст/предикт — через annotation, не vline
    split_ts = pred["timestamp"].iloc[0]
    fig.add_annotation(
        x=split_ts, y=1.02,
        xref="x2", yref="paper",
        text="▼ предикт",
        showarrow=False,
        font=dict(size=11, color="gray"),
    )

    fig.update_layout(
        title=dict(
            text=(
                f"Маршрут {route_id}  ·  WAPE={wape_val:.2f}  ·  vol={vol_val:.0f}"
            )
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.05,
                    xanchor="center", x=0.5),
        height=680,
    )
    fig.update_xaxes(title_text="Время", row=1, col=1)
    fig.update_xaxes(title_text="Время", row=2, col=1)
    fig.update_yaxes(title_text="Значение", row=1, col=1)
    fig.update_yaxes(title_text="Значение", row=2, col=1)

    fig.show()

In [ ]:
print(top_routes)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# НАСТРОЙКИ
# ============================================================
TOP_N         = 15   # сколько маршрутов смотреть
N_HIST_LONG   = 720  # точек в верхнем subplot
N_HIST_SHORT  = 48   # точек контекста перед предиктом ← настраивай

TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

def mae(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / y_true.shape[0])

# ============================================================
# Загрузка
# ============================================================
chr_raw  = pd.read_csv("chronos2_cv_raw.csv")
train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

cv5 = chr_raw[chr_raw["fold"] == 5].copy()

# Достаём timestamp для fold 5 из train_df (последние 8 точек каждого маршрута)
fold5_ts = (
    train_df.groupby("route_id", group_keys=False)
    .tail(8)[["route_id","timestamp"]]
    .copy()
    .reset_index(drop=True)
)
fold5_ts["h"] = fold5_ts.groupby("route_id").cumcount() + 1
cv5 = cv5.drop(columns=["timestamp"], errors="ignore")
cv5 = cv5.merge(fold5_ts, on=["route_id","h"], how="left")

# ============================================================
# Топ-N проблемных маршрутов
# ============================================================
route_wape = (
    cv5.groupby("route_id")
    .apply(lambda df: pd.Series({
        "wape_chr": mae(df["y_true"], df["y_pred"]),
        "vol":      float(df["y_true"].sum()),
    }))
    .reset_index()
)
route_wape = route_wape[route_wape["vol"] > 0].sort_values("wape_chr", ascending=True)
top_routes = route_wape.head(TOP_N)["route_id"].tolist()
print(route_wape.head(TOP_N)[["route_id","wape_chr","vol"]].to_string(index=False))

# ============================================================
# Графики
# ============================================================
for route_id in top_routes:
    hist = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    pred = cv5[cv5["route_id"] == route_id].sort_values("h").reset_index(drop=True)
    wape_val = wape_rbias(pred["y_true"], pred["y_pred"])
    vol_val  = route_wape[route_wape["route_id"] == route_id]["vol"].values[0]

    n_long  = min(N_HIST_LONG,  len(hist) - TOTAL_VAL_PTS)
    n_short = min(N_HIST_SHORT, len(hist) - TOTAL_VAL_PTS)

    hist_long  = hist.iloc[-(TOTAL_VAL_PTS + n_long) : -TOTAL_VAL_PTS].reset_index(drop=True)
    FOLD5_PTS  = 8  # размер fold 5
    hist_short = hist.iloc[-(FOLD5_PTS + n_short): -FOLD5_PTS]

    # Собираем нижний subplot как один непрерывный ряд
    context_df = pd.DataFrame({
        "timestamp": hist_short["timestamp"],
        "y_true":    hist_short[TARGET_COL].values,
        "y_pred":    np.nan,
        "part":      "context",
    })
    pred_df = pd.DataFrame({
        "timestamp": pred["timestamp"],
        "y_true":    pred["y_true"].values,
        "y_pred":    pred["y_pred"].values,
        "part":      "pred",
    })
    # Стыковочная точка — последняя точка контекста добавляется в pred для непрерывности линии
    bridge = context_df.iloc[[-1]].copy()
    bridge["part"] = "pred"

    lower = pd.concat([context_df, bridge, pred_df], ignore_index=True).sort_values("timestamp")

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"История: последние {n_long} точек (~{n_long//24} дней)",
            f"Контекст ({n_short} т.) + предикт fold 5 (h=1..8)",
        ],
        vertical_spacing=0.20,
        row_heights=[0.40, 0.60],
    )

    # ── Верхний: история ──
    fig.add_trace(go.Scatter(
        x=hist_long["timestamp"].tolist(),
        y=hist_long[TARGET_COL].tolist(),
        mode="lines",
        name="История",
        line=dict(width=1.2),
    ), row=1, col=1)

    # ── Нижний: факт (контекст + fold 5) ──
    fig.add_trace(go.Scatter(
        x=lower["timestamp"].tolist(),
        y=lower["y_true"].tolist(),
        mode="lines+markers",
        name="Факт",
        line=dict(width=2),
        marker=dict(size=5),
    ), row=2, col=1)

    # ── Нижний: предикт Chronos (только на горизонте pred) ──
    fig.add_trace(go.Scatter(
        x=lower[lower["part"] == "pred"]["timestamp"].tolist(),
        y=lower[lower["part"] == "pred"]["y_pred"].tolist(),
        mode="lines+markers",
        name="Chronos",
        line=dict(width=2, dash="dash"),
        marker=dict(size=7, symbol="diamond"),
    ), row=2, col=1)

    # Граница контекст/предикт — через annotation, не vline
    split_ts = pred["timestamp"].iloc[0]
    fig.add_annotation(
        x=split_ts, y=1.02,
        xref="x2", yref="paper",
        text="▼ предикт",
        showarrow=False,
        font=dict(size=11, color="gray"),
    )

    fig.update_layout(
        title=dict(
            text=(
                f"Маршрут {route_id}  ·  WAPE={wape_val:.2f}  ·  vol={vol_val:.0f}"
            )
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.05,
                    xanchor="center", x=0.5),
        height=680,
    )
    fig.update_xaxes(title_text="Время", row=1, col=1)
    fig.update_xaxes(title_text="Время", row=2, col=1)
    fig.update_yaxes(title_text="Значение", row=1, col=1)
    fig.update_yaxes(title_text="Значение", row=2, col=1)

    fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# НАСТРОЙКИ
# ============================================================
TOP_N         = 15   # сколько маршрутов смотреть
N_HIST_LONG   = 720  # точек в верхнем subplot
N_HIST_SHORT  = 48   # точек контекста перед предиктом ← настраивай

TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

def mae(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / y_true.shape[0])

# ============================================================
# Загрузка
# ============================================================
chr_raw  = pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv")
train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

cv5 = chr_raw[chr_raw["fold"] == 5].copy()

# Достаём timestamp для fold 5 из train_df (последние 8 точек каждого маршрута)
fold5_ts = (
    train_df.groupby("route_id", group_keys=False)
    .tail(8)[["route_id","timestamp"]]
    .copy()
    .reset_index(drop=True)
)
fold5_ts["h"] = fold5_ts.groupby("route_id").cumcount() + 1
cv5 = cv5.drop(columns=["timestamp"], errors="ignore")
cv5 = cv5.merge(fold5_ts, on=["route_id","h"], how="left")

# ============================================================
# Топ-N проблемных маршрутов
# ============================================================
route_wape = (
    cv5.groupby("route_id")
    .apply(lambda df: pd.Series({
        "wape_chr": mae(df["y_true"], df["y_pred"]),
        "vol":      float(df["y_true"].sum()),
    }))
    .reset_index()
)
route_wape = route_wape[route_wape["vol"] > 0].sort_values("wape_chr", ascending=True)
top_routes = route_wape.head(TOP_N)["route_id"].tolist()
print(route_wape.head(TOP_N)[["route_id","wape_chr","vol"]].to_string(index=False))

# ============================================================
# Графики
# ============================================================
for route_id in [485, 192, 702, 999, 446, 782, 725, 768, 445, 817, 329, 594, 123, 948, 263]:
    hist = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    pred = cv5[cv5["route_id"] == route_id].sort_values("h").reset_index(drop=True)
    wape_val = wape_rbias(pred["y_true"], pred["y_pred"])
    vol_val  = route_wape[route_wape["route_id"] == route_id]["vol"].values[0]

    n_long  = min(N_HIST_LONG,  len(hist) - TOTAL_VAL_PTS)
    n_short = min(N_HIST_SHORT, len(hist) - TOTAL_VAL_PTS)

    hist_long  = hist.iloc[-(TOTAL_VAL_PTS + n_long) : -TOTAL_VAL_PTS].reset_index(drop=True)
    FOLD5_PTS  = 8  # размер fold 5
    hist_short = hist.iloc[-(FOLD5_PTS + n_short): -FOLD5_PTS]

    # Собираем нижний subplot как один непрерывный ряд
    context_df = pd.DataFrame({
        "timestamp": hist_short["timestamp"],
        "y_true":    hist_short[TARGET_COL].values,
        "y_pred":    np.nan,
        "part":      "context",
    })
    pred_df = pd.DataFrame({
        "timestamp": pred["timestamp"],
        "y_true":    pred["y_true"].values,
        "y_pred":    pred["y_pred"].values,
        "part":      "pred",
    })
    # Стыковочная точка — последняя точка контекста добавляется в pred для непрерывности линии
    bridge = context_df.iloc[[-1]].copy()
    bridge["part"] = "pred"

    lower = pd.concat([context_df, bridge, pred_df], ignore_index=True).sort_values("timestamp")

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"История: последние {n_long} точек (~{n_long//24} дней)",
            f"Контекст ({n_short} т.) + предикт fold 5 (h=1..8)",
        ],
        vertical_spacing=0.20,
        row_heights=[0.40, 0.60],
    )

    # ── Верхний: история ──
    fig.add_trace(go.Scatter(
        x=hist_long["timestamp"].tolist(),
        y=hist_long[TARGET_COL].tolist(),
        mode="lines",
        name="История",
        line=dict(width=1.2),
    ), row=1, col=1)

    # ── Нижний: факт (контекст + fold 5) ──
    fig.add_trace(go.Scatter(
        x=lower["timestamp"].tolist(),
        y=lower["y_true"].tolist(),
        mode="lines+markers",
        name="Факт",
        line=dict(width=2),
        marker=dict(size=5),
    ), row=2, col=1)

    # ── Нижний: предикт Chronos (только на горизонте pred) ──
    fig.add_trace(go.Scatter(
        x=lower[lower["part"] == "pred"]["timestamp"].tolist(),
        y=lower[lower["part"] == "pred"]["y_pred"].tolist(),
        mode="lines+markers",
        name="Chronos",
        line=dict(width=2, dash="dash"),
        marker=dict(size=7, symbol="diamond"),
    ), row=2, col=1)

    # Граница контекст/предикт — через annotation, не vline
    split_ts = pred["timestamp"].iloc[0]
    fig.add_annotation(
        x=split_ts, y=1.02,
        xref="x2", yref="paper",
        text="▼ предикт",
        showarrow=False,
        font=dict(size=11, color="gray"),
    )

    fig.update_layout(
        title=dict(
            text=(
                f"Маршрут {route_id}  ·  WAPE={wape_val:.2f}  ·  vol={vol_val:.0f}"
            )
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.05,
                    xanchor="center", x=0.5),
        height=680,
    )
    fig.update_xaxes(title_text="Время", row=1, col=1)
    fig.update_xaxes(title_text="Время", row=2, col=1)
    fig.update_yaxes(title_text="Значение", row=1, col=1)
    fig.update_yaxes(title_text="Значение", row=2, col=1)

    fig.show()

In [ ]:
# ============================================================
# Диагностика временных рядов
# Stationarity + Seasonality + ACF + Volatility + Distribution
# ============================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.signal import periodogram
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tsa.seasonal import STL
from tqdm import tqdm
import os

warnings.filterwarnings("ignore")
os.makedirs("diagnostics", exist_ok=True)

TRAIN_PATH  = "train_solo_track.parquet"
TARGET_COL  = "target_1h"
FREQ        = "30min"
MAX_ROUTES  = 999   # поставь 20-30 для быстрого теста, 999 = все

# ── Load ──
train_df = pd.read_parquet(TRAIN_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
route_ids = train_df["route_id"].unique()[:MAX_ROUTES]
print(f"Маршрутов: {len(route_ids)}  |  Всего строк: {len(train_df)}\n")


# ============================================================
# ФУНКЦИИ ТЕСТОВ
# ============================================================

def run_adf(series: np.ndarray):
    """ADF: H0 = есть единичный корень (нестационарен)"""
    try:
        res = adfuller(series, autolag="AIC", maxlag=20)
        return {"adf_stat": res[0], "adf_p": res[1], "adf_stationary": res[1] < 0.05}
    except:
        return {"adf_stat": np.nan, "adf_p": np.nan, "adf_stationary": None}


def run_kpss(series: np.ndarray):
    """KPSS: H0 = стационарен (противоположно ADF!)"""
    try:
        res = kpss(series, regression="c", nlags="auto")
        return {"kpss_stat": res[0], "kpss_p": res[1], "kpss_stationary": res[1] > 0.05}
    except:
        return {"kpss_stat": np.nan, "kpss_p": np.nan, "kpss_stationary": None}


def stationarity_verdict(adf_stat, kpss_stat):
    """Объединённый вердикт ADF+KPSS"""
    if adf_stat is True  and kpss_stat is True:  return "stationary"
    if adf_stat is False and kpss_stat is False: return "non_stationary"
    if adf_stat is True  and kpss_stat is False: return "difference_stationary"  # тренд
    if adf_stat is False and kpss_stat is True:  return "trend_stationary"       # стохастический тренд
    return "unknown"


def detect_seasonality(series: np.ndarray, fs: int = 48):
    """
    Спектральный анализ: ищем пики на частотах
    суточной (48 отсчётов), недельной (336), 12-ч (24) сезонности
    """
    n = len(series)
    if n < 100:
        return {}
    freqs, power = periodogram(series - series.mean())
    # Индексы характерных периодов
    periods = {"daily_48": 48, "halfday_24": 24, "weekly_336": 336, "4h_8": 8}
    result  = {}
    total_power = power.sum() + 1e-9
    for name, period in periods.items():
        target_freq = 1.0 / period
        idx = np.argmin(np.abs(freqs - target_freq))
        # Сила пика относительно суммарной мощности
        window = max(1, int(0.002 * n))
        peak_p = power[max(0,idx-window): idx+window+1].sum()
        result[f"seasonal_{name}_power"] = float(peak_p / total_power)
    result["dominant_period"] = int(1.0 / freqs[np.argmax(power[1:]) + 1])
    return result


def compute_acf_features(series: np.ndarray):
    """ACF/PACF: автокорреляции на ключевых лагах"""
    n = len(series)
    if n < 50:
        return {}
    nlags = min(336, n // 2 - 1)
    try:
        acf_vals  = acf(series,  nlags=nlags, fft=True)
        pacf_vals = pacf(series, nlags=min(50, n//2 - 1))
        return {
            "acf_lag1":   float(acf_vals[1])  if len(acf_vals)  > 1   else np.nan,
            "acf_lag48":  float(acf_vals[48]) if len(acf_vals)  > 48  else np.nan,
            "acf_lag336": float(acf_vals[336])if len(acf_vals)  > 336 else np.nan,
            "pacf_lag1":  float(pacf_vals[1]) if len(pacf_vals) > 1   else np.nan,
            "pacf_lag2":  float(pacf_vals[2]) if len(pacf_vals) > 2   else np.nan,
            # Значимых автокорреляций (|acf| > 2/sqrt(n))
            "n_sig_acf":  int(np.sum(np.abs(acf_vals[1:50]) > 2/np.sqrt(n))),
        }
    except:
        return {}


def compute_volatility(series: np.ndarray):
    """ARCH-тест + rolling std ratio"""
    if len(series) < 30:
        return {}
    try:
        lm_stat, lm_p, f_stat, f_p = het_arch(series, nlags=12)
        arch_present = lm_p < 0.05
    except:
        lm_p, arch_present = np.nan, None

    # Коэффициент вариации volatility: std(rolling_std) / mean(rolling_std)
    rs = pd.Series(series).rolling(24, min_periods=1).std().dropna()
    vol_cv = float(rs.std() / (rs.mean() + 1e-9))
    return {
        "arch_p":       float(lm_p),
        "arch_present": arch_present,    # True = гетероскедастичность (volatile)
        "vol_cv":       vol_cv,          # > 1 = очень нестабильная волатильность
    }


def compute_distribution(series: np.ndarray):
    """Скос, куртоз, доля нулей, хвостатость"""
    s  = pd.Series(series)
    nz = float((s == 0).mean())
    try:
        _, norm_p = stats.normaltest(series)
    except:
        norm_p = np.nan
    return {
        "mean":       float(s.mean()),
        "std":        float(s.std()),
        "skewness":   float(s.skew()),
        "kurtosis":   float(s.kurtosis()),    # excess kurtosis (>0 = тяжёлые хвосты)
        "zero_frac":  nz,
        "cv":         float(s.std() / (s.mean() + 1e-9)),    # коэф. вариации
        "normal_p":   float(norm_p),          # < 0.05 → не нормальное
        "p99_p50":    float(s.quantile(0.99) / (s.quantile(0.50) + 1e-9)),  # острота пиков
    }


def compute_trend(series: np.ndarray):
    """Линейный тренд через Mann-Kendall (rank-based, не требует нормальности)"""
    n = len(series)
    if n < 20:
        return {}
    # Mann-Kendall вручную (быстрая версия)
    s_stat = 0
    for i in range(n - 1):
        s_stat += np.sign(series[i+1:] - series[i]).sum()
    var_s = n * (n-1) * (2*n+5) / 18
    z = (s_stat - np.sign(s_stat)) / np.sqrt(var_s) if var_s > 0 else 0
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    # Угол наклона (Theil-Sen)
    slope, intercept, _, _, _ = stats.linregress(np.arange(n), series)
    return {
        "trend_mk_z":  float(z),
        "trend_mk_p":  float(p),
        "trend_slope": float(slope),
        "has_trend":   p < 0.05,
    }


def compute_intermittency(series: np.ndarray):
    """ADI (Average Demand Interval) + CV² — классика для intermittent demand"""
    s = pd.Series(series)
    nonzero = s[s > 0]
    if len(nonzero) < 2:
        return {"adi": np.inf, "cv2": np.nan, "demand_type": "lumpy"}
    adi  = len(s) / len(nonzero)          # средний интервал между ненулевыми
    cv2  = (nonzero.std() / nonzero.mean()) ** 2  # квадрат CV ненулевых значений
    # Классификация по Syntetos-Boylan
    if adi < 1.32 and cv2 < 0.49:   dtype = "smooth"
    elif adi < 1.32 and cv2 >= 0.49: dtype = "erratic"
    elif adi >= 1.32 and cv2 < 0.49: dtype = "intermittent"
    else:                             dtype = "lumpy"
    return {"adi": float(adi), "cv2": float(cv2), "demand_type": dtype}


def run_stl(series: pd.Series, period: int = 48):
    """STL декомпозиция: доля дисперсии объясняемая трендом и сезонностью"""
    if len(series) < period * 3:
        return {}
    try:
        stl   = STL(series, period=period, robust=True).fit()
        var_t = np.var(stl.trend)
        var_s = np.var(stl.seasonal)
        var_r = np.var(stl.resid)
        total = var_t + var_s + var_r + 1e-9
        return {
            "stl_trend_ratio":    float(var_t / total),
            "stl_seasonal_ratio": float(var_s / total),
            "stl_resid_ratio":    float(var_r / total),
        }
    except:
        return {}


# ============================================================
# ОСНОВНОЙ ЦИКЛ
# ============================================================
print("Анализ маршрутов...")
results = []

for route_id in tqdm(route_ids):
    grp = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[TARGET_COL]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )
    s = grp.values.astype(float)
    n = len(s)

    row = {"route_id": route_id, "n_points": n}
    row.update(run_adf(s))
    row.update(run_kpss(s))
    row["stationarity"] = stationarity_verdict(row["adf_stationary"], row["kpss_stationary"])
    row.update(detect_seasonality(s))
    row.update(compute_acf_features(s))
    row.update(compute_volatility(s))
    row.update(compute_distribution(s))
    row.update(compute_trend(s))
    row.update(compute_intermittency(s))
    row.update(run_stl(grp))
    results.append(row)

diag_df = pd.DataFrame(results)
diag_df.to_csv("diagnostics/ts_diagnostics_full.csv", index=False)
print(f"\nПолная диагностика: diagnostics/ts_diagnostics_full.csv\n")


# ============================================================
# СВОДКА ПО ВСЕМ МАРШРУТАМ
# ============================================================
print("=" * 65)
print("СВОДКА ПО ВСЕМ МАРШРУТАМ")
print("=" * 65)

n_total = len(diag_df)

# ── Стационарность ──
stat_counts = diag_df["stationarity"].value_counts()
print("\n── Стационарность (ADF + KPSS) ──")
for k, v in stat_counts.items():
    print(f"  {k:<30s}  {v:>4d}  ({100*v/n_total:.1f}%)")

# ── Сезонность ──
print("\n── Сила сезонных компонент (медиана по маршрутам) ──")
for col in ["seasonal_daily_48_power", "seasonal_halfday_24_power",
            "seasonal_weekly_336_power", "seasonal_4h_8_power"]:
    if col in diag_df:
        med = diag_df[col].median()
        pct = (diag_df[col] > 0.05).mean() * 100
        label = col.replace("seasonal_","").replace("_power","")
        print(f"  {label:<20s}  median_power={med:.4f}   доля маршрутов с пиком>{5}%: {pct:.1f}%")

# ── Тренд ──
if "has_trend" in diag_df:
    has_trend = diag_df["has_trend"].mean() * 100
    print(f"\n── Тренд (Mann-Kendall p<0.05) ──")
    print(f"  Маршрутов с трендом: {has_trend:.1f}%")
    up   = (diag_df["trend_slope"] > 0).mean() * 100
    down = (diag_df["trend_slope"] < 0).mean() * 100
    print(f"  Из них растущих: {up:.1f}%  |  убывающих: {down:.1f}%")

# ── Волатильность ──
if "arch_present" in diag_df:
    arch_pct = diag_df["arch_present"].mean() * 100
    print(f"\n── Гетероскедастичность (ARCH-тест) ──")
    print(f"  Маршрутов с ARCH-эффектом: {arch_pct:.1f}%  (нестабильная дисперсия)")
    print(f"  Медиана vol_cv: {diag_df['vol_cv'].median():.3f}  (>1 = сильно нестабильная)")

# ── Распределение ──
print(f"\n── Распределение target_1h (медианы по маршрутам) ──")
for col in ["mean", "std", "skewness", "kurtosis", "zero_frac", "cv", "p99_p50"]:
    if col in diag_df:
        print(f"  {col:<15s}  {diag_df[col].median():.4f}")

# ── Intermittency ──
if "demand_type" in diag_df:
    print(f"\n── Тип спроса (Syntetos-Boylan) ──")
    for k, v in diag_df["demand_type"].value_counts().items():
        print(f"  {k:<20s}  {v:>4d}  ({100*v/n_total:.1f}%)")

# ── ACF ──
print(f"\n── Автокорреляции (медианы по маршрутам) ──")
for col in ["acf_lag1", "acf_lag48", "acf_lag336", "pacf_lag1", "n_sig_acf"]:
    if col in diag_df:
        print(f"  {col:<15s}  {diag_df[col].median():.4f}")

# ── STL ──
print(f"\n── STL декомпозиция (медианы) ──")
for col in ["stl_trend_ratio", "stl_seasonal_ratio", "stl_resid_ratio"]:
    if col in diag_df:
        print(f"  {col:<25s}  {diag_df[col].median():.4f}")


# ============================================================
# ВЫВОД: РЕКОМЕНДАЦИИ ПО МОДЕЛЯМ
# ============================================================
print("\n" + "=" * 65)
print("РЕКОМЕНДАЦИИ ПО МОДЕЛЯМ")
print("=" * 65)

arch_pct      = diag_df["arch_present"].mean()      if "arch_present" in diag_df else 0
daily_season  = diag_df["seasonal_daily_48_power"].median() if "seasonal_daily_48_power" in diag_df else 0
weekly_season = diag_df["seasonal_weekly_336_power"].median() if "seasonal_weekly_336_power" in diag_df else 0
zero_frac     = diag_df["zero_frac"].median()        if "zero_frac" in diag_df else 0
skewness      = diag_df["skewness"].median()          if "skewness" in diag_df else 0
stl_resid     = diag_df["stl_resid_ratio"].median()   if "stl_resid_ratio" in diag_df else 0
cv_med        = diag_df["cv"].median()                if "cv" in diag_df else 0
acf1          = diag_df["acf_lag1"].median()           if "acf_lag1" in diag_df else 0
acf48         = diag_df["acf_lag48"].median()          if "acf_lag48" in diag_df else 0

model_scores = {
    "N-HiTS (текущий)":        0,
    "N-BEATS":                  0,
    "LightGBM + lag features":  0,
    "SARIMA":                   0,
    "Prophet":                  0,
    "Exponential Smoothing":    0,
    "Croston (intermittent)":   0,
    "TiRex (zero-shot)":        0,
}

def add_score(d, keys, delta, reason):
    for k in keys:
        if k in d:
            d[k] += delta
    return reason

reasons = []

if daily_season > 0.05:
    add_score(model_scores,
        ["N-HiTS (текущий)", "N-BEATS", "LightGBM + lag features",
         "SARIMA", "Prophet", "N-HiTS (текущий)"], +2,
        f"✅ Сильная суточная сезонность ({daily_season:.3f}) → модели со seasonal декомпозицией")
    reasons.append(f"Суточная сезонность мощная ({daily_season:.3f}): важны lag_48, hour_sin/cos, SARIMA(48)")

if weekly_season > 0.03:
    add_score(model_scores,
        ["N-HiTS (текущий)", "LightGBM + lag features", "Prophet"], +2,
        f"✅ Недельная сезонность → lag_336, INPUT_SIZE>=336")
    reasons.append(f"Недельная сезонность ({weekly_season:.3f}): нужны lag_336, INPUT_SIZE=336")

if arch_pct > 0.5:
    add_score(model_scores,
        ["N-HiTS (текущий)", "LightGBM + lag features", "TiRex (zero-shot)"], +2,
        "⚠️ ARCH-эффект у >50% маршрутов → гетероскедастичность, нейросети + GBDT предпочтительнее ARIMA")
    reasons.append(f"ARCH-эффект у {arch_pct*100:.0f}% маршрутов: волатильность нестабильна → ARIMA/ETS плохо работают")

if stl_resid > 0.5:
    add_score(model_scores,
        ["N-HiTS (текущий)", "LightGBM + lag features", "TiRex (zero-shot)"], +2,
        "⚠️ STL-остаток >50% дисперсии → высокий непредсказуемый шум, нейросети + GBDT")
    reasons.append(f"STL-остаток = {stl_resid:.2f} дисперсии: ряд плохо разложим, нужны нелинейные модели")

if zero_frac > 0.3:
    add_score(model_scores, ["Croston (intermittent)", "LightGBM + lag features"], +3,
        f"⚠️ {zero_frac*100:.0f}% нулей → intermittent demand, нужен Croston/ADIDA или LGBM с нулями")
    reasons.append(f"Много нулей ({zero_frac*100:.0f}%): intermittent demand")

if acf48 > 0.3:
    add_score(model_scores,
        ["SARIMA", "LightGBM + lag features", "Exponential Smoothing"], +1,
        f"✅ ACF(lag48)={acf48:.2f} → сильная суточная автокорреляция")
    reasons.append(f"ACF(lag48)={acf48:.2f}: вчерашнее значение хорошо предсказывает сегодня")

if cv_med > 1.0:
    add_score(model_scores,
        ["N-HiTS (текущий)", "LightGBM + lag features", "TiRex (zero-shot)"], +1,
        f"⚠️ CV={cv_med:.2f} > 1 → высокая вариативность, сложный ряд")
    reasons.append(f"CV={cv_med:.2f}: пики значительно выше среднего")

if skewness > 1.0:
    add_score(model_scores,
        ["LightGBM + lag features", "N-HiTS (текущий)"], +1,
        f"⚠️ Скос={skewness:.2f} → правосторонний хвост (пики), log-трансформация может помочь")
    reasons.append(f"Скос={skewness:.2f}: тяжёлый правый хвост → log1p трансформация")

print("\n── Ключевые свойства рядов ──")
for r in reasons:
    print(f"  • {r}")

print("\n── Рейтинг моделей (по накопленному score) ──")
for model, score in sorted(model_scores.items(), key=lambda x: -x[1]):
    bar = "█" * score
    print(f"  {model:<35s}  score={score:>2d}  {bar}")


# ============================================================
# ВИЗУАЛИЗАЦИЯ
# ============================================================
print("\nСтрою графики...")

# ── 1. Распределение типов стационарности ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Диагностика временных рядов — сводка", fontsize=14, fontweight="bold")

# Stationarity pie
ax = axes[0, 0]
counts = diag_df["stationarity"].value_counts()
ax.pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Стационарность (ADF+KPSS)")

# Demand type pie
ax = axes[0, 1]
if "demand_type" in diag_df:
    counts2 = diag_df["demand_type"].value_counts()
    ax.pie(counts2, labels=counts2.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Тип спроса (Syntetos-Boylan)")

# ACF lag48 distribution
ax = axes[0, 2]
vals = diag_df["acf_lag48"].dropna()
ax.hist(vals, bins=30, color="steelblue", edgecolor="white")
ax.axvline(vals.median(), color="red", linestyle="--", label=f"median={vals.median():.2f}")
ax.axvline(0, color="black", linewidth=0.5)
ax.set_title("ACF(lag=48) — суточная автокорреляция")
ax.set_xlabel("ACF")
ax.legend()

# CV distribution
ax = axes[1, 0]
vals = diag_df["cv"].dropna().clip(0, 5)
ax.hist(vals, bins=30, color="coral", edgecolor="white")
ax.axvline(vals.median(), color="red", linestyle="--", label=f"median={vals.median():.2f}")
ax.set_title("Коэффициент вариации (CV)")
ax.set_xlabel("std/mean")
ax.legend()

# STL ratios
ax = axes[1, 1]
cols_stl = ["stl_trend_ratio", "stl_seasonal_ratio", "stl_resid_ratio"]
labels_stl = ["Тренд", "Сезонность", "Остаток"]
medians = [diag_df[c].median() for c in cols_stl if c in diag_df]
if medians:
    ax.bar(labels_stl[:len(medians)], medians,
           color=["#4CAF50", "#2196F3", "#FF5722"])
    ax.set_title("STL: доля дисперсии\n(медиана по маршрутам)")
    ax.set_ylabel("Доля дисперсии")
    ax.set_ylim(0, 1)
    for i, v in enumerate(medians):
        ax.text(i, v + 0.01, f"{v:.2f}", ha="center")

# Seasonal powers
ax = axes[1, 2]
sp_cols   = ["seasonal_4h_8_power", "seasonal_halfday_24_power",
             "seasonal_daily_48_power", "seasonal_weekly_336_power"]
sp_labels = ["4ч", "12ч", "Суточная", "Недельная"]
sp_vals   = [diag_df[c].median() for c in sp_cols if c in diag_df]
if sp_vals:
    bars = ax.bar(sp_labels[:len(sp_vals)], sp_vals,
                  color=["#9C27B0", "#FF9800", "#2196F3", "#4CAF50"])
    ax.set_title("Спектральная мощность\nсезонных компонент")
    ax.set_ylabel("Относительная мощность")
    for bar, v in zip(bars, sp_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.0005,
                f"{v:.4f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("diagnostics/ts_summary_plots.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ diagnostics/ts_summary_plots.png")


# ── 2. Примеры 4 маршрутов с STL ──
sample_routes = diag_df.sort_values("stl_resid_ratio", ascending=False)["route_id"].iloc[:4].tolist()

fig, axes = plt.subplots(4, 4, figsize=(20, 14))
fig.suptitle("Примеры маршрутов: ряд + STL + ACF + PACF", fontsize=13, fontweight="bold")

for i, rid in enumerate(sample_routes):
    grp = (
        train_df[train_df["route_id"] == rid]
        .sort_values("timestamp")
        .set_index("timestamp")[TARGET_COL]
        .asfreq(FREQ).interpolate(method="time").bfill().ffill()
    )
    s = grp.values
    n = len(s)

    # Raw series
    ax = axes[i, 0]
    ax.plot(grp.index[-336:], s[-336:], linewidth=0.7, color="steelblue")
    ax.set_title(f"Route {rid}\n(последние 7 суток)", fontsize=8)
    ax.set_ylabel("target_1h")

    # STL
    ax = axes[i, 1]
    try:
        stl_res = STL(grp, period=48, robust=True).fit()
        ax.plot(grp.index[-336:], stl_res.seasonal[-336:], color="orange", linewidth=0.8)
        ax.set_title("STL seasonal (суточный)", fontsize=8)
    except:
        ax.set_title("STL N/A", fontsize=8)

    # ACF
    ax = axes[i, 2]
    try:
        nlags = min(100, n//2 - 1)
        acf_v = acf(s, nlags=nlags, fft=True)
        ax.stem(range(len(acf_v)), acf_v,
                linefmt="C0-", markerfmt="C0o", basefmt="k-")
        ax.axhline(2/np.sqrt(n), color="red", linestyle="--", linewidth=0.8)
        ax.axhline(-2/np.sqrt(n), color="red", linestyle="--", linewidth=0.8)
        ax.set_title("ACF", fontsize=8)
        ax.set_xlim(0, nlags)
    except:
        pass

    # PACF
    ax = axes[i, 3]
    try:
        pacf_v = pacf(s, nlags=min(50, n//4))
        ax.stem(range(len(pacf_v)), pacf_v,
                linefmt="C1-", markerfmt="C1o", basefmt="k-")
        ax.axhline(2/np.sqrt(n), color="red", linestyle="--", linewidth=0.8)
        ax.axhline(-2/np.sqrt(n), color="red", linestyle="--", linewidth=0.8)
        ax.set_title("PACF", fontsize=8)
    except:
        pass

plt.tight_layout()
plt.savefig("diagnostics/ts_sample_routes.png", dpi=130, bbox_inches="tight")
plt.close()
print("✅ diagnostics/ts_sample_routes.png")

print("\n✅ diagnostics/ts_diagnostics_full.csv")
print("\nВсё готово! Смотри СВОДКУ выше для итоговых выводов.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Параметры ──
TRAIN_PATH = "train_solo_track.parquet"
TARGET_COL = "target_1h"
FREQ       = "30min"
SHOW_LAST  =  48 * 7  # точек в итоговом графике (= 24 часа)
STL_PERIOD = 4   # суточная сезонность

# ── Загрузка (один раз) ──
if "train_df" not in dir():
    train_df = pd.read_parquet(TRAIN_PATH)
    train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
    train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

all_routes = sorted(train_df["route_id"].unique().tolist())

# ── Функция отрисовки ──
def plot_stl(route_id):
    grp = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[TARGET_COL]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )

    n = len(grp)
    if n < STL_PERIOD * 2:
        print(f"⚠️ Недостаточно данных для route_id={route_id} (n={n})")
        return

    # STL на всём ряду, показываем последние SHOW_LAST точек
    stl_res = STL(grp, period=STL_PERIOD, robust=True).fit()

    idx   = grp.index[-SHOW_LAST:]
    raw   = grp.values[-SHOW_LAST:]
    trend = stl_res.trend[-SHOW_LAST:]
    seas  = stl_res.seasonal[-SHOW_LAST:]
    resid = stl_res.resid[-SHOW_LAST:]

    # ── Метрики ──
    var_t = np.var(stl_res.trend)
    var_s = np.var(stl_res.seasonal)
    var_r = np.var(stl_res.resid)
    total = var_t + var_s + var_r + 1e-9
    snr   = (var_t + var_s) / (var_r + 1e-9)
    acf_v = acf(grp.values, nlags=48, fft=True)

    # ── График ──
    fig = plt.figure(figsize=(14, 10))
    fig.suptitle(
        f"STL-декомпозиция  |  route_id = {route_id}  |  последние {SHOW_LAST} точек ({SHOW_LAST//2}ч)",
        fontsize=13, fontweight="bold"
    )
    gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.55, wspace=0.35)

    colors = {"raw": "#1976D2", "trend": "#388E3C", "seas": "#F57C00", "resid": "#C62828"}

    # Исходный ряд
    ax0 = fig.add_subplot(gs[0, :])
    ax0.plot(idx, raw,   color=colors["raw"],   lw=1.2, label="Исходный ряд")
    ax0.plot(idx, trend, color=colors["trend"], lw=1.8, linestyle="--", label="Тренд")
    ax0.set_ylabel(TARGET_COL)
    ax0.legend(loc="upper left", fontsize=8)
    ax0.set_title("Наблюдаемый ряд + тренд", fontsize=9)
    ax0.grid(alpha=0.3)

    # Сезонная компонента
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.plot(idx, seas, color=colors["seas"], lw=1.2)
    ax1.axhline(0, color="black", lw=0.5)
    ax1.set_title(f"Сезонность (период={STL_PERIOD})", fontsize=9)
    ax1.set_ylabel("seasonal")
    ax1.grid(alpha=0.3)

    # Остаток
    ax2 = fig.add_subplot(gs[1, 1])
    ax2.plot(idx, resid, color=colors["resid"], lw=1.0, alpha=0.8)
    ax2.axhline(0, color="black", lw=0.5)
    ax2.axhline(+2*resid.std(), color="red", lw=0.7, linestyle=":")
    ax2.axhline(-2*resid.std(), color="red", lw=0.7, linestyle=":")
    ax2.set_title("Остаток (resid ± 2σ)", fontsize=9)
    ax2.set_ylabel("resid")
    ax2.grid(alpha=0.3)

    # Доля дисперсии — bar chart
    ax3 = fig.add_subplot(gs[2, 0])
    labels_v = ["Тренд", "Сезонность", "Остаток"]
    vals_v   = [var_t/total, var_s/total, var_r/total]
    bar_colors = [colors["trend"], colors["seas"], colors["resid"]]
    bars = ax3.bar(labels_v, vals_v, color=bar_colors, edgecolor="white")
    for b, v in zip(bars, vals_v):
        ax3.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.2%}", ha="center", fontsize=9)
    ax3.set_ylim(0, 1.05)
    ax3.set_title("Доля объяснённой дисперсии", fontsize=9)
    ax3.set_ylabel("Доля")
    ax3.grid(axis="y", alpha=0.3)

    # ACF lag 1..48 (суточный цикл)
    ax4 = fig.add_subplot(gs[2, 1])
    ci  = 2 / np.sqrt(n)
    ax4.bar(range(1, 49), acf_v[1:49],
            color=["#1976D2" if abs(v) > ci else "#90CAF9" for v in acf_v[1:49]])
    ax4.axhline(+ci, color="red", linestyle="--", lw=0.8)
    ax4.axhline(-ci, color="red", linestyle="--", lw=0.8)
    ax4.set_title("ACF (lag 1–48)", fontsize=9)
    ax4.set_xlabel("lag")
    ax4.set_ylabel("ACF")
    ax4.grid(alpha=0.3)

    # Гистограмма остатка
    ax5 = fig.add_subplot(gs[3, :])
    ax5.hist(stl_res.resid, bins=40, color=colors["resid"], edgecolor="white", alpha=0.8)
    ax5.axvline(0, color="black", lw=1)
    ax5.set_title("Распределение остатка (весь ряд)", fontsize=9)
    ax5.set_xlabel("resid")
    ax5.set_ylabel("count")
    ax5.grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # ── Текстовая сводка ──
    print(f"\n{'─'*45}")
    print(f"  route_id         : {route_id}  (n={n} точек)")
    print(f"  Тренд (дисп.)    : {var_t/total:.1%}")
    print(f"  Сезонность       : {var_s/total:.1%}")
    print(f"  Остаток          : {var_r/total:.1%}")
    print(f"  SNR (T+S)/R      : {snr:.2f}  {'✅ хорошо' if snr > 1 else '⚠️ шумно'}")
    print(f"  ACF(lag=48)      : {acf_v[48]:.3f}  {'✅ сильная суточная' if acf_v[48] > 0.3 else '—'}")
    print(f"  Resid std        : {stl_res.resid.std():.3f}")
    print(f"{'─'*45}")


# ── Виджет ──
out = widgets.Output()

route_input = widgets.BoundedIntText(
    value=all_routes[0],
    min=min(all_routes),
    max=max(all_routes),
    step=1,
    description="route_id:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)

btn = widgets.Button(
    description="▶ Показать STL",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)

def on_click(_):
    with out:
        clear_output(wait=True)
        plot_stl(route_input.value)

btn.on_click(on_click)

display(widgets.HBox([route_input, btn]))
display(out)

In [ ]:
"""
Визуализация "плохо разложимого" ряда для слайда.
Запускается в Jupyter или как скрипт.

Зависимости: numpy, pandas, matplotlib, statsmodels
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import het_arch
import warnings
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════
#  ← ВСТАВЬ СЮДА НУЖНЫЙ route_id
# ══════════════════════════════════════════════════════════════
ROUTE_ID   = 192
TRAIN_PATH = "train_solo_track.parquet"
TARGET_COL = "target_1h"
FREQ       = "30min"
STL_PERIOD = 48          # суточная сезонность (48 × 30 мин = 24 ч)
SHOW_LAST  = 48 * 7      # последние 7 суток в визуале
SAVE_PNG   = True        # True → сохранит PNG рядом со скриптом
# ══════════════════════════════════════════════════════════════

# ── Палитра (под тёмно-синий слайд) ──────────────────────────
SLIDE_BG  = "#0D0A2E"
SLIDE_BG2 = "#100D38"
C_RAW    = "#7EC8E3"   # голубой  — исходный ряд
C_TREND  = "#C4ADFF"   # лавандовый — тренд
C_SEAS   = "#64B5F6"   # синий — сезонность
C_RESID  = "#FF5370"   # красный — остаток
C_ACCENT = "#00E5FF"   # яркий cyan — акцент / метрики
C_TEXT   = "#FFFFFF"
C_MUTED  = "#A0A0C8"
C_GRID   = "#FFFFFF18"

# ── Загрузка ──────────────────────────────────────────────────
train_df = pd.read_parquet(TRAIN_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

grp = (
    train_df[train_df["route_id"] == ROUTE_ID]
    .sort_values("timestamp")
    .set_index("timestamp")[TARGET_COL]
    .asfreq(FREQ)
    .interpolate(method="time")
    .bfill().ffill()
)
n = len(grp)
print(f"  route_id={ROUTE_ID}  |  n={n} точек")

# ── STL ───────────────────────────────────────────────────────
stl_res = STL(grp, period=STL_PERIOD, robust=True).fit()

idx   = grp.index[-SHOW_LAST:]
y_raw = grp.values[-SHOW_LAST:]
y_tre = stl_res.trend[-SHOW_LAST:]
y_sea = stl_res.seasonal[-SHOW_LAST:]
y_res = stl_res.resid[-SHOW_LAST:]

# ── Метрики ───────────────────────────────────────────────────
var_t = np.var(stl_res.trend)
var_s = np.var(stl_res.seasonal)
var_r = np.var(stl_res.resid)
total = var_t + var_s + var_r + 1e-9

resid_frac = var_r / total
cv         = grp.std() / (grp.mean() + 1e-9)
p50        = np.percentile(grp, 50)
p99        = np.percentile(grp, 99)
ratio      = p99 / (p50 + 1e-9)

arch_stat, arch_p, _, _ = het_arch(stl_res.resid, nlags=10)
arch_detected = arch_p < 0.05

print(f"  STL resid  = {resid_frac:.0%} дисперсии")
print(f"  CV         = {cv:.2f}")
print(f"  p99/p50    = {ratio:.2f}×")
print(f"  ARCH p-val = {arch_p:.4f}  →  {'обнаружен ✓' if arch_detected else 'не обнаружен'}")

# ── Оформление ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": SLIDE_BG, "axes.facecolor": SLIDE_BG2,
    "axes.edgecolor": "#FFFFFF25", "axes.labelcolor": C_MUTED,
    "xtick.color": C_MUTED, "ytick.color": C_MUTED,
    "text.color": C_TEXT, "grid.color": C_GRID,
    "grid.linewidth": 0.5, "font.family": "DejaVu Sans", "font.size": 9,
})

fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor(SLIDE_BG)
gs = gridspec.GridSpec(3, 3, figure=fig,
                       hspace=0.58, wspace=0.42,
                       left=0.055, right=0.97, top=0.88, bottom=0.07)

fig.text(0.5, 0.95,
         f"Почему ряд трудно предсказать  ·  route_id = {ROUTE_ID}",
         ha="center", va="top", fontsize=15, fontweight="bold", color=C_TEXT)

# ── 0. Исходный ряд + тренд ──────────────────────────────────
ax0 = fig.add_subplot(gs[0, :])
ax0.plot(idx, y_raw, color=C_RAW,   lw=1.0, alpha=0.85, label="Исходный ряд")
ax0.plot(idx, y_tre, color=C_TREND, lw=2.0, linestyle="--", label="Тренд (STL)")
spike_thresh = np.percentile(y_raw, 93)
ax0.fill_between(idx, spike_thresh, y_raw,
                 where=(y_raw > spike_thresh),
                 color=C_RESID, alpha=0.40, label="Пики > p93")
ax0.set_ylabel(TARGET_COL, color=C_MUTED, fontsize=8)
ax0.legend(loc="upper left", fontsize=8, framealpha=0.15, edgecolor="none")
ax0.set_title("Наблюдаемый ряд: нестабильная дисперсия и случайные пики",
              fontsize=9, color=C_MUTED, pad=4)
ax0.grid(alpha=0.15); ax0.set_xlim(idx[0], idx[-1])

# ── 1. Сезонность ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1, 0])
ax1.plot(idx, y_sea, color=C_SEAS, lw=1.0, alpha=0.85)
ax1.axhline(0, color="#FFFFFF20", lw=0.7)
ax1.set_title(f"Сезонность (период={STL_PERIOD})", fontsize=9, color=C_MUTED)
ax1.grid(alpha=0.15); ax1.set_xlim(idx[0], idx[-1])
for sp in ["top","right"]: ax1.spines[sp].set_visible(False)

# ── 2. Остаток ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 1])
ax2.plot(idx, y_res, color=C_RESID, lw=0.8, alpha=0.85)
ax2.axhline(0, color="#FFFFFF20", lw=0.7)
sigma = y_res.std()
ax2.axhline(+2*sigma, color=C_ACCENT, lw=0.9, linestyle=":", alpha=0.8)
ax2.axhline(-2*sigma, color=C_ACCENT, lw=0.9, linestyle=":", alpha=0.8)
ax2.set_title(f"STL-остаток  ·  {resid_frac:.0%} дисперсии ряда",
              fontsize=9, color=C_RESID)
ax2.grid(alpha=0.15); ax2.set_xlim(idx[0], idx[-1])
for sp in ["top","right"]: ax2.spines[sp].set_visible(False)

# ── 3. Бар-чарт дисперсий ─────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2])
labels_v = ["Тренд", "Сезонность", "Остаток"]
vals_v   = [var_t/total, var_s/total, var_r/total]
bar_cols = [C_TREND, C_SEAS, C_RESID]
bars = ax3.bar(labels_v, vals_v, color=bar_cols, edgecolor="none", width=0.52)
for b, v in zip(bars, vals_v):
    ax3.text(b.get_x() + b.get_width()/2, v + 0.015,
             f"{v:.0%}", ha="center", fontsize=10, fontweight="bold",
             color=b.get_facecolor())
ax3.set_ylim(0, 1.12)
ax3.set_title("Доля объяснённой дисперсии", fontsize=9, color=C_MUTED)
ax3.grid(axis="y", alpha=0.15); ax3.set_axisbelow(True)
for sp in ["top","right"]: ax3.spines[sp].set_visible(False)

# ── 4. ARCH: скользящая σ остатка ─────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
abs_r = np.abs(stl_res.resid)
roll  = pd.Series(abs_r).rolling(STL_PERIOD, min_periods=1).std().values
all_t = np.arange(len(abs_r))
ax4.fill_between(all_t, abs_r, alpha=0.25, color=C_RESID)
ax4.plot(all_t, roll, color=C_ACCENT, lw=1.6,
         label=f"rolling σ (окно={STL_PERIOD})")
ax4.set_title("ARCH-эффект: скользящая σ остатка", fontsize=9, color=C_MUTED)
ax4.grid(alpha=0.15); ax4.set_xlabel("Точки ряда", fontsize=8)
ax4.legend(fontsize=7, framealpha=0.15, edgecolor="none")
for sp in ["top","right"]: ax4.spines[sp].set_visible(False)

# ── 5. Тяжёлые хвосты ────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
vals_all = grp.values
ax5.hist(vals_all, bins=55, color=C_RAW, edgecolor="none", alpha=0.75)
ax5.axvline(p50, color=C_ACCENT, lw=1.7, linestyle="--",
            label=f"p50 = {p50:.1f}")
ax5.axvline(p99, color=C_RESID,  lw=1.7, linestyle="--",
            label=f"p99 = {p99:.1f}")
ax5.set_title(f"Тяжёлые хвосты  ·  p99/p50 = {ratio:.1f}×",
              fontsize=9, color=C_MUTED)
ax5.grid(axis="y", alpha=0.15)
ax5.legend(fontsize=7, framealpha=0.15, edgecolor="none")
for sp in ["top","right"]: ax5.spines[sp].set_visible(False)

# ── 6. Карточка с метриками ───────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6.set_xlim(0, 1); ax6.set_ylim(0, 1); ax6.axis("off")
ax6.set_title("Ключевые метрики", fontsize=9, color=C_MUTED)

metrics = [
    ("STL residual",  f"{resid_frac:.0%} дисперсии",                 C_RESID),
    ("CV",            f"{cv:.2f}",                                    C_RAW),
    ("p99 / p50",     f"{ratio:.1f}×",                                C_SEAS),
    ("ARCH-эффект",   "обнаружен ✓" if arch_detected else "не обнаружен", C_ACCENT),
]
for i, (label, val, col) in enumerate(metrics):
    y = 0.84 - i * 0.22
    ax6.text(0.08, y,       label, fontsize=8,  color=C_MUTED, va="top")
    ax6.text(0.08, y-0.10,  val,   fontsize=12, color=col,     va="top",
             fontweight="bold")

# ── Подпись внизу ─────────────────────────────────────────────
fig.text(0.5, 0.015,
         '"Линейные модели и классические декомпозиции не справятся"',
         ha="center", va="bottom", fontsize=11, color=C_ACCENT,
         fontstyle="italic", alpha=0.9)

plt.tight_layout(rect=[0, 0.04, 1, 0.93])

if SAVE_PNG:
    fname = f"stl_hard_route_{ROUTE_ID}.png"
    plt.savefig(fname, dpi=160, bbox_inches="tight",
                facecolor=SLIDE_BG, edgecolor="none")
    print(f"\n  Сохранено → {fname}")

plt.show()

In [ ]:
[485, 192, 702, 999, 446, 782, 725, 768, 445, 817, 329, 594, 123, 948, 263]

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")

TRAIN_PATH = "train_solo_track.parquet"
TARGET_COL = "target_1h"
STATUS_COLS = [f"status_{i}" for i in range(1, 7)]

train_df = pd.read_parquet(TRAIN_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Загружено строк: {len(train_df):,}  | Маршрутов: {train_df['route_id'].nunique()}")

from tqdm import tqdm

# ── 1. Базовая статистика по каждому статусу ──
print("\n" + "="*65)
print("1. БАЗОВАЯ СТАТИСТИКА СТАТУСОВ vs TARGET")
print("="*65)

base_stats = {}
for col in STATUS_COLS + [TARGET_COL]:
    s = train_df[col]
    base_stats[col] = {
        "mean":      s.mean(),
        "median":    s.median(),
        "std":       s.std(),
        "zero_frac": (s == 0).mean(),
        "p25":       s.quantile(0.25),
        "p75":       s.quantile(0.75),
        "p99":       s.quantile(0.99),
        "max":       s.max(),
        "cv":        s.std() / (s.mean() + 1e-9),
    }

base_df = pd.DataFrame(base_stats).T
print(base_df.round(2).to_string())

# ── 2. Корреляции с таргетом (Pearson + Spearman) ──
print("\n" + "="*65)
print("2. КОРРЕЛЯЦИИ STATUS → TARGET_1H")
print("="*65)

corr_rows = []
for col in STATUS_COLS:
    mask = train_df[[col, TARGET_COL]].notna().all(axis=1)
    x = train_df.loc[mask, col].values
    y = train_df.loc[mask, TARGET_COL].values

    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = spearmanr(x, y)
    corr_rows.append({
        "status":         col,
        "pearson_r":      round(r_p, 4),
        "pearson_p":      p_p,
        "spearman_r":     round(r_s, 4),
        "spearman_p":     p_s,
        "significant":    p_s < 0.001,
    })

corr_df = pd.DataFrame(corr_rows).set_index("status")
print(corr_df[["pearson_r","spearman_r","significant"]].to_string())

# ── 3. Лаговые корреляции: status[t-k] → target[t] ──
print("\n" + "="*65)
print("3. ЛАГОВЫЕ КОРРЕЛЯЦИИ (Spearman) STATUS[t-k] → TARGET[t]")
print("= lag 0 = одновременно, lag 1 = 30 мин назад, lag 2 = 1ч, lag 4 = 2ч =")
print("="*65)

LAG_LIST = [0, 1, 2, 3, 4, 6, 8, 12, 24, 48]
lag_results = {col: {} for col in STATUS_COLS}

for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Lag corr"):
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    y = grp[TARGET_COL].values
    for col in STATUS_COLS:
        x = grp[col].values
        for lag in LAG_LIST:
            if lag == 0:
                xi, yi = x, y
            else:
                xi, yi = x[:-lag], y[lag:]
            if len(xi) < 30:
                continue
            r, _ = spearmanr(xi, yi)
            if not np.isnan(r):
                lag_results[col].setdefault(lag, []).append(r)

print(f"\n{'status':<12}", end="")
for lag in LAG_LIST:
    print(f"  lag{lag:>3}", end="")
print()

lag_summary = {}
for col in STATUS_COLS:
    print(f"{col:<12}", end="")
    lag_summary[col] = {}
    for lag in LAG_LIST:
        vals = lag_results[col].get(lag, [np.nan])
        med = np.nanmedian(vals)
        lag_summary[col][lag] = med
        star = "**" if abs(med) > 0.3 else ("*" if abs(med) > 0.15 else "  ")
        print(f"  {med:+.3f}{star}", end="")
    print()

print("\n** = |r|>0.3 (сильная),  * = |r|>0.15 (умеренная)")

# ── 4. Взаимные корреляции между статусами ──
print("\n" + "="*65)
print("4. КОРРЕЛЯЦИИ МЕЖДУ САМИМИ СТАТУСАМИ (Spearman, глобально)")
print("="*65)

status_data = train_df[STATUS_COLS]
inter_corr = status_data.corr(method="spearman")
print(inter_corr.round(3).to_string())

# ── 5. Доля нулей в разрезе: когда target=0 vs target>0 ──
print("\n" + "="*65)
print("5. СРЕДНЕЕ ЗНАЧЕНИЕ СТАТУСОВ ПРИ target=0 vs target>0")
print("="*65)

df_zero   = train_df[train_df[TARGET_COL] == 0]
df_nonzero= train_df[train_df[TARGET_COL]  > 0]

for col in STATUS_COLS:
    m0 = df_zero[col].mean()
    m1 = df_nonzero[col].mean()
    ratio = m1 / (m0 + 1e-9)
    print(f"  {col}:  target=0 → mean={m0:.1f}  |  target>0 → mean={m1:.1f}  |  ratio={ratio:.2f}x")

print(f"\n  Строк с target=0: {len(df_zero):,} ({len(df_zero)/len(train_df):.1%})")
print(f"  Строк с target>0: {len(df_nonzero):,} ({len(df_nonzero)/len(train_df):.1%})")

# ── 5. Доля нулей в разрезе: когда target=0 vs target>0 ──
print("\n" + "="*65)
print("5. СРЕДНЕЕ ЗНАЧЕНИЕ СТАТУСОВ ПРИ target=0 vs target>0")
print("="*65)

df_zero   = train_df[train_df[TARGET_COL] == 0]
df_nonzero= train_df[train_df[TARGET_COL]  > 0]

for col in STATUS_COLS:
    m0 = df_zero[col].mean()
    m1 = df_nonzero[col].mean()
    ratio = m1 / (m0 + 1e-9)
    print(f"  {col}:  target=0 → mean={m0:.1f}  |  target>0 → mean={m1:.1f}  |  ratio={ratio:.2f}x")

print(f"\n  Строк с target=0: {len(df_zero):,} ({len(df_zero)/len(train_df):.1%})")
print(f"  Строк с target>0: {len(df_nonzero):,} ({len(df_nonzero)/len(train_df):.1%})")
# ── 6. Сумма статусов 1-3 vs 4-6 vs target ──
print("\n" + "="*65)
print("6. СУММА ГРУПП СТАТУСОВ vs TARGET")
print("="*65)

train_df["sum_current_wh"]  = train_df[["status_1","status_2","status_3"]].sum(axis=1)  # текущий склад
train_df["sum_prev_wh"]     = train_df[["status_4","status_5","status_6"]].sum(axis=1)  # предыдущий склад
train_df["sum_all_status"]  = train_df[STATUS_COLS].sum(axis=1)

for col in ["sum_current_wh", "sum_prev_wh", "sum_all_status"]:
    r_s, _ = spearmanr(train_df[col], train_df[TARGET_COL])
    r_p, _ = stats.pearsonr(train_df[col], train_df[TARGET_COL])
    print(f"  {col:<22}: pearson={r_p:+.4f}  spearman={r_s:+.4f}")

    # ── 7. Групповой анализ по перцентилям суммы статусов ──
print("\n" + "="*65)
print("7. TARGET ~ PERCENTILE(sum_all_status)")
print("="*65)

train_df["status_bin"] = pd.qcut(train_df["sum_all_status"], q=10, labels=False, duplicates="drop")
grp_stats = (
    train_df.groupby("status_bin")[TARGET_COL]
    .agg(["mean","median","std","count"])
    .round(1)
)
print(grp_stats.to_string())

# ── 8. Какой статус наиболее предсказателен? (лучший лаг) ──
print("\n" + "="*65)
print("8. ТОП СТАТУСОВ ПО МАКСИМАЛЬНОЙ ЛАГОВОЙ КОРРЕЛЯЦИИ")
print("="*65)

best_lag_per_status = {}
for col in STATUS_COLS:
    best_lag = max(lag_summary[col], key=lambda l: abs(lag_summary[col][l]))
    best_r   = lag_summary[col][best_lag]
    best_lag_per_status[col] = (best_lag, best_r)
    print(f"  {col}: best_lag={best_lag:>3} (={best_lag*30}мин)  r={best_r:+.4f}")


# ── 9. ФИНАЛЬНОЕ САММАРИ ДЛЯ LLM ──
print("\n" + "="*65)
print("САММАРИ ДЛЯ LLM (скопируй всё ниже)")
print("="*65)

zero_frac = len(df_zero)/len(train_df)
r_cur,  _ = spearmanr(train_df["sum_current_wh"],  train_df[TARGET_COL])
r_prev, _ = spearmanr(train_df["sum_prev_wh"],     train_df[TARGET_COL])
r_all,  _ = spearmanr(train_df["sum_all_status"],  train_df[TARGET_COL])

print(f"""
=== ОПИСАНИЕ ДАННЫХ ===
Задача: прогноз target_1h (объём отгрузок по маршруту за последний час), 
интервал 30 мин, ~1000 маршрутов.

=== СТРУКТУРА СТАТУСОВ ===
status_1, status_2, status_3 — кол-во товаров на ТЕКУЩЕМ складе (этапы обработки),
  прошедших обработку за последние 30 мин по данному маршруту.
status_4, status_5, status_6 — кол-во товаров на ПРЕДЫДУЩЕМ складе,
  направляющихся на текущий, за последние 30 мин.
ВАЖНО: в тесте статусы НЕДОСТУПНЫ — это только исторические ковариаты.

=== БАЗОВАЯ СТАТИСТИКА ===
{base_df[['mean','median','std','zero_frac','cv']].round(3).to_string()}

=== КОРРЕЛЯЦИИ С TARGET ===
{corr_df[['pearson_r','spearman_r']].to_string()}

Сумма статусов текущего склада (1-3): spearman={r_cur:+.4f}
Сумма статусов предыдущего склада (4-6): spearman={r_prev:+.4f}
Сумма всех статусов: spearman={r_all:+.4f}

=== ЛАГОВЫЕ КОРРЕЛЯЦИИ (медиана Spearman по маршрутам) ===
lag = кол-во 30-мин интервалов до момента target
{"".join([f"  {col}: best_lag={best_lag_per_status[col][0]} ({best_lag_per_status[col][0]*30}мин)  r={best_lag_per_status[col][1]:+.4f}" + chr(10) for col in STATUS_COLS])}
=== НУЛИ ===
Доля строк с target=0: {zero_frac:.1%}
При target=0 статусы {"ниже" if df_zero[STATUS_COLS].mean().mean() < df_nonzero[STATUS_COLS].mean().mean() else "не ниже"} чем при target>0
""")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import spearmanr

FREQ      = "30min"
SHOW_LAST = 48       # точек на графике
LAG_H6    = 12       # 6 часов = 12 интервалов по 30 мин

all_routes = sorted(train_df["route_id"].unique().tolist())

def plot_status_vs_target(route_id):
    grp = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[["status_5", "target_1h"]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )

    n = len(grp)
    if n < LAG_H6 + SHOW_LAST:
        print(f"⚠️ Мало данных для route_id={route_id} (n={n})")
        return

    # Берём последние 48 точек + 12 хвост для сдвинутого статуса
    window = grp.iloc[-(SHOW_LAST + LAG_H6):]
    idx      = window.index[LAG_H6:]          # временная ось таргета
    target   = window["target_1h"].values[LAG_H6:]
    s5_now   = window["status_5"].values[LAG_H6:]     # status_5 без сдвига
    s5_lag   = window["status_5"].values[:SHOW_LAST]  # status_5 сдвинутый на +6ч

    # Корреляция за весь ряд при lag=12
    full_s5  = grp["status_5"].values
    full_tgt = grp["target_1h"].values
    r_lag0, _ = spearmanr(full_s5,               full_tgt)
    r_lag12,_ = spearmanr(full_s5[:-LAG_H6],     full_tgt[LAG_H6:])

    # ── График ──
    fig, axes = plt.subplots(3, 1, figsize=(13, 10),
                              gridspec_kw={"height_ratios": [2.5, 2.5, 1.2]})
    fig.suptitle(f"Route {route_id}  —  status_5 vs target_1h  |  последние {SHOW_LAST} точек (24ч)",
                 fontsize=12, fontweight="bold")

    # ── Панель 1: оба ряда на двойной оси ──
    ax1 = axes[0]
    ax1r = ax1.twinx()

    l1, = ax1.plot(idx, s5_lag, color="#FF7043", lw=1.5, alpha=0.85,
                   label=f"status_5  [t−6ч]  (сдвиг +{LAG_H6} лагов)")
    l2, = ax1r.plot(idx, target, color="#1976D2", lw=1.8,
                    label="target_1h  [t]")

    ax1.set_ylabel("status_5", color="#FF7043")
    ax1r.set_ylabel("target_1h", color="#1976D2")
    ax1.tick_params(axis="y", labelcolor="#FF7043")
    ax1r.tick_params(axis="y", labelcolor="#1976D2")
    ax1.set_title(
        f"status_5[t−6ч] vs target_1h[t]  │  "
        f"Spearman lag=0: {r_lag0:+.3f}   lag=12 (+6ч): {r_lag12:+.3f}",
        fontsize=9
    )
    ax1.legend(handles=[l1, l2], loc="upper left", fontsize=8)
    ax1.grid(alpha=0.25)

    # ── Панель 2: status_5 без сдвига (для сравнения) ──
    ax2 = axes[1]
    ax2r = ax2.twinx()

    l3, = ax2.plot(idx, s5_now, color="#FFA726", lw=1.2, alpha=0.7,
                   linestyle="--", label="status_5  [t]  (без сдвига)")
    l4, = ax2r.plot(idx, target, color="#1976D2", lw=1.8,
                    label="target_1h  [t]")

    ax2.set_ylabel("status_5", color="#FFA726")
    ax2r.set_ylabel("target_1h", color="#1976D2")
    ax2.tick_params(axis="y", labelcolor="#FFA726")
    ax2r.tick_params(axis="y", labelcolor="#1976D2")
    ax2.set_title("status_5[t] vs target_1h[t]  (без сдвига — для сравнения)", fontsize=9)
    ax2.legend(handles=[l3, l4], loc="upper left", fontsize=8)
    ax2.grid(alpha=0.25)

    # ── Панель 3: scatter status_5[t-12] vs target[t] ──
    ax3 = axes[2]
    ax3.scatter(s5_lag, target, alpha=0.45, s=18, color="#E53935", edgecolors="none")

    # Линия тренда
    if s5_lag.std() > 0:
        m, b = np.polyfit(s5_lag, target, 1)
        xline = np.linspace(s5_lag.min(), s5_lag.max(), 100)
        ax3.plot(xline, m * xline + b, color="black", lw=1.2, linestyle="--")

    ax3.set_xlabel("status_5  [t−6ч]")
    ax3.set_ylabel("target_1h  [t]")
    ax3.set_title(f"Scatter: status_5[t−6ч] → target_1h[t]  │  Spearman r={r_lag12:+.3f}", fontsize=9)
    ax3.grid(alpha=0.25)

    # Вердикт
    if abs(r_lag12) > abs(r_lag0) and abs(r_lag12) > 0.25:
        verdict = f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: lag=12 ({r_lag12:+.3f}) > lag=0 ({r_lag0:+.3f})"
        color   = "#2E7D32"
    elif abs(r_lag12) > 0.25:
        verdict = f"🔶 Умеренная связь при lag=12 (r={r_lag12:+.3f}), но lag=0 сильнее ({r_lag0:+.3f})"
        color   = "#E65100"
    else:
        verdict = f"❌ Гипотеза слабая: r={r_lag12:+.3f} при lag=12 — связь незначимая"
        color   = "#B71C1C"

    fig.text(0.5, -0.01, verdict, ha="center", fontsize=10,
             fontweight="bold", color=color)

    plt.tight_layout()
    plt.show()


# ── Виджет ──
out = widgets.Output()

route_input = widgets.BoundedIntText(
    value=all_routes[0],
    min=min(all_routes), max=max(all_routes), step=1,
    description="route_id:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)

btn = widgets.Button(
    description="▶ Показать",
    button_style="primary",
    layout=widgets.Layout(width="140px"),
)

def on_click(_):
    with out:
        clear_output(wait=True)
        plot_status_vs_target(route_input.value)

btn.on_click(on_click)

display(widgets.HBox([route_input, btn]))
display(out)

In [ ]:
# ── Ячейка 1: ищем лучший лаг в скользящем окне перед последними 48 точками ──
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm import tqdm

FREQ      = "30min"
SHOW_LAST = 48    # точки для графика (не трогаем)
MAX_LAG   = 96    # максимальный лаг для поиска

# Размер окна поиска лага (меняй здесь)
WINDOW_OPTIONS = {
    "1 день (48)"    : 48,
    "3 дня (144)"    : 144,
    "1 неделя (336)" : 336,
}

def find_best_lag_in_window(s5: np.ndarray, tgt: np.ndarray,
                             max_lag: int, min_obs: int = 30):
    """Ищет лаг с максимальным |Spearman r| внутри переданного окна."""
    best_lag, best_r = 0, 0.0
    profile = []
    for lag in range(0, max_lag + 1):
        xi = s5[:-lag] if lag > 0 else s5
        yi = tgt[lag:] if lag > 0 else tgt
        if len(xi) < min_obs:
            profile.append(np.nan)
            continue
        r, _ = spearmanr(xi, yi)
        r = float(r) if not np.isnan(r) else 0.0
        profile.append(r)
        if abs(r) > abs(best_r):
            best_r, best_lag = r, lag
    return best_lag, best_r, profile


# Считаем для всех окон сразу
all_results = {name: {} for name in WINDOW_OPTIONS}

for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Поиск лагов"):
    grp = (
        grp.sort_values("timestamp")
        .set_index("timestamp")[["status_5", "target_1h"]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )
    n = len(grp)

    for name, win_size in WINDOW_OPTIONS.items():
        # Окно поиска: [-(SHOW_LAST + win_size + MAX_LAG) : -(SHOW_LAST)]
        # чтобы при любом лаге до MAX_LAG хватало точек
        end_idx   = n - SHOW_LAST
        start_idx = max(0, end_idx - win_size - MAX_LAG)
        window    = grp.iloc[start_idx:end_idx]

        if len(window) < MAX_LAG + 30:
            all_results[name][route_id] = {"best_lag": 0, "best_r": np.nan, "lag_profile": []}
            continue

        s5  = window["status_5"].values
        tgt = window["target_1h"].values
        best_lag, best_r, profile = find_best_lag_in_window(s5, tgt, MAX_LAG)
        all_results[name][route_id] = {
            "best_lag":    best_lag,
            "best_r":      best_r,
            "lag_profile": profile,
        }

# ── Сводка ──
print("\n" + "="*60)
print("СРАВНЕНИЕ ОКОН ПОИСКА ЛАГА (status_5 → target_1h)")
print("="*60)

for name, res in all_results.items():
    lags = pd.Series({r: v["best_lag"] for r, v in res.items()})
    cors = pd.Series({r: v["best_r"]   for r, v in res.items()})
    print(f"\n  Окно: {name}")
    print(f"    Медиана лага  : {lags.median():.0f} × 30мин = {lags.median()*0.5:.1f} ч")
    print(f"    Медиана |r|   : {cors.abs().median():.4f}")
    print(f"    |r| > 0.3     : {(cors.abs()>0.3).sum()} маршрутов")
    bins = [0,6,12,24,48,96]
    blab = ["0–3ч","3–6ч","6–12ч","12–24ч","24–48ч"]
    for i, lb in enumerate(blab):
        cnt = ((lags >= bins[i]) & (lags < bins[i+1])).sum()
        print(f"      {lb:<10} {cnt:>4} маршрутов")

In [ ]:
# ── Ячейка 2: виджет с выбором окна поиска ──
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def plot_best_lag_window(route_id, window_name):
    info     = all_results[window_name][route_id]
    best_lag = info["best_lag"]
    best_r   = info["best_r"]
    profile  = np.array(info["lag_profile"], dtype=float)

    grp = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[["status_5", "target_1h"]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )

    n    = len(grp)
    need = SHOW_LAST + best_lag
    if n < need + 10:
        print(f"⚠️ Мало данных (n={n}, нужно ≥{need})")
        return

    # График окно: последние 48 точек таргета
    window   = grp.iloc[-(SHOW_LAST + best_lag):]
    idx      = window.index[best_lag:]
    target   = window["target_1h"].values[best_lag:]
    s5_shift = window["status_5"].values[:SHOW_LAST]   # status_5[t - best_lag]
    s5_now   = window["status_5"].values[best_lag:]    # status_5[t]

    # lag=0 корреляция в том же окне для сравнения
    win_size = WINDOW_OPTIONS[window_name]
    end_idx  = n - SHOW_LAST
    start_idx= max(0, end_idx - win_size - MAX_LAG)
    search_w = grp.iloc[start_idx:end_idx]
    r0, _    = spearmanr(search_w["status_5"], search_w["target_1h"])

    # ── Фигура ──
    fig = plt.figure(figsize=(14, 11))
    fig.suptitle(
        f"Route {route_id}  │  Окно поиска: {window_name}  │  "
        f"Best lag = {best_lag} ({best_lag*30//60}ч {best_lag*30%60:02d}м)  │  r = {best_r:+.3f}",
        fontsize=11, fontweight="bold"
    )
    gs = plt.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.35)

    # ── 1. Основной: сдвинутый статус + таргет ──
    ax1  = fig.add_subplot(gs[0, :])
    ax1r = ax1.twinx()
    l1, = ax1.plot(idx, s5_shift, color="#FF7043", lw=1.6,
                   label=f"status_5 [t−{best_lag}] = [{best_lag*30//60}ч {best_lag*30%60:02d}м назад]")
    l2, = ax1r.plot(idx, target, color="#1976D2", lw=1.8,
                    label="target_1h [t]")
    ax1.set_ylabel("status_5", color="#FF7043")
    ax1r.set_ylabel("target_1h", color="#1976D2")
    ax1.tick_params(axis="y", labelcolor="#FF7043")
    ax1r.tick_params(axis="y", labelcolor="#1976D2")
    ax1.legend(handles=[l1, l2], loc="upper left", fontsize=8)
    ax1.set_title(
        f"status_5[t−{best_lag}] vs target_1h[t]  │  "
        f"r(best_lag)={best_r:+.3f}   r(lag=0)={r0:+.3f}",
        fontsize=9
    )
    ax1.grid(alpha=0.25)

    # ── 2. Профиль корреляций ──
    ax2 = fig.add_subplot(gs[1, 0])
    lags_x = np.arange(len(profile))
    ax2.plot(lags_x, profile, color="#546E7A", lw=1.2)
    ax2.axvline(best_lag, color="#E53935", lw=1.8, linestyle="--",
                label=f"best lag={best_lag}")
    ax2.axhline(0, color="black", lw=0.5)
    ax2.fill_between(lags_x, profile, 0,
                     where=profile > 0, alpha=0.15, color="#1976D2")
    ax2.fill_between(lags_x, profile, 0,
                     where=profile < 0, alpha=0.15, color="#E53935")
    # Подписи осей X в часах
    tick_lags = [0, 12, 24, 48, 72, 96]
    ax2.set_xticks([t for t in tick_lags if t < len(profile)])
    ax2.set_xticklabels([f"{t//2}ч" for t in tick_lags if t < len(profile)])
    ax2.set_xlabel("lag")
    ax2.set_ylabel("Spearman r")
    ax2.set_title(f"Профиль лагов (окно: {window_name})", fontsize=9)
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.25)

    # ── 3. Scatter ──
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.scatter(s5_shift, target, alpha=0.45, s=20,
                color="#FF7043", edgecolors="none")
    if s5_shift.std() > 0:
        m, b = np.polyfit(s5_shift, target, 1)
        xline = np.linspace(s5_shift.min(), s5_shift.max(), 100)
        ax3.plot(xline, m*xline+b, "k--", lw=1.2)
    ax3.set_xlabel(f"status_5 [t−{best_lag}]")
    ax3.set_ylabel("target_1h [t]")
    ax3.set_title(f"Scatter  │  r = {best_r:+.3f}", fontsize=9)
    ax3.grid(alpha=0.25)

    # ── 4. Все три окна — сравнение профилей ──
    ax4 = fig.add_subplot(gs[2, :])
    palette = {"1 день (48)": "#E53935", "3 дня (144)": "#FF9800", "1 неделя (336)": "#1976D2"}
    for wname, wres in all_results.items():
        p = np.array(wres[route_id]["lag_profile"], dtype=float)
        bl = wres[route_id]["best_lag"]
        if len(p) == 0:
            continue
        lx = np.arange(len(p))
        ax4.plot(lx, p, color=palette[wname], lw=1.2, alpha=0.85,
                 label=f"{wname}  best={bl} ({bl//2}ч)  r={wres[route_id]['best_r']:+.3f}")
        ax4.axvline(bl, color=palette[wname], lw=0.8, linestyle=":")
    ax4.axhline(0, color="black", lw=0.5)
    ax4.set_xticks([t for t in [0,12,24,48,72,96] if t < len(profile)])
    ax4.set_xticklabels([f"{t//2}ч" for t in [0,12,24,48,72,96] if t < len(profile)])
    ax4.set_xlabel("lag")
    ax4.set_ylabel("Spearman r")
    ax4.set_title("Сравнение профилей лагов по всем окнам поиска", fontsize=9)
    ax4.legend(fontsize=8, loc="upper right")
    ax4.grid(alpha=0.25)

    # Вердикт
    gain = abs(best_r) - abs(r0)
    if abs(best_r) > 0.4 and gain > 0.05:
        verdict = f"✅ Чёткий лаг {best_lag} ({best_lag*30//60}ч)  |  gain vs lag=0: +{gain:.3f}"
        col = "#1B5E20"
    elif abs(best_r) > 0.25:
        verdict = f"🔶 Умеренная связь r={best_r:+.3f} при lag={best_lag}  |  gain={gain:+.3f}"
        col = "#E65100"
    else:
        verdict = f"❌ Слабая связь r={best_r:+.3f} — status_5 слабо предсказывает target на этом окне"
        col = "#B71C1C"

    fig.text(0.5, -0.015, verdict, ha="center", fontsize=10,
             fontweight="bold", color=col)
    plt.tight_layout()
    plt.show()


# ── Виджет ──
out3 = widgets.Output()

route_input3 = widgets.BoundedIntText(
    value=all_routes[0], min=min(all_routes), max=max(all_routes), step=1,
    description="route_id:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)
window_select = widgets.Dropdown(
    options=list(WINDOW_OPTIONS.keys()),
    value="3 дня (144)",
    description="Окно:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="200px"),
)
btn3 = widgets.Button(
    description="▶ Показать",
    button_style="success",
    layout=widgets.Layout(width="140px"),
)

def on_click3(_):
    with out3:
        clear_output(wait=True)
        plot_best_lag_window(route_input3.value, window_select.value)

btn3.on_click(on_click3)
display(widgets.HBox([route_input3, window_select, btn3]))
display(out3)

In [ ]:
[485, 192, 702, 999, 446, 782, 725, 768, 445, 817, 329, 594, 123, 948, 263]

In [ ]:
# ── Ячейка 1: ищем лучший лаг в скользящем окне перед последними 48 точками ──
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm import tqdm

FREQ      = "30min"
SHOW_LAST = 48    # точки для графика (не трогаем)
MAX_LAG   = 96    # максимальный лаг для поиска

# Размер окна поиска лага (меняй здесь)
WINDOW_OPTIONS = {
    "1 день (48)"    : 48,
    "3 дня (144)"    : 144,
    "1 неделя (336)" : 336,
}

def find_best_lag_in_window(s5: np.ndarray, tgt: np.ndarray,
                             max_lag: int, min_obs: int = 30):
    """Ищет лаг с максимальным |Spearman r| внутри переданного окна."""
    best_lag, best_r = 0, 0.0
    profile = []
    for lag in range(0, max_lag + 1):
        xi = s5[:-lag] if lag > 0 else s5
        yi = tgt[lag:] if lag > 0 else tgt
        if len(xi) < min_obs:
            profile.append(np.nan)
            continue
        r, _ = spearmanr(xi, yi)
        r = float(r) if not np.isnan(r) else 0.0
        profile.append(r)
        if abs(r) > abs(best_r):
            best_r, best_lag = r, lag
    return best_lag, best_r, profile


# Считаем для всех окон сразу
all_results = {name: {} for name in WINDOW_OPTIONS}

for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Поиск лагов"):
    grp = (
        grp.sort_values("timestamp")
        .set_index("timestamp")[["status_6", "target_1h"]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )
    n = len(grp)

    for name, win_size in WINDOW_OPTIONS.items():
        # Окно поиска: [-(SHOW_LAST + win_size + MAX_LAG) : -(SHOW_LAST)]
        # чтобы при любом лаге до MAX_LAG хватало точек
        end_idx   = n - SHOW_LAST
        start_idx = max(0, end_idx - win_size - MAX_LAG)
        window    = grp.iloc[start_idx:end_idx]

        if len(window) < MAX_LAG + 30:
            all_results[name][route_id] = {"best_lag": 0, "best_r": np.nan, "lag_profile": []}
            continue

        s5  = window["status_6"].values
        tgt = window["target_1h"].values
        best_lag, best_r, profile = find_best_lag_in_window(s5, tgt, MAX_LAG)
        all_results[name][route_id] = {
            "best_lag":    best_lag,
            "best_r":      best_r,
            "lag_profile": profile,
        }

# ── Сводка ──
print("\n" + "="*60)
print("СРАВНЕНИЕ ОКОН ПОИСКА ЛАГА (status_6 → target_1h)")
print("="*60)

for name, res in all_results.items():
    lags = pd.Series({r: v["best_lag"] for r, v in res.items()})
    cors = pd.Series({r: v["best_r"]   for r, v in res.items()})
    print(f"\n  Окно: {name}")
    print(f"    Медиана лага  : {lags.median():.0f} × 30мин = {lags.median()*0.5:.1f} ч")
    print(f"    Медиана |r|   : {cors.abs().median():.4f}")
    print(f"    |r| > 0.3     : {(cors.abs()>0.3).sum()} маршрутов")
    bins = [0,6,12,24,48,96]
    blab = ["0–3ч","3–6ч","6–12ч","12–24ч","24–48ч"]
    for i, lb in enumerate(blab):
        cnt = ((lags >= bins[i]) & (lags < bins[i+1])).sum()
        print(f"      {lb:<10} {cnt:>4} маршрутов")

In [ ]:
# ── Ячейка 2: виджет с выбором окна поиска ──
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def plot_best_lag_window(route_id, window_name):
    info     = all_results[window_name][route_id]
    best_lag = info["best_lag"]
    best_r   = info["best_r"]
    profile  = np.array(info["lag_profile"], dtype=float)

    grp = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[["status_6", "target_1h"]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )

    n    = len(grp)
    need = SHOW_LAST + best_lag
    if n < need + 10:
        print(f"⚠️ Мало данных (n={n}, нужно ≥{need})")
        return

    # График окно: последние 48 точек таргета
    window   = grp.iloc[-(SHOW_LAST + best_lag):]
    idx      = window.index[best_lag:]
    target   = window["target_1h"].values[best_lag:]
    s5_shift = window["status_6"].values[:SHOW_LAST]   # status_6[t - best_lag]
    s5_now   = window["status_6"].values[best_lag:]    # status_6[t]

    # lag=0 корреляция в том же окне для сравнения
    win_size = WINDOW_OPTIONS[window_name]
    end_idx  = n - SHOW_LAST
    start_idx= max(0, end_idx - win_size - MAX_LAG)
    search_w = grp.iloc[start_idx:end_idx]
    r0, _    = spearmanr(search_w["status_6"], search_w["target_1h"])

    # ── Фигура ──
    fig = plt.figure(figsize=(14, 11))
    fig.suptitle(
        f"Route {route_id}  │  Окно поиска: {window_name}  │  "
        f"Best lag = {best_lag} ({best_lag*30//60}ч {best_lag*30%60:02d}м)  │  r = {best_r:+.3f}",
        fontsize=11, fontweight="bold"
    )
    gs = plt.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.35)

    # ── 1. Основной: сдвинутый статус + таргет ──
    ax1  = fig.add_subplot(gs[0, :])
    ax1r = ax1.twinx()
    l1, = ax1.plot(idx, s5_shift, color="#FF7043", lw=1.6,
                   label=f"status_6 [t−{best_lag}] = [{best_lag*30//60}ч {best_lag*30%60:02d}м назад]")
    l2, = ax1r.plot(idx, target, color="#1976D2", lw=1.8,
                    label="target_1h [t]")
    ax1.set_ylabel("status_6", color="#FF7043")
    ax1r.set_ylabel("target_1h", color="#1976D2")
    ax1.tick_params(axis="y", labelcolor="#FF7043")
    ax1r.tick_params(axis="y", labelcolor="#1976D2")
    ax1.legend(handles=[l1, l2], loc="upper left", fontsize=8)
    ax1.set_title(
        f"status_6[t−{best_lag}] vs target_1h[t]  │  "
        f"r(best_lag)={best_r:+.3f}   r(lag=0)={r0:+.3f}",
        fontsize=9
    )
    ax1.grid(alpha=0.25)

    # ── 2. Профиль корреляций ──
    ax2 = fig.add_subplot(gs[1, 0])
    lags_x = np.arange(len(profile))
    ax2.plot(lags_x, profile, color="#546E7A", lw=1.2)
    ax2.axvline(best_lag, color="#E53935", lw=1.8, linestyle="--",
                label=f"best lag={best_lag}")
    ax2.axhline(0, color="black", lw=0.5)
    ax2.fill_between(lags_x, profile, 0,
                     where=profile > 0, alpha=0.15, color="#1976D2")
    ax2.fill_between(lags_x, profile, 0,
                     where=profile < 0, alpha=0.15, color="#E53935")
    # Подписи осей X в часах
    tick_lags = [0, 12, 24, 48, 72, 96]
    ax2.set_xticks([t for t in tick_lags if t < len(profile)])
    ax2.set_xticklabels([f"{t//2}ч" for t in tick_lags if t < len(profile)])
    ax2.set_xlabel("lag")
    ax2.set_ylabel("Spearman r")
    ax2.set_title(f"Профиль лагов (окно: {window_name})", fontsize=9)
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.25)

    # ── 3. Scatter ──
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.scatter(s5_shift, target, alpha=0.45, s=20,
                color="#FF7043", edgecolors="none")
    if s5_shift.std() > 0:
        m, b = np.polyfit(s5_shift, target, 1)
        xline = np.linspace(s5_shift.min(), s5_shift.max(), 100)
        ax3.plot(xline, m*xline+b, "k--", lw=1.2)
    ax3.set_xlabel(f"status_6 [t−{best_lag}]")
    ax3.set_ylabel("target_1h [t]")
    ax3.set_title(f"Scatter  │  r = {best_r:+.3f}", fontsize=9)
    ax3.grid(alpha=0.25)

    # ── 4. Все три окна — сравнение профилей ──
    ax4 = fig.add_subplot(gs[2, :])
    palette = {"1 день (48)": "#E53935", "3 дня (144)": "#FF9800", "1 неделя (336)": "#1976D2"}
    for wname, wres in all_results.items():
        p = np.array(wres[route_id]["lag_profile"], dtype=float)
        bl = wres[route_id]["best_lag"]
        if len(p) == 0:
            continue
        lx = np.arange(len(p))
        ax4.plot(lx, p, color=palette[wname], lw=1.2, alpha=0.85,
                 label=f"{wname}  best={bl} ({bl//2}ч)  r={wres[route_id]['best_r']:+.3f}")
        ax4.axvline(bl, color=palette[wname], lw=0.8, linestyle=":")
    ax4.axhline(0, color="black", lw=0.5)
    ax4.set_xticks([t for t in [0,12,24,48,72,96] if t < len(profile)])
    ax4.set_xticklabels([f"{t//2}ч" for t in [0,12,24,48,72,96] if t < len(profile)])
    ax4.set_xlabel("lag")
    ax4.set_ylabel("Spearman r")
    ax4.set_title("Сравнение профилей лагов по всем окнам поиска", fontsize=9)
    ax4.legend(fontsize=8, loc="upper right")
    ax4.grid(alpha=0.25)

    # Вердикт
    gain = abs(best_r) - abs(r0)
    if abs(best_r) > 0.4 and gain > 0.05:
        verdict = f"✅ Чёткий лаг {best_lag} ({best_lag*30//60}ч)  |  gain vs lag=0: +{gain:.3f}"
        col = "#1B5E20"
    elif abs(best_r) > 0.25:
        verdict = f"🔶 Умеренная связь r={best_r:+.3f} при lag={best_lag}  |  gain={gain:+.3f}"
        col = "#E65100"
    else:
        verdict = f"❌ Слабая связь r={best_r:+.3f} — status_6 слабо предсказывает target на этом окне"
        col = "#B71C1C"

    fig.text(0.5, -0.015, verdict, ha="center", fontsize=10,
             fontweight="bold", color=col)
    plt.tight_layout()
    plt.show()


# ── Виджет ──
out3 = widgets.Output()

route_input3 = widgets.BoundedIntText(
    value=all_routes[0], min=min(all_routes), max=max(all_routes), step=1,
    description="route_id:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)
window_select = widgets.Dropdown(
    options=list(WINDOW_OPTIONS.keys()),
    value="3 дня (144)",
    description="Окно:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="200px"),
)
btn3 = widgets.Button(
    description="▶ Показать",
    button_style="success",
    layout=widgets.Layout(width="140px"),
)

def on_click3(_):
    with out3:
        clear_output(wait=True)
        plot_best_lag_window(route_input3.value, window_select.value)

btn3.on_click(on_click3)
display(widgets.HBox([route_input3, window_select, btn3]))
display(out3)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output

FREQ      = "30min"
SHOW_LAST = 48

STATUS_COLS  = [f"status_{i}" for i in range(1, 7)]
STATUS_COLORS = {
    "status_1": "#E53935",
    "status_2": "#FF9800",
    "status_3": "#FDD835",
    "status_4": "#43A047",
    "status_5": "#1E88E5",
    "status_6": "#8E24AA",
}
STATUS_LABELS = {
    "status_1": "s1 (тек. склад)",
    "status_2": "s2 (тек. склад)",
    "status_3": "s3 (тек. склад)",
    "status_4": "s4 (пред. склад)",
    "status_5": "s5 (пред. склад)",
    "status_6": "s6 (пред. склад)",
}

def minmax(arr):
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

def plot_all_statuses(route_id):
    cols = ["target_1h"] + STATUS_COLS
    grp  = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[cols]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )

    if len(grp) < SHOW_LAST:
        print(f"⚠️ Недостаточно данных (n={len(grp)})")
        return

    window = grp.iloc[-SHOW_LAST:]
    idx    = window.index
    target = window["target_1h"].values

    # ── Фигура ──
    fig = plt.figure(figsize=(15, 11))
    fig.suptitle(
        f"Route {route_id}  │  Последние {SHOW_LAST} точек ({SHOW_LAST//2}ч)  │  target_1h + все статусы",
        fontsize=12, fontweight="bold"
    )
    gs = gridspec.GridSpec(
        4, 2, figure=fig,
        height_ratios=[2.5, 1, 1, 1],
        hspace=0.55, wspace=0.35
    )

    # ══════════════════════════════════════════════
    # ПАНЕЛЬ 1 (верхняя, вся ширина):
    # Таргет (левая ось) + все 6 статусов нормированных (правая ось)
    # ══════════════════════════════════════════════
    ax_main  = fig.add_subplot(gs[0, :])
    ax_right = ax_main.twinx()

    # Статусы — нормированы 0..1, тонкие, полупрозрачные
    for col in STATUS_COLS:
        vals = minmax(window[col].values)
        ax_right.plot(idx, vals,
                      color=STATUS_COLORS[col], lw=1.1,
                      alpha=0.55, linestyle="--",
                      label=STATUS_LABELS[col])
        # Подсвечиваем пики (> 0.75 от нормированного)
        spike_mask = vals > 0.75
        if spike_mask.any():
            ax_right.scatter(idx[spike_mask], vals[spike_mask],
                             color=STATUS_COLORS[col], s=22, zorder=4, alpha=0.8)

    # Таргет — толстый, поверх всего
    ax_main.plot(idx, target,
                 color="#0D47A1", lw=2.8, zorder=5,
                 label="target_1h")
    ax_main.fill_between(idx, target, alpha=0.08, color="#0D47A1")

    ax_main.set_ylabel("target_1h", color="#0D47A1", fontsize=10)
    ax_right.set_ylabel("статусы (норм. 0–1)", fontsize=9, color="#555")
    ax_main.tick_params(axis="y", labelcolor="#0D47A1")
    ax_right.set_ylim(-0.05, 1.5)   # чуть вниз чтобы не сливались

    # Легенда: объединяем оба
    h1, l1 = ax_main.get_legend_handles_labels()
    h2, l2 = ax_right.get_legend_handles_labels()
    ax_main.legend(h1+h2, l1+l2,
                   loc="upper left", fontsize=7.5,
                   ncol=4, framealpha=0.7)
    ax_main.set_title("target_1h (ось L, абс.) + статусы (ось R, норм. 0–1)  │  пунктир = статусы",
                      fontsize=9)
    ax_main.grid(alpha=0.2)

    # ══════════════════════════════════════════════
    # ПАНЕЛИ 2–7: каждый статус отдельно vs таргет
    # ══════════════════════════════════════════════
    positions = [(1,0),(1,1),(2,0),(2,1),(3,0),(3,1)]

    for i, col in enumerate(STATUS_COLS):
        r, c  = positions[i]
        ax    = fig.add_subplot(gs[r, c])
        axr   = ax.twinx()

        raw   = window[col].values
        norm  = minmax(raw)

        # Статус (нормированный)
        axr.fill_between(idx, norm,
                         alpha=0.18, color=STATUS_COLORS[col])
        axr.plot(idx, norm,
                 color=STATUS_COLORS[col], lw=1.2, alpha=0.8,
                 label=col)

        # Таргет (нормированный — для сравнения формы)
        tgt_norm = minmax(target)
        ax.plot(idx, tgt_norm,
                color="#0D47A1", lw=1.6, alpha=0.9,
                label="target (норм.)")

        # Spearman r
        from scipy.stats import spearmanr
        r_val, _ = spearmanr(raw, target)

        ax.set_ylim(-0.05, 1.15)
        axr.set_ylim(-0.05, 1.15)
        ax.set_yticks([])
        axr.set_yticks([])
        ax.set_title(
            f"{STATUS_LABELS[col]}  │  r={r_val:+.3f}",
            fontsize=8, color=STATUS_COLORS[col], fontweight="bold"
        )

        # Подсветка выбросов статуса
        spike_mask = norm > 0.75
        if spike_mask.any():
            axr.vlines(idx[spike_mask], 0, norm[spike_mask],
                       color=STATUS_COLORS[col], alpha=0.3, lw=0.8)

        ax.grid(alpha=0.2)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

    # ── Текстовая сводка ──
    from scipy.stats import spearmanr
    print(f"\n{'─'*48}")
    print(f"  route_id: {route_id}  |  последние {SHOW_LAST} точек")
    print(f"  {'Статус':<12} {'mean':>9} {'max':>9} {'zero%':>7} {'r(target)':>10}")
    for col in STATUS_COLS:
        s    = window[col]
        r, _ = spearmanr(s.values, target)
        print(f"  {col:<12} {s.mean():>9.0f} {s.max():>9.0f} "
              f"{(s==0).mean()*100:>6.1f}% {r:>+10.3f}")
    print(f"{'─'*48}")


# ── Виджет ──
out_all = widgets.Output()

route_input_all = widgets.BoundedIntText(
    value=all_routes[0],
    min=min(all_routes), max=max(all_routes), step=1,
    description="route_id:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)
btn_all = widgets.Button(
    description="▶ Показать",
    button_style="primary",
    layout=widgets.Layout(width="140px"),
)

def on_click_all(_):
    with out_all:
        clear_output(wait=True)
        plot_all_statuses(route_input_all.value)

btn_all.on_click(on_click_all)
display(widgets.HBox([route_input_all, btn_all]))
display(out_all)

In [ ]:
[485, 192, 702, 999, 446, 782, 725, 768, 445, 817, 329, 594, 123, 948, 263]

In [ ]:
# ── Ячейка 1: многокритериальное ранжирование статусов ──
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

FREQ      = "30min"
SHOW_LAST = 48          # последние 48 точек НЕ используем — честный hold-out
LAG_LIST  = [0,1,2,3,4,6,8,12,24,48]   # лаги для поиска
STATUS_COLS = [f"status_{i}" for i in range(1, 7)]

print("Считаем метрики по маршрутам...")

# Накапливаем по каждому маршруту
route_metrics = {col: {
    "spearman_lag0":  [],
    "spearman_best":  [],
    "best_lag":       [],
    "pearson_lag0":   [],
    "mi_lag0":        [],   # mutual information
    "partial_r":      [],   # partial corr: убираем влияние остальных статусов
} for col in STATUS_COLS}

for route_id, grp in train_df.groupby("route_id"):
    grp = (
        grp.sort_values("timestamp")
        .set_index("timestamp")[["target_1h"] + STATUS_COLS]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )
    # Обучающая часть — без последних 48 точек
    train_part = grp.iloc[:-SHOW_LAST]
    if len(train_part) < 100:
        continue

    tgt = train_part["target_1h"].values

    for col in STATUS_COLS:
        s5 = train_part[col].values

        # Spearman lag=0
        r0, _ = spearmanr(s5, tgt)
        route_metrics[col]["spearman_lag0"].append(r0 if not np.isnan(r0) else 0)

        # Pearson lag=0
        rp, _ = pearsonr(s5, tgt)
        route_metrics[col]["pearson_lag0"].append(rp if not np.isnan(rp) else 0)

        # Mutual information lag=0
        try:
            mi = mutual_info_regression(s5.reshape(-1,1), tgt, random_state=42)[0]
        except:
            mi = 0
        route_metrics[col]["mi_lag0"].append(mi)

        # Лучший лаг (spearman)
        best_r, best_lag = 0, 0
        for lag in LAG_LIST:
            xi = s5[:-lag] if lag > 0 else s5
            yi = tgt[lag:]  if lag > 0 else tgt
            if len(xi) < 50:
                continue
            r, _ = spearmanr(xi, yi)
            if not np.isnan(r) and abs(r) > abs(best_r):
                best_r, best_lag = r, lag
        route_metrics[col]["spearman_best"].append(best_r)
        route_metrics[col]["best_lag"].append(best_lag)

        # Partial correlation: убираем линейное влияние остальных статусов
        other_cols = [c for c in STATUS_COLS if c != col]
        X_other = train_part[other_cols].values
        scaler  = StandardScaler()
        X_other_s = scaler.fit_transform(X_other)
        s5_s    = StandardScaler().fit_transform(s5.reshape(-1,1)).ravel()
        tgt_s   = StandardScaler().fit_transform(tgt.reshape(-1,1)).ravel()
        # Регрессия s5 ~ others
        try:
            beta_s5  = np.linalg.lstsq(X_other_s, s5_s,  rcond=None)[0]
            beta_tgt = np.linalg.lstsq(X_other_s, tgt_s, rcond=None)[0]
            resid_s5  = s5_s  - X_other_s @ beta_s5
            resid_tgt = tgt_s - X_other_s @ beta_tgt
            r_part, _ = spearmanr(resid_s5, resid_tgt)
        except:
            r_part = 0
        route_metrics[col]["partial_r"].append(r_part if not np.isnan(r_part) else 0)

print("✅ Метрики собраны")

In [ ]:
# ── Ячейка 2: LightGBM feature importance ──
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

print("Считаем LightGBM importance...")

# Собираем датасет: таргет + статусы + лаговые фичи
records = []
for route_id, grp in train_df.groupby("route_id"):
    grp = (
        grp.sort_values("timestamp")
        .set_index("timestamp")[["target_1h"] + STATUS_COLS]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
        .iloc[:-SHOW_LAST]   # hold-out
    )
    if len(grp) < 80:
        continue
    df_r = grp.copy()
    for col in STATUS_COLS:
        for lag in [0, 1, 2, 4, 6, 12, 24, 48]:
            df_r[f"{col}_lag{lag}"] = df_r[col].shift(lag)
    df_r = df_r.dropna()
    records.append(df_r)

full_df = pd.concat(records)
feat_cols = [c for c in full_df.columns if c != "target_1h"]
X = full_df[feat_cols].values
y = full_df["target_1h"].values

model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                           num_leaves=31, random_state=42, n_jobs=-1,
                           verbosity=-1)
model.fit(X, y)

fi_df = pd.DataFrame({
    "feature":    feat_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

# Агрегируем importance по статусу (сумма всех лагов)
fi_df["status"] = fi_df["feature"].str.extract(r"(status_\d)")
status_fi = fi_df.groupby("status")["importance"].sum().sort_values(ascending=False)
status_fi_norm = status_fi / status_fi.sum()

print("LightGBM importance (нормированная):")
for s, v in status_fi_norm.items():
    print(f"  {s}: {v:.4f}")

In [ ]:
# ── Ячейка 3: сводная таблица + скоринг ──
print("\n" + "="*60)
print("ИТОГОВЫЙ РЕЙТИНГ СТАТУСОВ")
print("="*60)

summary = {}
for col in STATUS_COLS:
    m = route_metrics[col]
    summary[col] = {
        "spearman_lag0_med":  np.nanmedian(m["spearman_lag0"]),
        "spearman_best_med":  np.nanmedian(m["spearman_best"]),
        "best_lag_med":       np.nanmedian(m["best_lag"]),
        "pearson_lag0_med":   np.nanmedian(m["pearson_lag0"]),
        "mi_lag0_med":        np.nanmedian(m["mi_lag0"]),
        "partial_r_med":      np.nanmedian(m["partial_r"]),
        "lgbm_importance":    float(status_fi_norm.get(col, 0)),
        # доля маршрутов с |r_best| > 0.2
        "pct_routes_r02":     np.mean(np.abs(m["spearman_best"]) > 0.2),
        # доля маршрутов с частичной корр > 0.1
        "pct_routes_partial": np.mean(np.abs(m["partial_r"]) > 0.1),
    }

sum_df = pd.DataFrame(summary).T

# Нормируем каждую метрику 0..1 и считаем composite score
metrics_for_score = [
    "spearman_best_med",   # вес 3 — самая важная
    "partial_r_med",       # вес 2 — убирает мультиколлинеарность
    "lgbm_importance",     # вес 2 — реальная полезность в модели
    "mi_lag0_med",         # вес 1 — нелинейная связь
    "pct_routes_r02",      # вес 1 — стабильность по маршрутам
]
weights = [3, 2, 2, 1, 1]

score_df = sum_df[metrics_for_score].copy().abs()
for col in score_df.columns:
    rng = score_df[col].max() - score_df[col].min()
    score_df[col] = (score_df[col] - score_df[col].min()) / (rng + 1e-9)

sum_df["composite_score"] = (score_df * weights).sum(axis=1) / sum(weights)
sum_df = sum_df.sort_values("composite_score", ascending=False)

print(sum_df[[
    "spearman_best_med", "best_lag_med", "partial_r_med",
    "lgbm_importance", "pct_routes_r02", "composite_score"
]].round(4).to_string())

# Рекомендация: порог по composite score
threshold = 0.3
keep  = sum_df[sum_df["composite_score"] >= threshold].index.tolist()
drop  = sum_df[sum_df["composite_score"]  < threshold].index.tolist()

print(f"\n  Порог composite score ≥ {threshold}")
print(f"  ✅ ОСТАВИТЬ : {keep}")
print(f"  ❌ УБРАТЬ   : {drop}")

In [ ]:
# ── Ячейка 4: визуализация ──
# from load_skill import load_skill  # если нужен chart skill, иначе убери
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

STATUS_COLORS = {
    "status_1": "#E53935", "status_2": "#FF9800", "status_3": "#FDD835",
    "status_4": "#43A047", "status_5": "#1E88E5", "status_6": "#8E24AA",
}
cols_ordered = sum_df.index.tolist()
colors_ord   = [STATUS_COLORS[c] for c in cols_ordered]

fig = plt.figure(figsize=(16, 13))
fig.suptitle("Ранжирование статусов: насколько помогают предсказывать target_1h",
             fontsize=13, fontweight="bold")
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.4)

def hbar(ax, values, labels, colors, title, xlabel, threshold=None):
    bars = ax.barh(labels[::-1], values[::-1], color=colors[::-1], edgecolor="white")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=8)
    if threshold is not None:
        ax.axvline(threshold, color="red", lw=1.2, linestyle="--", label=f"порог={threshold}")
        ax.legend(fontsize=7)
    for bar, v in zip(bars, values[::-1]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8)
    ax.grid(axis="x", alpha=0.25)

# 1. Composite score
ax = fig.add_subplot(gs[0, :2])
hbar(ax, sum_df["composite_score"].values, cols_ordered, colors_ord,
     "🏆 Composite Score (взвешенный, ≥0.3 = оставить)",
     "score", threshold=threshold)
# Подсвечиваем DROP
for patch, col in zip(ax.patches[::-1], cols_ordered):
    if col in drop:
        patch.set_alpha(0.35)
        patch.set_hatch("//")

# 2. Spearman best lag
ax2 = fig.add_subplot(gs[0, 2])
hbar(ax2, sum_df["spearman_best_med"].abs().values, cols_ordered, colors_ord,
     "Spearman |r| (лучший лаг)", "|r|")

# 3. Partial correlation
ax3 = fig.add_subplot(gs[1, 0])
hbar(ax3, sum_df["partial_r_med"].abs().values, cols_ordered, colors_ord,
     "Partial corr |r|\n(очищен от других статусов)", "|r|")

# 4. LightGBM importance
ax4 = fig.add_subplot(gs[1, 1])
hbar(ax4, sum_df["lgbm_importance"].values, cols_ordered, colors_ord,
     "LightGBM importance\n(сумма по всем лагам, норм.)", "importance")

# 5. Mutual information
ax5 = fig.add_subplot(gs[1, 2])
hbar(ax5, sum_df["mi_lag0_med"].values, cols_ordered, colors_ord,
     "Mutual Information\n(нелинейная связь)", "MI")

# 6. Лучший медианный лаг
ax6 = fig.add_subplot(gs[2, 0])
lags_h = sum_df["best_lag_med"].values * 0.5   # в часах
hbar(ax6, lags_h, cols_ordered, colors_ord,
     "Медиана лучшего лага (часы)", "ч")

# 7. % маршрутов с |r|>0.2
ax7 = fig.add_subplot(gs[2, 1])
hbar(ax7, sum_df["pct_routes_r02"].values, cols_ordered, colors_ord,
     "Доля маршрутов\nс |r_best| > 0.2", "доля")

# 8. Вердикт — текстовая таблица
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis("off")
table_data = []
for col in cols_ordered:
    verdict = "✅ KEEP" if col in keep else "❌ DROP"
    score   = sum_df.loc[col, "composite_score"]
    table_data.append([col, f"{score:.3f}", verdict])

tbl = ax8.table(
    cellText=table_data,
    colLabels=["Статус", "Score", "Вердикт"],
    loc="center", cellLoc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.8)
for (row, col_idx), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor("#37474F")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        status_name = table_data[row-1][0]
        verdict_val = table_data[row-1][2]
        if "DROP" in verdict_val:
            cell.set_facecolor("#FFEBEE")
        else:
            cell.set_facecolor("#E8F5E9")
ax8.set_title("Итоговый вердикт", fontsize=9, fontweight="bold")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("diagnostics/status_ranking.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ diagnostics/status_ranking.png")

In [ ]:
# ── Ячейка 4: визуализация ──
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- WB-like palette ---
BG_DARK   = "#140221"   # общий фон
BG_PANEL  = "#1D0A35"   # фон подграфиков
GRID_CLR  = "#4E3B78"   # мягкая сетка
TEXT_CLR  = "#FFFFFF"   # основной текст
SUB_CLR   = "#D8C9FF"   # подписи осей / вторичный текст
TITLE_CLR = "#F3ECFF"   # заголовки
THRESHOLD = "#FF78D6"   # линия порога
EDGE_CLR  = "#2A1248"   # контур баров

plt.rcParams.update({
    "text.color":        TEXT_CLR,
    "axes.labelcolor":   SUB_CLR,
    "xtick.color":       SUB_CLR,
    "ytick.color":       TEXT_CLR,
    "axes.edgecolor":    BG_PANEL,
    "axes.facecolor":    BG_PANEL,
    "figure.facecolor":  BG_DARK,
    "axes.titlecolor":   TITLE_CLR,
    "font.family":       "DejaVu Sans",
})

# Цвета статусов — одна WB-семья оттенков
STATUS_COLORS = {
    "status_1": "#E85AD8",
    "status_2": "#D94FD4",
    "status_3": "#C44BCF",
    "status_4": "#AD55D8",
    "status_5": "#8F63FF",
    "status_6": "#6F6CF6",
}

cols_ordered = sum_df.index.tolist()
colors_ord   = [STATUS_COLORS.get(c, "#C44BCF") for c in cols_ordered]

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor(BG_DARK)

fig.suptitle(
    "Ранжирование статусов: насколько помогают предсказывать target_1h",
    fontsize=16, fontweight="bold", color=TEXT_CLR, y=0.985
)

gs = gridspec.GridSpec(
    3, 2,
    figure=fig,
    hspace=0.58,
    wspace=0.28,
    top=0.93,
    bottom=0.06,
    left=0.08,
    right=0.96
)

def styled_hbar(ax, values, labels, colors, title, xlabel, threshold=None):
    rev_labels = labels[::-1]
    rev_values = values[::-1]
    rev_colors = colors[::-1]

    bars = ax.barh(
        rev_labels,
        rev_values,
        color=rev_colors,
        edgecolor=EDGE_CLR,
        linewidth=0.8,
        height=0.56
    )

    ax.set_title(title, fontsize=10, fontweight="bold", color=TITLE_CLR, loc="left", pad=10)
    ax.set_xlabel(xlabel, fontsize=9, color=SUB_CLR)
    ax.tick_params(axis="x", labelsize=8.5)
    ax.tick_params(axis="y", labelsize=9, pad=4)

    ax.grid(axis="x", color=GRID_CLR, alpha=0.35, linewidth=0.8, linestyle="--")
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    if threshold is not None:
        ax.axvline(threshold, color=THRESHOLD, lw=1.6, linestyle="--")
        ax.text(
            threshold, 1.02, f"порог = {threshold:.1f}",
            transform=ax.get_xaxis_transform(),
            ha="left", va="bottom",
            fontsize=8, color=THRESHOLD
        )

    max_v = max(values) if max(values) > 0 else 1
    ax.set_xlim(0, max_v * 1.22)

    for bar, v in zip(bars, rev_values):
        ax.text(
            bar.get_width() + max_v * 0.015,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.3f}",
            va="center", ha="left",
            fontsize=8.5, color=TEXT_CLR
        )

    return bars


# 1. Composite score — широкий верхний график
ax1 = fig.add_subplot(gs[0, :])
bars1 = styled_hbar(
    ax1,
    sum_df["composite_score"].values,
    cols_ordered,
    colors_ord,
    "Composite Score (взвешенный, >= 0.3 — оставить)",
    "score",
    threshold=threshold
)

# Подсветка DROP
for patch, col in zip(bars1, cols_ordered[::-1]):
    if col in drop:
        patch.set_alpha(0.28)
        patch.set_hatch("//")
        patch.set_edgecolor("#BCA7E8")


# 2. Spearman best lag
ax2 = fig.add_subplot(gs[1, 0])
styled_hbar(
    ax2,
    sum_df["spearman_best_med"].abs().values,
    cols_ordered,
    colors_ord,
    "Spearman |r| (лучший лаг)",
    "|r|"
)


# 3. Partial correlation
ax3 = fig.add_subplot(gs[1, 1])
styled_hbar(
    ax3,
    sum_df["partial_r_med"].abs().values,
    cols_ordered,
    colors_ord,
    "Partial corr |r| (очищен от др. статусов)",
    "|r|"
)


# 4. LightGBM importance
ax4 = fig.add_subplot(gs[2, 0])
styled_hbar(
    ax4,
    sum_df["lgbm_importance"].values,
    cols_ordered,
    colors_ord,
    "LightGBM importance (сумма по лагам, норм.)",
    "importance"
)


# 5. Mutual information
ax5 = fig.add_subplot(gs[2, 1])
styled_hbar(
    ax5,
    sum_df["mi_lag0_med"].values,
    cols_ordered,
    colors_ord,
    "Mutual Information (нелинейная связь)",
    "MI"
)

fig.text(
    0.5, 0.018,
    "Заштрихованные и приглушённые бары — статусы ниже порога",
    ha="center",
    fontsize=9,
    color=SUB_CLR,
    style="italic"
)

plt.savefig("diagnostics/status_ranking.png", dpi=160, bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print("✅ diagnostics/status_ranking.png")

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# ── Загрузка ──
raw_cv_df = pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv", parse_dates=["timestamp"])
train_df  = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df  = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

# ── Per-route CV метрики ──
route_err = (
    raw_cv_df.groupby("route_id")
    .apply(lambda df: pd.Series({
        "mae":          df["ae"].mean(),
        "wape":         np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
        "std_err":      df["err"].std(),
        "mean_err":     df["err"].mean(),        # систематическое смещение
        "calib_scale":  df["y_true"].sum() / (df["y_pred"].sum() + 1e-9),
        "mean_true":    df["y_true"].mean(),
        # Точность на пиках: топ-10% таргетов
        "peak_wape":    (lambda peak_mask:
            np.abs(df.loc[peak_mask,"err"]).sum() /
            (df.loc[peak_mask,"y_true"].sum() + 1e-9)
        )(df["y_true"] >= df["y_true"].quantile(0.9)),
        # Точность в обычные моменты: нижние 50%
        "base_wape":    (lambda base_mask:
            np.abs(df.loc[base_mask,"err"]).sum() /
            (df.loc[base_mask,"y_true"].sum() + 1e-9)
        )(df["y_true"] < df["y_true"].quantile(0.5)),
    }))
    .reset_index()
)

# ── Характеристики рядов из train ──
route_features = []
for route_id, grp in train_df.groupby("route_id"):
    y = grp["target_1h"].values.astype(float)
    if len(y) < 100:
        continue
    mean_y = y.mean()
    std_y  = y.std()

    try:    acf1 = acf(y, nlags=1, fft=True)[1]
    except: acf1 = 0.0

    t          = np.arange(len(y))
    slope      = np.polyfit(t, y, 1)[0]
    trend_norm = slope * len(y) / (mean_y + 1e-9)

    s        = pd.Series(y)
    roll_cv  = (s.rolling(48).std() / (s.rolling(48).mean() + 1e-9)).mean()
    peak_ratio = np.percentile(y, 99) / (np.percentile(y, 50) + 1e-9)  # p99/p50

    route_features.append({
        "route_id":   route_id,
        "mean_y":     mean_y,
        "cv":         std_y / (mean_y + 1e-9),
        "zero_frac":  (y == 0).mean(),
        "peak_ratio": peak_ratio,
        "acf1":       acf1,
        "trend_norm": trend_norm,
        "roll_cv":    roll_cv,
    })

feat_df  = pd.DataFrame(route_features)
analysis = route_err.merge(feat_df, on="route_id", how="inner")

# Квартили по WAPE
analysis["wape_q"] = pd.qcut(
    analysis["wape"], q=4, labels=["Q1_easy","Q2","Q3","Q4_hard"]
)

# ============================================================
# 1. Главная таблица: Q1 vs Q4 — что их разделяет
# ============================================================
cols = ["cv","peak_ratio","acf1","trend_norm","roll_cv",
        "zero_frac","mean_y","mean_err","calib_scale",
        "peak_wape","base_wape"]
q_stats = analysis.groupby("wape_q")[cols].mean().round(3)
print("── Средние по квартилям WAPE ──")
print(q_stats.to_string())

# Ключевой вопрос: откуда ошибки — с пиков или с базы?
print("\n── Peak vs Base WAPE (Q1 easy vs Q4 hard) ──")
for q in ["Q1_easy","Q4_hard"]:
    row = q_stats.loc[q]
    print(f"  {q}:  peak_wape={row['peak_wape']:.3f}  "
          f"base_wape={row['base_wape']:.3f}  "
          f"ratio={row['peak_wape']/row['base_wape']:.2f}x  "
          f"peak_ratio={row['peak_ratio']:.1f}")

# ============================================================
# 2. Корреляция характеристик ряда с WAPE модели
# ============================================================
print("\n── Корреляция фич ряда с WAPE (Spearman) ──")
corr_cols = ["cv","peak_ratio","acf1","trend_norm","roll_cv","zero_frac","mean_y"]
for col in corr_cols:
    r = analysis[["wape", col]].dropna().corr(method="spearman").iloc[0,1]
    bar = "█" * int(abs(r) * 30)
    sign = "+" if r > 0 else "-"
    print(f"  {col:<15} {sign}{abs(r):.3f}  {bar}")

# ============================================================
# 3. Типизация маршрутов по паттерну ошибки
# ============================================================
# Тип A: peak_wape >> base_wape  → модель ошибается именно на пиках
# Тип B: peak_wape ≈ base_wape   → модель ошибается равномерно
# Тип C: calib_scale >> 1        → систематическое занижение
# Тип D: calib_scale << 1        → систематическое завышение

analysis["peak_base_ratio"] = analysis["peak_wape"] / (analysis["base_wape"] + 1e-9)
analysis["error_type"] = "B_uniform"
analysis.loc[analysis["peak_base_ratio"] > 1.5, "error_type"] = "A_peak_driven"
analysis.loc[analysis["calib_scale"] > 1.15,    "error_type"] = "C_underpredict"
analysis.loc[analysis["calib_scale"] < 0.85,    "error_type"] = "D_overpredict"

print("\n── Типы ошибок по маршрутам ──")
print(analysis["error_type"].value_counts())

print("\n── WAPE по типам ошибок ──")
print(analysis.groupby("error_type")["wape"].describe().round(3))

analysis.to_csv("route_error_analysis.csv", index=False)
print("\n✅ route_error_analysis.csv")

# ============================================================
# 4. Чарты
# ============================================================
import plotly.io as pio

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "WAPE vs peak_ratio (Q1/Q4)",
        "WAPE vs acf1",
        "WAPE vs roll_cv",
        "peak_wape vs base_wape (тип ошибки)",
    ]
)
colors = {"Q1_easy":"#22c55e","Q2":"#84cc16","Q3":"#f59e0b","Q4_hard":"#ef4444"}

for q, grp in analysis.groupby("wape_q"):
    c = colors[str(q)]
    fig.add_trace(go.Scatter(
        x=grp["peak_ratio"], y=grp["wape"], mode="markers",
        marker=dict(color=c, size=5, opacity=0.6),
        name=str(q), legendgroup=str(q), showlegend=True,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=grp["acf1"], y=grp["wape"], mode="markers",
        marker=dict(color=c, size=5, opacity=0.6),
        name=str(q), legendgroup=str(q), showlegend=False,
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=grp["roll_cv"], y=grp["wape"], mode="markers",
        marker=dict(color=c, size=5, opacity=0.6),
        name=str(q), legendgroup=str(q), showlegend=False,
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=grp["base_wape"], y=grp["peak_wape"], mode="markers",
        marker=dict(color=c, size=5, opacity=0.6),
        name=str(q), legendgroup=str(q), showlegend=False,
    ), row=2, col=2)

# diagonal reference line (peak_wape = base_wape)
fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode="lines",
    line=dict(color="gray", dash="dash", width=1),
    showlegend=False,
), row=2, col=2)

fig.update_xaxes(title_text="peak_ratio",  row=1, col=1)
fig.update_xaxes(title_text="acf lag1",    row=1, col=2)
fig.update_xaxes(title_text="rolling cv",  row=2, col=1)
fig.update_xaxes(title_text="base wape",   row=2, col=2)
fig.update_yaxes(title_text="wape",        row=1, col=1)
fig.update_yaxes(title_text="wape",        row=1, col=2)
fig.update_yaxes(title_text="wape",        row=2, col=1)
fig.update_yaxes(title_text="peak wape",   row=2, col=2)

fig.update_layout(
    title=dict(text="Route error anatomy — what drives hard routes"),
    height=800,
)
fig.write_image("route_error_anatomy.png")
print("✅ route_error_anatomy.png")

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

raw_cv_df  = pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv", parse_dates=["timestamp"])
analysis   = pd.read_csv("route_error_analysis.csv")
train_df   = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

# ============================================================
# 1. D_overpredict: смотрим на динамику конца ряда
#    Гипотеза: маршрут упал/прекратил работу незадолго до теста
# ============================================================
d_routes = analysis[analysis["error_type"]=="D_overpredict"]["route_id"].tolist()
print(f"D_overpredict routes: {len(d_routes)}")
print("="*65)

d_stats = []
for rid in d_routes:
    y = (train_df[train_df["route_id"]==rid]
         .sort_values("timestamp")["target_1h"].values.astype(float))
    if len(y) < 96:
        continue
    mean_all   = y.mean()
    mean_last48 = y[-48:].mean()   # последние сутки
    mean_last96 = y[-96:].mean()   # последние двое суток
    mean_first  = y[:len(y)//2].mean()
    recent_ratio = mean_last48 / (mean_all + 1e-9)  # < 1 = упал, > 1 = вырос
    cv_row = analysis[analysis["route_id"]==rid].iloc[0]
    d_stats.append({
        "route_id":       rid,
        "mean_all":       round(mean_all),
        "mean_last48":    round(mean_last48),
        "mean_last96":    round(mean_last96),
        "recent_ratio":   round(recent_ratio, 3),
        "calib_scale":    round(cv_row["calib_scale"], 3),
        "wape":           round(cv_row["wape"], 3),
        "zero_frac":      round(cv_row["zero_frac"], 3),
        "n_zeros_last48": int((y[-48:] == 0).sum()),
    })

df_d = pd.DataFrame(d_stats).sort_values("recent_ratio")
print("D_overpredict — динамика конца ряда (сортировка по recent_ratio):")
print(df_d.to_string(index=False))

print(f"\n  Маршрутов с recent_ratio < 0.5 (упали в 2+ раза): "
      f"{(df_d['recent_ratio'] < 0.5).sum()}")
print(f"  Маршрутов с recent_ratio < 0.1 (почти остановились): "
      f"{(df_d['recent_ratio'] < 0.1).sum()}")
print(f"  Маршрутов с n_zeros_last48 > 24 (>50% нулей в последние сутки): "
      f"{(df_d['n_zeros_last48'] > 24).sum()}")

# ============================================================
# 2. C_underpredict: смотрим на тренд (растут ли к концу)
#    Гипотеза: маршрут резко вырос ближе к концу обучения
# ============================================================
c_routes = analysis[analysis["error_type"]=="C_underpredict"]["route_id"].tolist()
print(f"\n\nC_underpredict routes: {len(c_routes)}")
print("="*65)

c_stats = []
for rid in c_routes:
    y = (train_df[train_df["route_id"]==rid]
         .sort_values("timestamp")["target_1h"].values.astype(float))
    if len(y) < 96:
        continue
    mean_all    = y.mean() + 1e-9
    mean_last48 = y[-48:].mean()
    mean_first  = y[:len(y)//2].mean() + 1e-9
    growth_recent   = mean_last48 / mean_all      # рост последних суток vs всей истории
    growth_overall  = y[-len(y)//4:].mean() / mean_first  # рост второй половины vs первой
    cv_row = analysis[analysis["route_id"]==rid].iloc[0]
    c_stats.append({
        "route_id":       rid,
        "mean_all":       round(mean_all),
        "mean_last48":    round(mean_last48),
        "growth_recent":  round(growth_recent, 3),
        "growth_overall": round(growth_overall, 3),
        "calib_scale":    round(cv_row["calib_scale"], 3),
        "wape":           round(cv_row["wape"], 3),
        "zero_frac":      round(cv_row["zero_frac"], 3),
    })

df_c = pd.DataFrame(c_stats).sort_values("growth_recent", ascending=False)
print("C_underpredict — динамика роста (сортировка по growth_recent):")
print(df_c.head(20).to_string(index=False))
print(f"\n  Маршрутов с growth_recent > 1.3 (выросли >30% к концу): "
      f"{(df_c['growth_recent'] > 1.3).sum()}")
print(f"  Маршрутов с growth_overall > 1.5 (тренд роста): "
      f"{(df_c['growth_overall'] > 1.5).sum()}")
print(f"  Медиана growth_recent: {df_c['growth_recent'].median():.3f}")

# ============================================================
# 3. Q4_hard: паттерн нулей и нерегулярность
#    Гипотеза: нули идут блоками (маршрут "спит" и "просыпается")
# ============================================================
q4_routes = analysis[analysis["wape_q"]=="Q4_hard"]["route_id"].tolist()
print(f"\n\nQ4_hard routes: {len(q4_routes)}")
print("="*65)

q4_stats = []
for rid in q4_routes:
    y = (train_df[train_df["route_id"]==rid]
         .sort_values("timestamp")["target_1h"].values.astype(float))
    if len(y) < 96:
        continue
    # Длина блоков нулей (прерывистость)
    is_zero = (y == 0).astype(int)
    # Считаем run-length encoding нулевых блоков
    zero_blocks = []
    current = 0
    for z in is_zero:
        if z == 1:
            current += 1
        elif current > 0:
            zero_blocks.append(current)
            current = 0
    if current > 0:
        zero_blocks.append(current)

    max_zero_block = max(zero_blocks) if zero_blocks else 0
    mean_zero_block = np.mean(zero_blocks) if zero_blocks else 0
    n_zero_blocks = len(zero_blocks)

    # Нестабильность уровня: std rolling mean
    roll_mean = pd.Series(y).rolling(48).mean().dropna()
    level_instability = roll_mean.std() / (roll_mean.mean() + 1e-9)

    cv_row = analysis[analysis["route_id"]==rid].iloc[0]
    q4_stats.append({
        "route_id":        rid,
        "zero_frac":       round(cv_row["zero_frac"], 3),
        "max_zero_block":  max_zero_block,
        "n_zero_blocks":   n_zero_blocks,
        "mean_zero_block": round(mean_zero_block, 1),
        "level_instab":    round(level_instability, 3),
        "cv":              round(cv_row["cv"], 3),
        "wape":            round(cv_row["wape"], 3),
    })

df_q4 = pd.DataFrame(q4_stats).sort_values("wape", ascending=False)
print("Q4_hard — топ-20 самых проблемных:")
print(df_q4.head(20).to_string(index=False))

print(f"\n  Маршрутов с max_zero_block > 48 (>24ч подряд нулей): "
      f"{(df_q4['max_zero_block'] > 48).sum()}")
print(f"  Маршрутов с n_zero_blocks > 10: "
      f"{(df_q4['n_zero_blocks'] > 10).sum()}")
print(f"  Медиана level_instab: {df_q4['level_instab'].median():.3f}")
print(f"  Q1_easy level_instab для сравнения:")

q1_routes = analysis[analysis["wape_q"]=="Q1_easy"]["route_id"].tolist()
q1_li = []
for rid in q1_routes[:50]:  # выборка
    y = train_df[train_df["route_id"]==rid].sort_values("timestamp")["target_1h"].values.astype(float)
    if len(y) < 96:
        continue
    roll_mean = pd.Series(y).rolling(48).mean().dropna()
    q1_li.append(roll_mean.std() / (roll_mean.mean() + 1e-9))
print(f"    медиана = {np.median(q1_li):.3f}")

# ============================================================
# 4. Итоговая классификация с рекомендацией стратегии
# ============================================================
print("\n\n" + "="*65)
print("СТРАТЕГИЯ ПО ТИПАМ МАРШРУТОВ")
print("="*65)

# D_overpredict: falling routes
falling_threshold = 0.3
falling_routes = set(
    df_d[df_d["recent_ratio"] < falling_threshold]["route_id"].tolist()
)

# C_underpredict: growing routes
growing_threshold = 1.2
growing_routes = set(
    df_c[df_c["growth_recent"] > growing_threshold]["route_id"].tolist()
)

# Q4_hard с блочными нулями (intermittent)
intermittent_routes = set(
    df_q4[df_q4["max_zero_block"] > 48]["route_id"].tolist()
)

print(f"\n  Falling routes (recent_ratio < {falling_threshold}):  {len(falling_routes)}")
print(f"  Growing routes (growth_recent > {growing_threshold}):  {len(growing_routes)}")
print(f"  Intermittent (max_zero_block > 48):   {len(intermittent_routes)}")

# Сохраняем для использования в постпроцессинге
segment_df = analysis[["route_id","wape","error_type","calib_scale",
                        "zero_frac","cv","roll_cv","mean_y"]].copy()
segment_df["is_falling"]      = segment_df["route_id"].isin(falling_routes)
segment_df["is_growing"]      = segment_df["route_id"].isin(growing_routes)
segment_df["is_intermittent"] = segment_df["route_id"].isin(intermittent_routes)
segment_df.to_csv("route_segments_detailed.csv", index=False)
print("\n✅ route_segments_detailed.csv")

In [ ]:
# ============================================================
# Правильная аналитика: вклад маршрута в агрегатный WAPE
# SAE = Σ|err| = вклад в числитель метрики
# Именно SAE надо уменьшать — это и есть "где деньги лежат"
# ============================================================
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf
import warnings
warnings.filterwarnings("ignore")

raw_cv_df = pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv", parse_dates=["timestamp"])
train_df  = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df  = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

# Global calibrated predictions (текущий лучший)
global_scale = raw_cv_df["y_true"].sum() / (raw_cv_df["y_pred"].sum() + 1e-9)
raw_cv_df["y_pred_cal"] = np.clip(raw_cv_df["y_pred"] * global_scale, 0, None)

total_sae      = np.abs(raw_cv_df["y_pred_cal"] - raw_cv_df["y_true"]).sum()
total_sum_true = raw_cv_df["y_true"].sum()
global_wape    = total_sae / total_sum_true
print(f"Global WAPE = {global_wape:.5f}  "
      f"(numerator={total_sae:.0f}, denominator={total_sum_true:.0f})")

# ============================================================
# 1. Per-route вклад в метрику
# ============================================================
route_stats = (
    raw_cv_df.groupby("route_id")
    .apply(lambda df: pd.Series({
        # Абсолютный вклад в числитель агрегатного WAPE
        "sae":           np.abs(df["y_pred_cal"] - df["y_true"]).sum(),
        "sum_true":      df["y_true"].sum(),
        "mae":           np.abs(df["y_pred_cal"] - df["y_true"]).mean(),
        "n_obs":         len(df),
        "mean_true":     df["y_true"].mean(),
        # Систематический сдвиг: + = завышаем, - = занижаем
        "mean_bias":     (df["y_pred_cal"] - df["y_true"]).mean(),
        "sum_bias":      (df["y_pred_cal"] - df["y_true"]).sum(),
        # MAE отдельно на верхних 20% и нижних 80%
        "mae_top20":     np.abs(df.loc[df["y_true"] >= df["y_true"].quantile(0.8),
                                        "y_pred_cal"] -
                                df.loc[df["y_true"] >= df["y_true"].quantile(0.8),
                                        "y_true"]).mean(),
        "mae_bot80":     np.abs(df.loc[df["y_true"] < df["y_true"].quantile(0.8),
                                        "y_pred_cal"] -
                                df.loc[df["y_true"] < df["y_true"].quantile(0.8),
                                        "y_true"]).mean(),
        "calib_scale":   df["y_true"].sum() / (df["y_pred"].sum() + 1e-9),
    }))
    .reset_index()
)

# % вклад в числитель агрегатного WAPE
route_stats["sae_pct"]  = route_stats["sae"]  / total_sae * 100
# % вклад в знаменатель (вес маршрута в метрике)
route_stats["denom_pct"] = route_stats["sum_true"] / total_sum_true * 100
# "Эффективность" ошибки: сколько % числителя приносит на 1% знаменателя
# > 1 = маршрут хуже среднего, < 1 = лучше
route_stats["error_efficiency"] = route_stats["sae_pct"] / (route_stats["denom_pct"] + 1e-9)

print(f"\nTop-20 маршрутов по АБСОЛЮТНОМУ ВКЛАДУ в числитель WAPE:")
print(f"{'route':>7} {'SAE':>14} {'sae%':>7} {'denom%':>8} "
      f"{'eff':>6} {'MAE':>12} {'mean_true':>12} {'bias':>12} {'scale':>7}")
print("─"*95)
for _, r in route_stats.nlargest(20, "sae").iterrows():
    print(f"{int(r['route_id']):>7} {r['sae']:>14.0f} {r['sae_pct']:>7.2f}% "
          f"{r['denom_pct']:>7.2f}% {r['error_efficiency']:>6.2f} "
          f"{r['mae']:>12.0f} {r['mean_true']:>12.0f} "
          f"{r['mean_bias']:>+12.0f} {r['calib_scale']:>7.3f}")

# Кумулятивный вклад
route_sorted = route_stats.sort_values("sae", ascending=False).reset_index(drop=True)
route_sorted["cum_sae_pct"] = route_sorted["sae"].cumsum() / total_sae * 100
n50 = (route_sorted["cum_sae_pct"] <= 50).sum()
n80 = (route_sorted["cum_sae_pct"] <= 80).sum()
print(f"\n50% числителя WAPE создают топ-{n50} маршрутов из 1000")
print(f"80% числителя WAPE создают топ-{n80} маршрутов из 1000")

# ============================================================
# 2. Характеристики рядов из train
# ============================================================
route_features = []
for route_id, grp in train_df.groupby("route_id"):
    y = grp["target_1h"].values.astype(float)
    if len(y) < 100:
        continue
    mean_y = y.mean()
    try:    acf1 = acf(y, nlags=1, fft=True)[1]
    except: acf1 = 0.0
    s = pd.Series(y)
    roll_cv = (s.rolling(48).std() / (s.rolling(48).mean() + 1e-9)).mean()
    # Тренд: рост конца ряда vs середины
    half = len(y) // 2
    growth_recent  = y[-48:].mean()  / (mean_y + 1e-9)
    growth_overall = y[half:].mean() / (y[:half].mean() + 1e-9)
    # Блочные нули
    is_zero = (y == 0).astype(int)
    zblocks = []
    cur = 0
    for z in is_zero:
        if z: cur += 1
        elif cur > 0: zblocks.append(cur); cur = 0
    if cur: zblocks.append(cur)

    route_features.append({
        "route_id":       route_id,
        "mean_y":         mean_y,
        "cv":             y.std() / (mean_y + 1e-9),
        "zero_frac":      (y==0).mean(),
        "acf1":           acf1,
        "roll_cv":        roll_cv,
        "growth_recent":  growth_recent,
        "growth_overall": growth_overall,
        "n_zero_blocks":  len(zblocks),
        "max_zero_block": max(zblocks) if zblocks else 0,
    })

feat_df = pd.DataFrame(route_features)
analysis = route_stats.merge(feat_df, on="route_id", how="inner")

# ============================================================
# 3. Типизация по MAE/bias — правильная классификация
# ============================================================
# Тип по направлению систематической ошибки
analysis["bias_ratio"] = analysis["sum_bias"] / (analysis["sum_true"] + 1e-9)
# + = завышаем (overpredict), - = занижаем (underpredict)
analysis["bias_type"] = "neutral"
analysis.loc[analysis["bias_ratio"] >  0.05, "bias_type"] = "overpredict"
analysis.loc[analysis["bias_ratio"] < -0.05, "bias_type"] = "underpredict"

# Тип по источнику ошибки: пики vs база
analysis["peak_mae_ratio"] = analysis["mae_top20"] / (analysis["mae_bot80"] + 1e-9)
analysis["error_source"] = "base_driven"
analysis.loc[analysis["peak_mae_ratio"] > 2.0, "error_source"] = "peak_driven"

print(f"\n\n── Типы по bias (MAE-based) ──")
for bt, grp in analysis.groupby("bias_type"):
    print(f"  {bt:<15}: {len(grp):>4} маршрутов  "
          f"ср.SAE={grp['sae'].mean():>12.0f}  "
          f"сумм.SAE={grp['sae'].sum():>14.0f}  "
          f"({grp['sae'].sum()/total_sae*100:.1f}% метрики)")

print(f"\n── Типы по источнику ошибки ──")
for et, grp in analysis.groupby("error_source"):
    print(f"  {et:<15}: {len(grp):>4} маршрутов  "
          f"ср.SAE={grp['sae'].mean():>12.0f}  "
          f"сумм.SAE={grp['sae'].sum():>14.0f}  "
          f"({grp['sae'].sum()/total_sae*100:.1f}% метрики)")

# ============================================================
# 4. Квартили по SAE — кто реально тянет метрику вниз
# ============================================================
analysis["sae_q"] = pd.qcut(analysis["sae"], q=4,
                             labels=["Q1_low","Q2","Q3","Q4_high"])
feat_cols = ["mean_y","cv","zero_frac","acf1","roll_cv",
             "growth_recent","n_zero_blocks","mean_bias","bias_ratio"]

print(f"\n\n── Средние характеристики по квартилям SAE ──")
q_stats = analysis.groupby("sae_q")[feat_cols + ["sae","denom_pct","error_efficiency"]].mean()
print(q_stats.round(3).to_string())

print(f"\n── Корреляция с SAE (Spearman) ──")
for col in feat_cols + ["mean_true"]:
    r = analysis[["sae", col]].dropna().corr(method="spearman").iloc[0,1]
    bar = "█" * int(abs(r) * 30)
    sign = "+" if r > 0 else "-"
    print(f"  {col:<20} {sign}{abs(r):.3f}  {bar}")

# ============================================================
# 5. Целевые маршруты для вмешательства
#    Критерий: высокий SAE + потенциально исправимая ошибка
# ============================================================
# Underpredict + растущий тренд — calib_scale < 1 при growth_recent > 1.15
target_underpredict = analysis[
    (analysis["bias_type"] == "underpredict") &
    (analysis["growth_recent"] > 1.15)
].sort_values("sae", ascending=False)

# Overpredict + падающий тренд — calib_scale > 1 при growth_recent < 0.7
target_overpredict = analysis[
    (analysis["bias_type"] == "overpredict") &
    (analysis["growth_recent"] < 0.7)
].sort_values("sae", ascending=False)

# Высокий SAE без явного тренда — base_driven, neutral bias
# Это фундаментальный шум — вмешательство не поможет
target_noise = analysis[
    (analysis["bias_type"] == "neutral") &
    (analysis["error_source"] == "base_driven")
].sort_values("sae", ascending=False)

print(f"\n\n── Целевые маршруты для вмешательства ──")
print(f"\n[1] Underpredict + растущий тренд (исправимо через тренд-scale):")
print(f"    {len(target_underpredict)} маршрутов, суммарный SAE = "
      f"{target_underpredict['sae'].sum():.0f} "
      f"({target_underpredict['sae'].sum()/total_sae*100:.1f}% метрики)")
print(target_underpredict[["route_id","sae","sae_pct","mean_bias",
                             "growth_recent","calib_scale"]].head(10).round(3).to_string(index=False))

print(f"\n[2] Overpredict + падающий тренд (исправимо через тренд-scale вниз):")
print(f"    {len(target_overpredict)} маршрутов, суммарный SAE = "
      f"{target_overpredict['sae'].sum():.0f} "
      f"({target_overpredict['sae'].sum()/total_sae*100:.1f}% метрики)")
print(target_overpredict[["route_id","sae","sae_pct","mean_bias",
                            "growth_recent","calib_scale"]].head(10).round(3).to_string(index=False))

print(f"\n[3] Высокий SAE, нет явного тренда (фундаментальный шум):")
print(f"    {len(target_noise)} маршрутов, суммарный SAE = "
      f"{target_noise['sae'].sum():.0f} "
      f"({target_noise['sae'].sum()/total_sae*100:.1f}% метрики)")
print(f"    → Вмешательство нецелесообразно, потенциал только через loss/input_size")

# Теоретический максимум улучшения
sae_actionable = (target_underpredict["sae"].sum() +
                  target_overpredict["sae"].sum())
# Если исправим bias_ratio до 0 на этих маршрутах:
sae_reducible = (np.abs(target_underpredict["sum_bias"]).sum() +
                 np.abs(target_overpredict["sum_bias"]).sum())
new_wape = (total_sae - sae_reducible) / total_sum_true
print(f"\n── Теоретический потенциал вмешательства ──")
print(f"  Текущий WAPE (calibrated):      {global_wape:.5f}")
print(f"  Если устраним весь bias [1]+[2]: {new_wape:.5f}")
print(f"  Теоретический потолок улучшения: {global_wape - new_wape:+.5f}")

analysis.to_csv("route_mae_analysis.csv", index=False)
target_underpredict.to_csv("routes_target_underpredict.csv", index=False)
target_overpredict.to_csv("routes_target_overpredict.csv", index=False)
print(f"\n✅ route_mae_analysis.csv")
print(f"✅ routes_target_underpredict.csv")
print(f"✅ routes_target_overpredict.csv")

In [ ]:
# ============================================================
# Быстрый grid search через предвычисление
#
# Ключевая идея:
# trend_ratio зависит только от (route_id, fold, window_h)
# и НЕ зависит от (growth_thresh, fall_thresh, alpha)
#
# Поэтому:
#   1. Считаем ratio для каждого (route × fold × window) ОДИН РАЗ
#   2. Grid search — чистый numpy, без DataFrame операций
# ============================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

raw_cv_df = pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv", parse_dates=["timestamp"])
train_df  = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df  = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
sub_raw   = pd.read_csv("submission_p2_blk_8_4_2_raw.csv")
test_df   = pd.read_parquet("test_solo_track.parquet")[["id","route_id"]]

N_FOLDS       = 5
SCALE_MIN, SCALE_MAX = 0.5, 2.0
WINDOWS       = [48, 96, 168]

global_scale  = raw_cv_df["y_true"].sum() / (raw_cv_df["y_pred"].sum() + 1e-9)

# ── Reference ──
ref_yt  = raw_cv_df["y_true"].values
ref_yp  = np.clip(raw_cv_df["y_pred"].values * global_scale, 0, None)
denom   = ref_yt.sum() + 1e-9
tot_ref = np.abs(ref_yp - ref_yt).sum() / denom + \
          np.abs(ref_yp.sum() / denom - 1)
print(f"Reference: {tot_ref:.5f}")

# ── Индексы фолдов ──
fold_arr    = raw_cv_df["fold"].values
route_arr   = raw_cv_df["route_id"].values
y_true_arr  = raw_cv_df["y_true"].values
y_pred_arr  = raw_cv_df["y_pred"].values

folds      = sorted(raw_cv_df["fold"].unique())
fold_cuts  = {f: raw_cv_df[raw_cv_df["fold"]==f]["timestamp"].min()
              for f in folds}

# ── История маршрутов в numpy (быстрее DataFrame) ──
print("Precomputing route histories...")
route_ids    = sorted(raw_cv_df["route_id"].unique())
route_idx    = {r: i for i, r in enumerate(route_ids)}
route_arr_i  = np.array([route_idx[r] for r in route_arr])

# Для каждого маршрута — массив (timestamp, value) отсортированный
route_hist = {}
for rid, grp in train_df.groupby("route_id"):
    grp = grp.sort_values("timestamp")
    route_hist[rid] = (grp["timestamp"].values, grp["target_1h"].values.astype(float))

# ============================================================
# ШАГ 1: Предвычисляем trend_ratio[window][fold][route_idx]
# Результат: словарь ratio_table[window_h] = np.array shape (N_FOLDS, N_ROUTES)
# ============================================================
print("Precomputing trend ratios (1 pass per window)...")
N_ROUTES = len(route_ids)
ratio_tables = {}  # window_h → (N_FOLDS, N_ROUTES)

for window_h in WINDOWS:
    ratio_mat = np.ones((N_FOLDS, N_ROUTES), dtype=np.float32)
    for fi, fold in enumerate(folds):
        cutoff_ts = fold_cuts[fold]
        for ri, rid in enumerate(route_ids):
            ts_arr, val_arr = route_hist.get(rid, (np.array([]), np.array([])))
            mask = ts_arr < cutoff_ts
            hist = val_arr[mask]
            if len(hist) < window_h * 2:
                ratio_mat[fi, ri] = 1.0   # нет истории → no-op
                continue
            mean_all    = hist.mean() + 1e-9
            mean_recent = hist[-window_h:].mean()
            ratio_mat[fi, ri] = mean_recent / mean_all
    ratio_tables[window_h] = ratio_mat
    print(f"  window={window_h}: done  "
          f"ratio>1.2: {(ratio_mat > 1.2).sum()}  "
          f"ratio<0.7: {(ratio_mat < 0.7).sum()}")

# Предвычисляем тест-ratios (из полной истории)
test_ratio = {}
last_ts_map = train_df.groupby("route_id")["timestamp"].max()
test_route_arr = sub_raw.merge(test_df, on="id")["route_id"].values

for window_h in WINDOWS:
    tr = np.ones(N_ROUTES, dtype=np.float32)
    for ri, rid in enumerate(route_ids):
        ts_arr, val_arr = route_hist.get(rid, (np.array([]), np.array([])))
        if len(val_arr) < window_h * 2:
            continue
        mean_all    = val_arr.mean() + 1e-9
        mean_recent = val_arr[-window_h:].mean()
        tr[ri] = mean_recent / mean_all
    test_ratio[window_h] = tr

print("Precomputation done.\n")

# ============================================================
# ШАГ 2: Grid search — чистый numpy, без pandas
# scale[fold][obs] = f(ratio, g_thresh, f_thresh, alpha)
# ============================================================
def grid_eval_fast(window_h, g_thresh, f_thresh, alpha):
    ratio_mat = ratio_tables[window_h]  # (N_FOLDS, N_ROUTES)

    # scale matrix: начинаем с global_scale для всех
    scale_mat = np.full((N_FOLDS, N_ROUTES), global_scale, dtype=np.float64)

    # Рост: ratio > g_thresh
    grow_mask = ratio_mat > g_thresh
    trend_s_grow = np.clip(ratio_mat * global_scale, SCALE_MIN, SCALE_MAX)
    scale_mat[grow_mask] = (alpha * global_scale +
                            (1 - alpha) * trend_s_grow[grow_mask])

    # Падение: ratio < f_thresh
    fall_mask = ratio_mat < f_thresh
    trend_s_fall = np.clip(ratio_mat * global_scale, SCALE_MIN, SCALE_MAX)
    scale_mat[fall_mask] = (alpha * global_scale +
                            (1 - alpha) * trend_s_fall[fall_mask])

    # Применяем к CV
    # scale для каждого наблюдения: fold → fi, route_id → ri
    obs_scales = scale_mat[fold_arr - 1, route_arr_i]  # (N_OBS,)
    yp_cal = np.clip(y_pred_arr * obs_scales, 0, None)

    wape  = np.abs(yp_cal - y_true_arr).sum() / denom
    rbias = np.abs(yp_cal.sum() / denom - 1)
    return wape + rbias, wape, rbias

# Grid
g_thresholds = [1.1, 1.15, 1.2, 1.25, 1.35, 1.5, 2.0]
f_thresholds = [0.3, 0.5, 0.7, 0.85]
alphas       = [0.0, 0.2, 0.3, 0.5, 0.7, 0.8, 1.0]

total_combos = len(WINDOWS) * len(g_thresholds) * len(f_thresholds) * len(alphas)
print(f"Grid: {len(WINDOWS)}×{len(g_thresholds)}×{len(f_thresholds)}×{len(alphas)}"
      f" = {total_combos} комбинаций (чистый numpy, должно быть быстро)...")

grid_rows = []
best = {"total": tot_ref, "params": None}

for window_h in WINDOWS:
    for g_thresh in g_thresholds:
        for f_thresh in f_thresholds:
            for alpha in alphas:
                tot, w, r = grid_eval_fast(window_h, g_thresh, f_thresh, alpha)
                grid_rows.append({
                    "window_h": window_h, "g_thresh": g_thresh,
                    "f_thresh": f_thresh, "alpha": alpha,
                    "total": tot, "wape": w, "rbias": r,
                })
                if tot < best["total"]:
                    best = {"total": tot, "wape": w, "rbias": r,
                            "params": (window_h, g_thresh, f_thresh, alpha)}

df_grid = pd.DataFrame(grid_rows).sort_values("total")
print(f"\nТоп-15 конфигураций:")
print(df_grid.head(15).round(5).to_string(index=False))

# ── Результат ──
if best["params"] is None:
    print(f"\n❌ Тренд-интервенция не улучшает reference")
    print(f"→ Постпроцессинг исчерпан. Ждём MAE/Huber loss экспериментов.")
else:
    window_h, g_thresh, f_thresh, alpha = best["params"]
    print(f"\n🏆 Best: window={window_h}, g={g_thresh}, f={f_thresh}, α={alpha}")
    print(f"   Total={best['total']:.5f}  Δ={best['total']-tot_ref:+.5f}")

    # Per-fold breakdown
    ratio_mat  = ratio_tables[window_h]
    scale_mat  = np.full((N_FOLDS, N_ROUTES), global_scale, dtype=np.float64)
    grow_mask  = ratio_mat > g_thresh
    fall_mask  = ratio_mat < f_thresh
    scale_mat[grow_mask] = (alpha*global_scale +
                            (1-alpha)*np.clip(ratio_mat[grow_mask]*global_scale,
                                              SCALE_MIN, SCALE_MAX))
    scale_mat[fall_mask] = (alpha*global_scale +
                            (1-alpha)*np.clip(ratio_mat[fall_mask]*global_scale,
                                              SCALE_MIN, SCALE_MAX))
    obs_scales = scale_mat[fold_arr - 1, route_arr_i]
    yp_final   = np.clip(y_pred_arr * obs_scales, 0, None)

    print("\nPer-fold:")
    for fold in folds:
        mask = fold_arr == fold
        t_r  = (np.abs(ref_yp[mask]-y_true_arr[mask]).sum() / y_true_arr[mask].sum()
                + np.abs(ref_yp[mask].sum()/y_true_arr[mask].sum()-1))
        t_n  = (np.abs(yp_final[mask]-y_true_arr[mask]).sum() / y_true_arr[mask].sum()
                + np.abs(yp_final[mask].sum()/y_true_arr[mask].sum()-1))
        print(f"  Fold {fold}  Ref={t_r:.4f} → New={t_n:.4f}  Δ={t_n-t_r:+.5f}")

    n_grow = grow_mask[-1].sum()
    n_fall = fall_mask[-1].sum()
    print(f"\nМаршрутов с коррекцией роста (fold 5): {n_grow}")
    print(f"Маршрутов с коррекцией падения (fold 5): {n_fall}")

    # ── Тест submission ──
    tr = test_ratio[window_h]
    test_scale_arr = np.full(N_ROUTES, global_scale, dtype=np.float64)
    gm = tr > g_thresh
    fm = tr < f_thresh
    test_scale_arr[gm] = (alpha*global_scale +
                          (1-alpha)*np.clip(tr[gm]*global_scale, SCALE_MIN, SCALE_MAX))
    test_scale_arr[fm] = (alpha*global_scale +
                          (1-alpha)*np.clip(tr[fm]*global_scale, SCALE_MIN, SCALE_MAX))
    test_scale_map = {rid: test_scale_arr[route_idx[rid]] for rid in route_ids}

    sub_out = sub_raw.merge(test_df, on="id", how="left")
    sub_out["scale"]  = sub_out["route_id"].map(test_scale_map).fillna(global_scale)
    sub_out["y_pred"] = np.clip(sub_out["y_pred"] * sub_out["scale"], 0, None)
    sub_out[["id","y_pred"]].sort_values("id").to_csv(
        "submission_trend_intervention.csv", index=False
    )
    print(f"\n✅ submission_trend_intervention.csv")

In [ ]:
# ============================================================
# NHITS horizon sweep for winner config: p2_blk_8_4_2
# - sweep horizons: 2, 4, 8, 16, 32, 48, 336
# - 3-fold chained CV with step_size = horizon
# - reduced max_steps for fast comparison
# - optional predict() only when test horizon matches model horizon
# ============================================================

import warnings, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Metric ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH      = "train_solo_track.parquet"
TEST_PATH       = "test_solo_track.parquet"
TARGET_COL      = "target_1h"
FREQ            = "30min"
CONTEXT_LEN     = 2048
BASE_INPUT_SIZE = 336
BATCH_SIZE      = 64

HORIZONS  = [2, 4, 8, 16, 32, 48, 48 * 7]
N_FOLDS   = 3
CV_STEPS  = 200          # быстрый прогон
FULL_STEPS_FOR_TEST = 400  # только если horizon совпадает с test horizon

BEST_CFG = {
    "name": "p2_blk_8_4_2",
    "n_freq_downsample": [48, 8, 1],
    "n_pool_kernel_size": [1, 1, 1],
    "n_blocks": [8, 4, 2],
}

if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_2", "status_3", "status_5"]])

FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]
HIST_EXOG = (status_cols + ["lag_48", "lag_336"]) or None

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)

    if not rows:
        return pd.DataFrame(columns=["unique_id", "ds", "y"] + status_cols)

    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df, horizon):
    rows = []
    for rid in route_ids:
        last_ts = train_df.loc[train_df["route_id"] == rid, "timestamp"].max()
        for ts in pd.date_range(
            start=last_ts + pd.Timedelta("30min"),
            periods=horizon,
            freq=FREQ
        ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id", "ds", "y", "cutoff"}
    num = [c for c in pred_df.columns if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    med = [c for c in num if any(x in c for x in ["median", "0.5", "50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {
        rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
        for rid, g in pred_df.groupby("unique_id")
    }

def resolve_input_size(horizon):
    # Чистый sweep по горизонту: input_size держим фиксированным.
    # Если захочешь "лучшую модель на каждом горизонте", попробуй:
    # return min(CONTEXT_LEN, max(BASE_INPUT_SIZE, 2 * horizon))
    return BASE_INPUT_SIZE

def make_nhits(cfg, horizon, input_size, max_steps):
    return NHITS(
        h=horizon,
        input_size=input_size,
        loss=MQLoss(level=[80]),
        n_freq_downsample=cfg["n_freq_downsample"],
        n_pool_kernel_size=cfg["n_pool_kernel_size"],
        n_blocks=cfg["n_blocks"],
        mlp_units=[[512, 512]] * len(cfg["n_blocks"]),
        max_steps=max_steps,
        batch_size=BATCH_SIZE,
        accelerator=ACCELERATOR,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        scaler_type="robust",
        enable_progress_bar=False,
    )

def run_cv(cfg, train_nf, horizon, input_size, max_steps, n_folds):
    nf = NeuralForecast(
        models=[make_nhits(cfg, horizon, input_size, max_steps)],
        freq=FREQ
    )
    cv_df = nf.cross_validation(
        df=train_nf,
        n_windows=n_folds,
        step_size=horizon,
        refit=True
    )

    pred_col = get_pred_col(cv_df)

    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i + 1 for i, c in enumerate(cuts)})
    cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
    cv_df["h_step"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

    yt = cv_df["y"].values
    yp = np.clip(cv_df[pred_col].values, 0, None)

    tot, wape, rb = wape_rbias(yt, yp)

    calib = float(yt.sum() / (yp.sum() + 1e-9))
    yp_c = np.clip(yp * calib, 0, None)
    tot_c, wape_c, rb_c = wape_rbias(yt, yp_c)

    fold_rows = []
    for fold in sorted(cv_df["fold"].unique()):
        fdf = cv_df[cv_df["fold"] == fold]
        t, w, r = wape_rbias(fdf["y"].values, np.clip(fdf[pred_col].values, 0, None))
        fold_rows.append({
            "fold": int(fold),
            "wape": round(w, 4),
            "rbias": round(r, 4),
            "total": round(t, 4),
        })

    del nf, cv_df
    gc.collect()

    return {
        "wape_raw": round(wape, 4),
        "rbias_raw": round(rb, 4),
        "total_raw": round(tot, 4),
        "wape_cal": round(wape_c, 4),
        "rbias_cal": round(rb_c, 4),
        "total_cal": round(tot_c, 4),
        "calib_scale": round(calib, 4),
        "fold_stats": fold_rows,
    }

def maybe_predict_test(cfg, horizon, input_size, max_steps):
    route_sizes = test_df.groupby("route_id").size()
    if route_sizes.nunique() != 1:
        print(f"Skip predict for h={horizon}: test has non-uniform route lengths")
        return None

    test_horizon = int(route_sizes.iloc[0])
    if test_horizon != horizon:
        print(f"Skip predict for h={horizon}: test horizon is {test_horizon}")
        return None

    print(f"Running predict() for h={horizon} ...")

    train_nf_full = to_nf_df(train_df, min_len=input_size + 1, context_len=CONTEXT_LEN)
    test_route_ids = test_df["route_id"].unique().tolist()

    nf_full = NeuralForecast(
        models=[make_nhits(cfg, horizon, input_size, max_steps)],
        freq=FREQ
    )
    nf_full.fit(train_nf_full)

    futr_df = make_futr_df(test_route_ids, train_df, horizon)
    test_pred_df = nf_full.predict(futr_df=futr_df)
    pred_col = get_pred_col(test_pred_df)
    test_preds = build_preds_dict(test_pred_df, pred_col)

    predictions_raw = {}
    for route_id in test_route_ids:
        route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
        preds = test_preds.get(route_id, np.zeros(horizon))
        for j, (_, row) in enumerate(route_test.iterrows()):
            pred = float(preds[j]) if j < len(preds) else float(preds[-1])
            predictions_raw[row["id"]] = max(0.0, pred)

    submission = (
        pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
        .sort_values("id")
        .reset_index(drop=True)
    )
    file_name = f"submission_{cfg['name']}_h{horizon}.csv"
    submission.to_csv(file_name, index=False)
    print(f"Saved: {file_name}")
    return file_name

# # ── Main sweep ──
# all_results = []
# all_fold_rows = []

# for horizon in HORIZONS:
#     input_size = resolve_input_size(horizon)
#     min_len = input_size + N_FOLDS * horizon + 1

#     train_nf = to_nf_df(
#         train_df,
#         min_len=min_len,
#         context_len=CONTEXT_LEN,
#     )

#     n_series = train_nf["unique_id"].nunique()
#     n_rows = len(train_nf)

#     print("\n" + "=" * 70)
#     print(f"HORIZON = {horizon}")
#     print(f"input_size={input_size} | min_len={min_len} | series={n_series} | rows={n_rows}")
#     print("=" * 70)

#     if n_series == 0:
#         print("No eligible series, skipping.")
#         continue

#     metrics = run_cv(
#         cfg=BEST_CFG,
#         train_nf=train_nf,
#         horizon=horizon,
#         input_size=input_size,
#         max_steps=CV_STEPS,
#         n_folds=N_FOLDS,
#     )

#     for fr in metrics["fold_stats"]:
#         print(f"Fold {fr['fold']}: WAPE={fr['wape']:.4f} |RBias|={fr['rbias']:.4f} Total={fr['total']:.4f}")
#         all_fold_rows.append({
#             "horizon": horizon,
#             "fold": fr["fold"],
#             "wape": fr["wape"],
#             "rbias": fr["rbias"],
#             "total": fr["total"],
#         })

#     print(f"RAW   : WAPE={metrics['wape_raw']:.4f} |RBias|={metrics['rbias_raw']:.4f} Total={metrics['total_raw']:.4f}")
#     print(f"CALIB : WAPE={metrics['wape_cal']:.4f} |RBias|={metrics['rbias_cal']:.4f} Total={metrics['total_cal']:.4f} scale={metrics['calib_scale']:.4f}")

#     submission_file = maybe_predict_test(
#         cfg=BEST_CFG,
#         horizon=horizon,
#         input_size=input_size,
#         max_steps=FULL_STEPS_FOR_TEST,
#     )

#     all_results.append({
#         "model_name": BEST_CFG["name"],
#         "horizon": horizon,
#         "input_size": input_size,
#         "n_folds": N_FOLDS,
#         "cv_steps": CV_STEPS,
#         "n_series": n_series,
#         "n_rows": n_rows,
#         "freq": str(BEST_CFG["n_freq_downsample"]),
#         "blocks": str(BEST_CFG["n_blocks"]),
#         "wape_raw": metrics["wape_raw"],
#         "rbias_raw": metrics["rbias_raw"],
#         "total_raw": metrics["total_raw"],
#         "wape_cal": metrics["wape_cal"],
#         "rbias_cal": metrics["rbias_cal"],
#         "total_cal": metrics["total_cal"],
#         "calib_scale": metrics["calib_scale"],
#         "submission_file": submission_file,
#     })

#     del train_nf
#     gc.collect()

# # ── Save tables ──
# results_df = pd.DataFrame(all_results).sort_values("horizon").reset_index(drop=True)
# folds_df   = pd.DataFrame(all_fold_rows).sort_values(["horizon", "fold"]).reset_index(drop=True)

# results_df.to_csv("nhits_horizon_sweep.csv", index=False)
# folds_df.to_csv("nhits_horizon_sweep_folds.csv", index=False)

# print("\nSaved: nhits_horizon_sweep.csv")
# print("Saved: nhits_horizon_sweep_folds.csv")
# print("\nLeaderboard by horizon:")
# print(results_df[["horizon", "n_series", "wape_raw", "rbias_raw", "total_raw", "total_cal", "calib_scale"]].to_string(index=False))

# # ── Plot ──
# if len(results_df) > 0:
#     fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#     # total metric vs horizon
#     ax = axes[0]
#     ax.plot(results_df["horizon"], results_df["total_raw"], marker="o", linewidth=2, label="total_raw")
#     ax.plot(results_df["horizon"], results_df["total_cal"], marker="o", linewidth=2, label="total_cal")
#     ax.set_xscale("log", base=2)
#     ax.set_xlabel("Forecast horizon")
#     ax.set_ylabel("Metric value")
#     ax.set_title("NHITS quality vs horizon")
#     ax.grid(True, alpha=0.3)
#     ax.legend()

#     # number of series per horizon
#     ax = axes[1]
#     ax.bar(results_df["horizon"].astype(str), results_df["n_series"])
#     ax.set_xlabel("Forecast horizon")
#     ax.set_ylabel("Eligible series")
#     ax.set_title("How many series remain in CV")
#     ax.grid(True, axis="y", alpha=0.3)

#     plt.tight_layout()
#     plt.savefig("nhits_horizon_sweep.png", dpi=220, bbox_inches="tight")
#     plt.close()

#     print("Saved: nhits_horizon_sweep.png")

In [ ]:
# ============================================================
# FULL ONE-CELL BENCHMARK: NHITS base inference
# - self-contained
# - reads train/test parquet
# - fits baseline NHITS (p2_blk_8_4_2, h=8)
# - measures cold and warm inference
# - saves CSV with benchmark results
# ============================================================

import warnings
import gc
import time
import numpy as np
import pandas as pd
import torch

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"

FREQ           = "30min"
FORECAST_STEPS = 8
CONTEXT_LEN    = 2048
INPUT_SIZE     = 336
BATCH_SIZE     = 64
FINAL_STEPS    = 800

N_WARMUP       = 2
N_RUNS         = 20

BASE_CFG = {
    "name": "p2_blk_8_4_2",
    "n_freq_downsample": [48, 8, 1],
    "n_pool_kernel_size": [1, 1, 1],
    "n_blocks": [8, 4, 2],
}

# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"

print(f"Device: {ACCELERATOR}")

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elif hasattr(torch, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.synchronize()
        except Exception:
            pass

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_2", "status_3", "status_5"]])

FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]
HIST_EXOG = (status_cols + ["lag_48", "lag_336"]) or None

# ── Helpers ──
def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)

    if not rows:
        raise ValueError("No eligible series after preprocessing.")

    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df, horizon=FORECAST_STEPS):
    rows = []
    for rid in route_ids:
        last_ts = train_df.loc[train_df["route_id"] == rid, "timestamp"].max()
        for ts in pd.date_range(
            start=last_ts + pd.Timedelta(FREQ),
            periods=horizon,
            freq=FREQ
        ):
            rows.append({"unique_id": rid, "ds": ts})
    futr_df = pd.DataFrame(rows)
    futr_df = add_calendar_features(futr_df)
    return futr_df

def get_pred_col(pred_df):
    for c in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id", "ds", "y", "cutoff"}
    num = [c for c in pred_df.columns if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    med = [c for c in num if any(x in c for x in ["median", "0.5", "50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {
        rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
        for rid, g in pred_df.groupby("unique_id")
    }

def make_nhits(cfg, max_steps):
    return NHITS(
        h=FORECAST_STEPS,
        input_size=INPUT_SIZE,
        loss=MQLoss(level=[80]),
        n_freq_downsample=cfg["n_freq_downsample"],
        n_pool_kernel_size=cfg["n_pool_kernel_size"],
        n_blocks=cfg["n_blocks"],
        mlp_units=[[512, 512]] * len(cfg["n_blocks"]),
        max_steps=max_steps,
        batch_size=BATCH_SIZE,
        accelerator=ACCELERATOR,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        scaler_type="robust",
        enable_progress_bar=False,
    )

def summarize(arr_sec):
    arr = np.asarray(arr_sec, dtype=float)
    return {
        "mean_ms": round(arr.mean() * 1000, 2),
        "std_ms":  round(arr.std(ddof=0) * 1000, 2),
        "min_ms":  round(arr.min() * 1000, 2),
        "p50_ms":  round(np.percentile(arr, 50) * 1000, 2),
        "p95_ms":  round(np.percentile(arr, 95) * 1000, 2),
        "p99_ms":  round(np.percentile(arr, 99) * 1000, 2),
        "max_ms":  round(arr.max() * 1000, 2),
    }

def postprocess_predictions(pred_df, test_df, test_route_ids):
    pred_col = get_pred_col(pred_df)
    test_preds = build_preds_dict(pred_df, pred_col)

    predictions_raw = {}
    for route_id in test_route_ids:
        route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
        preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
        for j, (_, row) in enumerate(route_test.iterrows()):
            pred = float(preds[j]) if j < len(preds) else float(preds[-1])
            predictions_raw[row["id"]] = max(0.0, pred)

    submission = (
        pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
        .sort_values("id")
        .reset_index(drop=True)
    )
    return submission

# ── Prepare inputs ──
train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
futr_df = make_futr_df(test_route_ids, train_df, horizon=FORECAST_STEPS)

route_sizes = test_df.groupby("route_id").size()
test_horizon = int(route_sizes.iloc[0]) if len(route_sizes) > 0 else None

print(f"Train NF shape: {train_nf_full.shape}")
print(f"Routes in test: {len(test_route_ids)}")
print(f"Test horizon per route: {test_horizon}")

if route_sizes.nunique() != 1:
    raise ValueError("test_df has non-uniform horizon across route_id; benchmark assumes fixed horizon.")
if test_horizon != FORECAST_STEPS:
    raise ValueError(f"test_df horizon = {test_horizon}, but model horizon = {FORECAST_STEPS}.")

# ── Fit once ──
nf_base = NeuralForecast(models=[make_nhits(BASE_CFG, FINAL_STEPS)], freq=FREQ)

t0 = time.perf_counter()
nf_base.fit(train_nf_full)
sync_device()
fit_sec = time.perf_counter() - t0

print(f"Fit time: {fit_sec:.3f} sec")

# ── Cold predict ──
t0 = time.perf_counter()
pred_df_cold = nf_base.predict(futr_df=futr_df)
sync_device()
cold_predict_sec = time.perf_counter() - t0

t1 = time.perf_counter()
submission_cold = postprocess_predictions(pred_df_cold, test_df, test_route_ids)
cold_post_sec = time.perf_counter() - t1
cold_full_sec = cold_predict_sec + cold_post_sec

# ── Warmup ──
for _ in range(N_WARMUP):
    _ = nf_base.predict(futr_df=futr_df)
    sync_device()

# ── Warm benchmark ──
predict_times = []
postprocess_times = []
full_times = []

for i in range(N_RUNS):
    gc.collect()

    t0 = time.perf_counter()
    pred_df = nf_base.predict(futr_df=futr_df)
    sync_device()
    t1 = time.perf_counter()

    _submission = postprocess_predictions(pred_df, test_df, test_route_ids)
    t2 = time.perf_counter()

    predict_times.append(t1 - t0)
    postprocess_times.append(t2 - t1)
    full_times.append(t2 - t0)

# ── Stats ──
pred_stats = summarize(predict_times)
post_stats = summarize(postprocess_times)
full_stats = summarize(full_times)

mean_predict_sec = float(np.mean(predict_times))
mean_full_sec = float(np.mean(full_times))

bench_rows = [
    {"metric": "device", "value": ACCELERATOR},
    {"metric": "model_name", "value": BASE_CFG["name"]},
    {"metric": "forecast_horizon", "value": FORECAST_STEPS},
    {"metric": "n_series", "value": len(test_route_ids)},
    {"metric": "n_rows_test", "value": len(test_df)},
    {"metric": "fit_once_sec", "value": round(fit_sec, 4)},

    {"metric": "cold_predict_ms", "value": round(cold_predict_sec * 1000, 2)},
    {"metric": "cold_postprocess_ms", "value": round(cold_post_sec * 1000, 2)},
    {"metric": "cold_full_ms", "value": round(cold_full_sec * 1000, 2)},

    {"metric": "predict_mean_ms", "value": pred_stats["mean_ms"]},
    {"metric": "predict_std_ms", "value": pred_stats["std_ms"]},
    {"metric": "predict_min_ms", "value": pred_stats["min_ms"]},
    {"metric": "predict_p50_ms", "value": pred_stats["p50_ms"]},
    {"metric": "predict_p95_ms", "value": pred_stats["p95_ms"]},
    {"metric": "predict_p99_ms", "value": pred_stats["p99_ms"]},
    {"metric": "predict_max_ms", "value": pred_stats["max_ms"]},

    {"metric": "postprocess_mean_ms", "value": post_stats["mean_ms"]},
    {"metric": "postprocess_std_ms", "value": post_stats["std_ms"]},
    {"metric": "postprocess_p50_ms", "value": post_stats["p50_ms"]},
    {"metric": "postprocess_p95_ms", "value": post_stats["p95_ms"]},
    {"metric": "postprocess_p99_ms", "value": post_stats["p99_ms"]},

    {"metric": "full_mean_ms", "value": full_stats["mean_ms"]},
    {"metric": "full_std_ms", "value": full_stats["std_ms"]},
    {"metric": "full_min_ms", "value": full_stats["min_ms"]},
    {"metric": "full_p50_ms", "value": full_stats["p50_ms"]},
    {"metric": "full_p95_ms", "value": full_stats["p95_ms"]},
    {"metric": "full_p99_ms", "value": full_stats["p99_ms"]},
    {"metric": "full_max_ms", "value": full_stats["max_ms"]},

    {"metric": "series_per_sec_predict", "value": round(len(test_route_ids) / mean_predict_sec, 2)},
    {"metric": "rows_per_sec_predict", "value": round(len(test_df) / mean_predict_sec, 2)},
    {"metric": "series_per_sec_full", "value": round(len(test_route_ids) / mean_full_sec, 2)},
    {"metric": "rows_per_sec_full", "value": round(len(test_df) / mean_full_sec, 2)},
]

bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv("nhits_base_inference_benchmark.csv", index=False)

submission_cold.to_csv("submission_base_cold_predict.csv", index=False)

print("\n=== BENCHMARK SUMMARY ===")
print(bench_df.to_string(index=False))

print("\nSaved files:")
print(" - nhits_base_inference_benchmark.csv")
print(" - submission_base_cold_predict.csv")